In [1]:
# ---------------------------------------------------------
# MODEL EVALUATION NOTEBOOK
#
# This notebook evaluates the recommendation system created
# in 07_Model_Building.ipynb.
#
# IMPORTANT:
# We do NOT retrain or rebuild the model here.
#
# Instead, we load the saved artifacts and evaluate:
#
#   - recommendation validity
#   - personalization
#   - similarity quality
#   - ranking behaviour
#   - diversity
#   - availability handling
#
# This separation makes the project structure closer to a
# real ML/MLOps workflow:
#
#   Data Processing
#          ↓
#   Model Building
#          ↓
#   Model Evaluation
#          ↓
#   Deployment
# ---------------------------------------------------------

import os
import pickle

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Define project paths.
# ---------------------------------------------------------

MODEL_DIR = "../models"
DATA_DIR = "../data/cleaned"


# ---------------------------------------------------------
# Display evaluation environment information.
# ---------------------------------------------------------

print("=" * 60)
print("MODEL EVALUATION ENVIRONMENT")
print("=" * 60)

print("\nModel directory:")
print(
    os.path.abspath(MODEL_DIR)
)

print("\nData directory:")
print(
    os.path.abspath(DATA_DIR)
)

print("\n✓ Evaluation environment initialized")

MODEL EVALUATION ENVIRONMENT

Model directory:
c:\Users\Rushi\Desktop\Travel_Agent\models

Data directory:
c:\Users\Rushi\Desktop\Travel_Agent\data\cleaned

✓ Evaluation environment initialized


In [3]:
# ---------------------------------------------------------
# LOAD SAVED MODEL ARTIFACTS
#
# Everything used for evaluation should come from the saved
# model artifacts rather than relying on variables from
# Notebook 07.
# ---------------------------------------------------------

# ---------------------------------------------------------
# Destination similarity matrix.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "destination_similarity.pkl"
    ),
    "rb"
) as file:

    destination_similarity = pickle.load(
        file
    )


# ---------------------------------------------------------
# Destination list used by the recommender.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_destinations.pkl"
    ),
    "rb"
) as file:

    recommendation_destinations = pickle.load(
        file
    )


# ---------------------------------------------------------
# PCA recommendation feature metadata.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_feature_columns.pkl"
    ),
    "rb"
) as file:

    recommendation_feature_columns = pickle.load(
        file
    )


# ---------------------------------------------------------
# Preference groups.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "preference_groups.pkl"
    ),
    "rb"
) as file:

    preference_groups = pickle.load(
        file
    )


# ---------------------------------------------------------
# Preference direction map.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "preference_direction_map.pkl"
    ),
    "rb"
) as file:

    preference_direction_map = pickle.load(
        file
    )


# ---------------------------------------------------------
# Destination interest profiles.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "interest_profiles.pkl"
    ),
    "rb"
) as file:

    interest_profiles = pickle.load(
        file
    )


# ---------------------------------------------------------
# Destination profile scores.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "destination_profile_scores.pkl"
    ),
    "rb"
) as file:

    destination_profile_scores = pickle.load(
        file
    )


# ---------------------------------------------------------
# Recommendation configuration.
# ---------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "recommendation_config.pkl"
    ),
    "rb"
) as file:

    recommendation_config = pickle.load(
        file
    )


# ---------------------------------------------------------
# Load the saved PCA destination dataset.
# ---------------------------------------------------------

pca_destination_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "pca_destination_features.csv"
    )
)


# ---------------------------------------------------------
# Load the complete model-feature dataset.
# ---------------------------------------------------------

model_features_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "model_features_scaled.csv"
    )
)


print("=" * 60)
print("MODEL ARTIFACTS LOADED")
print("=" * 60)

print(
    "\nDestination count:",
    len(recommendation_destinations)
)

print(
    "Similarity matrix:",
    np.asarray(
        destination_similarity
    ).shape
)

print(
    "PCA components:",
    len(
        recommendation_feature_columns
    )
)

print(
    "Preference groups:",
    len(preference_groups)
)

print(
    "Interest profiles:",
    len(interest_profiles)
)

print(
    "Destination profile rows:",
    len(destination_profile_scores)
)

print(
    "\n✓ All evaluation artifacts loaded successfully"
)

MODEL ARTIFACTS LOADED

Destination count: 50
Similarity matrix: (50, 50)
PCA components: 15
Preference groups: 5
Interest profiles: 4
Destination profile rows: 50

✓ All evaluation artifacts loaded successfully


In [4]:
# ---------------------------------------------------------
# ARTIFACT VALIDATION
#
# Before evaluating recommendation quality, make sure the
# evaluation artifacts are structurally valid.
# ---------------------------------------------------------

print("=" * 60)
print("EVALUATION ARTIFACT VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Validate destination count.
# ---------------------------------------------------------

if len(
    recommendation_destinations
) != 50:

    raise ValueError(
        "Expected exactly 50 destinations."
    )


# ---------------------------------------------------------
# Validate destination uniqueness.
# ---------------------------------------------------------

if len(
    set(
        recommendation_destinations
    )
) != 50:

    raise ValueError(
        "Duplicate destinations detected."
    )


# ---------------------------------------------------------
# Convert similarity matrix to NumPy array for validation.
# ---------------------------------------------------------

similarity_array = np.asarray(
    destination_similarity
)


# ---------------------------------------------------------
# Validate similarity matrix dimensions.
# ---------------------------------------------------------

if similarity_array.shape != (
    50,
    50
):

    raise ValueError(
        "Similarity matrix must be 50 × 50."
    )


# ---------------------------------------------------------
# Validate symmetry.
# ---------------------------------------------------------

if not np.allclose(
    similarity_array,
    similarity_array.T,
    atol=1e-10
):

    raise ValueError(
        "Similarity matrix is not symmetric."
    )


# ---------------------------------------------------------
# Validate PCA dataset.
# ---------------------------------------------------------

expected_pca_columns = (
    ["destination"]
    +
    list(
        recommendation_feature_columns
    )
)


if (
    pca_destination_df.columns.tolist()
    != expected_pca_columns
):

    raise ValueError(
        "PCA destination dataset columns "
        "do not match recommendation features."
    )


# ---------------------------------------------------------
# Validate PCA missing values.
# ---------------------------------------------------------

if (
    pca_destination_df
    .isna()
    .sum()
    .sum()
    > 0
):

    raise ValueError(
        "PCA destination dataset contains missing values."
    )


# ---------------------------------------------------------
# Validate destination alignment.
# ---------------------------------------------------------

if (
    pca_destination_df[
        "destination"
    ].tolist()
    !=
    list(
        recommendation_destinations
    )
):

    raise ValueError(
        "Destination ordering mismatch."
    )


print(
    "\nDestination count: ✓"
)

print(
    "Destination uniqueness: ✓"
)

print(
    "Similarity matrix dimensions: ✓"
)

print(
    "Similarity matrix symmetry: ✓"
)

print(
    "PCA dataset structure: ✓"
)

print(
    "Destination alignment: ✓"
)


print("\n" + "=" * 60)
print("✓ EVALUATION ARTIFACT VALIDATION PASSED")
print("=" * 60)

EVALUATION ARTIFACT VALIDATION

Destination count: ✓
Destination uniqueness: ✓
Similarity matrix dimensions: ✓
Similarity matrix symmetry: ✓
PCA dataset structure: ✓
Destination alignment: ✓

✓ EVALUATION ARTIFACT VALIDATION PASSED


In [5]:
# ---------------------------------------------------------
# SIMILARITY MODEL QUALITY EVALUATION
#
# We evaluate the saved destination similarity matrix using
# several objective checks:
#
#   1. Self-similarity
#   2. Similarity range
#   3. Symmetry
#   4. Average similarity
#   5. Most similar destination pairs
#   6. Most dissimilar destination pairs
#
# This does not require labeled recommendation data.
# ---------------------------------------------------------

print("=" * 60)
print("SIMILARITY MODEL QUALITY EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Convert the saved similarity matrix to a NumPy array.
# ---------------------------------------------------------

similarity_array = np.asarray(
    destination_similarity,
    dtype=float
)


# ---------------------------------------------------------
# Basic statistics.
# ---------------------------------------------------------

n_destinations = (
    len(
        recommendation_destinations
    )
)


# ---------------------------------------------------------
# Extract diagonal values.
#
# Each destination should have similarity approximately 1
# with itself.
# ---------------------------------------------------------

self_similarity = np.diag(
    similarity_array
)


# ---------------------------------------------------------
# Extract only the upper triangle.
#
# This avoids counting each destination pair twice.
# ---------------------------------------------------------

upper_triangle_indices = np.triu_indices(
    n_destinations,
    k=1
)


pairwise_similarity = (
    similarity_array[
        upper_triangle_indices
    ]
)


# ---------------------------------------------------------
# Calculate summary statistics.
# ---------------------------------------------------------

minimum_similarity = (
    pairwise_similarity.min()
)

maximum_similarity = (
    pairwise_similarity.max()
)

average_similarity = (
    pairwise_similarity.mean()
)

median_similarity = (
    np.median(
        pairwise_similarity
    )
)


# ---------------------------------------------------------
# Self-similarity validation.
# ---------------------------------------------------------

maximum_self_error = np.max(
    np.abs(
        self_similarity - 1.0
    )
)


# ---------------------------------------------------------
# Symmetry validation.
# ---------------------------------------------------------

symmetry_error = np.max(
    np.abs(
        similarity_array
        -
        similarity_array.T
    )
)


print(
    "\nNumber of destinations:",
    n_destinations
)

print(
    "Number of unique destination pairs:",
    len(pairwise_similarity)
)

print(
    "\nSimilarity statistics:"
)

print(
    f"Minimum pair similarity: "
    f"{minimum_similarity:.6f}"
)

print(
    f"Maximum pair similarity: "
    f"{maximum_similarity:.6f}"
)

print(
    f"Average pair similarity: "
    f"{average_similarity:.6f}"
)

print(
    f"Median pair similarity: "
    f"{median_similarity:.6f}"
)

print(
    "\nSelf-similarity:"
)

print(
    f"Minimum self-similarity: "
    f"{self_similarity.min():.15f}"
)

print(
    f"Maximum self-similarity: "
    f"{self_similarity.max():.15f}"
)

print(
    f"Maximum self-similarity error: "
    f"{maximum_self_error:.15f}"
)

print(
    "\nMatrix symmetry error:",
    f"{symmetry_error:.15f}"
)


# ---------------------------------------------------------
# Validate self-similarity.
# ---------------------------------------------------------

if maximum_self_error > 1e-10:

    raise ValueError(
        "Self-similarity validation failed."
    )


# ---------------------------------------------------------
# Validate symmetry.
# ---------------------------------------------------------

if symmetry_error > 1e-10:

    raise ValueError(
        "Similarity matrix symmetry validation failed."
    )


# ---------------------------------------------------------
# Find the strongest destination pairs.
#
# These pairs represent destinations that are most similar
# according to the learned PCA representation.
# ---------------------------------------------------------

sorted_pair_indices = np.argsort(
    pairwise_similarity
)[::-1]


print("\n" + "=" * 60)
print("TOP 10 MOST SIMILAR DESTINATION PAIRS")
print("=" * 60)


shown_pairs = 0


for flat_index in sorted_pair_indices:

    row_index = (
        upper_triangle_indices[0][
            flat_index
        ]
    )

    column_index = (
        upper_triangle_indices[1][
            flat_index
        ]
    )

    destination_a = (
        recommendation_destinations[
            row_index
        ]
    )

    destination_b = (
        recommendation_destinations[
            column_index
        ]
    )

    score = (
        similarity_array[
            row_index,
            column_index
        ]
    )

    shown_pairs += 1

    print(
        f"{shown_pairs:2d}. "
        f"{destination_a} ↔ "
        f"{destination_b} : "
        f"{score:.4f}"
    )

    if shown_pairs >= 10:

        break


# ---------------------------------------------------------
# Find the weakest destination pairs.
# ---------------------------------------------------------

weakest_pair_indices = np.argsort(
    pairwise_similarity
)


print("\n" + "=" * 60)
print("BOTTOM 10 DESTINATION PAIRS")
print("=" * 60)


shown_pairs = 0


for flat_index in weakest_pair_indices:

    row_index = (
        upper_triangle_indices[0][
            flat_index
        ]
    )

    column_index = (
        upper_triangle_indices[1][
            flat_index
        ]
    )

    destination_a = (
        recommendation_destinations[
            row_index
        ]
    )

    destination_b = (
        recommendation_destinations[
            column_index
        ]
    )

    score = (
        similarity_array[
            row_index,
            column_index
        ]
    )

    shown_pairs += 1

    print(
        f"{shown_pairs:2d}. "
        f"{destination_a} ↔ "
        f"{destination_b} : "
        f"{score:.4f}"
    )

    if shown_pairs >= 10:

        break


# ---------------------------------------------------------
# Final evaluation result.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("✓ SIMILARITY MODEL QUALITY EVALUATION PASSED")
print("=" * 60)

SIMILARITY MODEL QUALITY EVALUATION

Number of destinations: 50
Number of unique destination pairs: 1225

Similarity statistics:
Minimum pair similarity: -0.709428
Maximum pair similarity: 0.901214
Average pair similarity: -0.013805
Median pair similarity: -0.052668

Self-similarity:
Minimum self-similarity: 1.000000000000000
Maximum self-similarity: 1.000000000000000
Maximum self-similarity error: 0.000000000000000

Matrix symmetry error: 0.000000000000000

TOP 10 MOST SIMILAR DESTINATION PAIRS
 1. Jaisalmer ↔ Jodhpur : 0.9012
 2. Darjeeling ↔ Nainital : 0.8669
 3. Kolkata ↔ Varanasi : 0.8473
 4. Amritsar ↔ Jodhpur : 0.8052
 5. Srinagar ↔ Udaipur : 0.7918
 6. Coorg ↔ Kodaikanal : 0.7918
 7. Bhubaneswar ↔ Ranchi : 0.7674
 8. Gangtok ↔ Ranchi : 0.7629
 9. Bengaluru ↔ Mumbai : 0.7581
10. Ladakh ↔ Pahalgam : 0.7562

BOTTOM 10 DESTINATION PAIRS
 1. Chennai ↔ Mussoorie : -0.7094
 2. Chennai ↔ Pahalgam : -0.6859
 3. Kochi ↔ Shimla : -0.6303
 4. Kodaikanal ↔ Kolkata : -0.5859
 5. Andaman ↔ Ja

In [7]:
# ---------------------------------------------------------
# REBUILD NORMALIZED PREFERENCE FEATURES
#
# 08_Model_Evaluation.ipynb is a fresh notebook, so variables
# created in 07_Model_Building.ipynb are not available here.
#
# We therefore reconstruct the normalized preference matrix
# using the saved dataset and saved preference configuration.
#
# This does NOT retrain or modify the model.
# ---------------------------------------------------------

print("=" * 60)
print("REBUILDING PREFERENCE FEATURES FOR EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Load the processed dataset.
#
# This contains the original travel features required for
# evaluating user-dependent recommendations.
# ---------------------------------------------------------

processed_df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "integrated_travel_dataset.csv"
    )
)


print(
    "\nProcessed dataset:",
    processed_df.shape
)


# ---------------------------------------------------------
# Collect every feature used by the preference groups.
# ---------------------------------------------------------

all_preference_features = []

for features in preference_groups.values():

    all_preference_features.extend(
        features
    )


# Remove duplicates while preserving the original order.

all_preference_features = list(
    dict.fromkeys(
        all_preference_features
    )
)


print(
    "Preference features:",
    len(
        all_preference_features
    )
)


# ---------------------------------------------------------
# Make sure every required preference feature exists.
# ---------------------------------------------------------

missing_features = [

    feature

    for feature
    in all_preference_features

    if feature not in processed_df.columns
]


if missing_features:

    raise ValueError(
        "Missing preference features: "
        + str(missing_features)
    )


# ---------------------------------------------------------
# Normalization function.
#
# Each feature is independently converted to [0, 1].
#
# Missing values remain NaN because unavailable external
# information must be handled through availability-aware
# scoring.
# ---------------------------------------------------------

def normalize_for_evaluation(
    series
):

    values = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = values.min()

    maximum = values.max()


    # No usable values.

    if pd.isna(
        minimum
    ) or pd.isna(
        maximum
    ):

        return pd.Series(
            np.nan,
            index=series.index
        )


    # Constant feature.

    if maximum == minimum:

        result = pd.Series(
            np.nan,
            index=series.index
        )

        result.loc[
            values.notna()
        ] = 0.5

        return result


    return (
        values - minimum
    ) / (
        maximum - minimum
    )


# ---------------------------------------------------------
# Create normalized preference matrix.
# ---------------------------------------------------------

normalized_features = pd.DataFrame(
    index=processed_df.index
)


for feature in all_preference_features:

    normalized_features[
        feature
    ] = normalize_for_evaluation(
        processed_df[
            feature
        ]
    )


# ---------------------------------------------------------
# Apply preference directions.
#
# After this step:
#
#     larger value = better
#
# for every preference feature.
# ---------------------------------------------------------

for feature in all_preference_features:

    direction = (
        preference_direction_map[
            feature
        ]
    )


    if direction == "lower":

        valid_mask = (
            normalized_features[
                feature
            ].notna()
        )

        normalized_features.loc[
            valid_mask,
            feature
        ] = (

            1
            -
            normalized_features.loc[
                valid_mask,
                feature
            ]
        )


# ---------------------------------------------------------
# Validate the reconstructed matrix.
# ---------------------------------------------------------

available_values = (
    normalized_features
    .stack()
)


minimum_value = (
    available_values.min()
)

maximum_value = (
    available_values.max()
)


print(
    "\nNormalized feature range:"
)

print(
    f"{minimum_value:.6f}"
    f" → "
    f"{maximum_value:.6f}"
)


if (
    minimum_value < 0
    or
    maximum_value > 1
):

    raise ValueError(
        "Normalized features are outside [0, 1]."
    )


print(
    "\nMissing values by preference feature:"
)

missing_summary = (
    normalized_features
    .isna()
    .sum()
)

print(
    missing_summary[
        missing_summary > 0
    ]
)


print("\n" + "=" * 60)
print("✓ PREFERENCE FEATURES REBUILT")
print("=" * 60)

REBUILDING PREFERENCE FEATURES FOR EVALUATION

Processed dataset: (50, 42)
Preference features: 29

Normalized feature range:
0.000000 → 1.000000

Missing values by preference feature:
avg_flight_price      42
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
flight_count          42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
hotel_count           11
room_count            11
avg_allotment         11
dtype: int64

✓ PREFERENCE FEATURES REBUILT


In [8]:
# ---------------------------------------------------------
# PERSONALIZATION EVALUATION
#
# This test checks whether different users receive different
# destination rankings.
# ---------------------------------------------------------

print("=" * 60)
print("PERSONALIZATION EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Define realistic user personas.
# ---------------------------------------------------------

user_personas = {

    "Budget Nature Traveler": {

        "preferences": {
            "budget": 1.00,
            "flight": 0.50,
            "accommodation": 0.70,
            "weather": 0.80,
            "destination_characteristics": 1.00
        },

        "interests": {
            "nature": 1.00,
            "sightseeing": 0.30,
            "water_coastal": 0.20,
            "wildlife": 0.80
        }
    },


    "City Sightseeing Traveler": {

        "preferences": {
            "budget": 0.50,
            "flight": 0.80,
            "accommodation": 0.60,
            "weather": 0.40,
            "destination_characteristics": 0.70
        },

        "interests": {
            "nature": 0.20,
            "sightseeing": 1.00,
            "water_coastal": 0.10,
            "wildlife": 0.10
        }
    },


    "Beach Traveler": {

        "preferences": {
            "budget": 0.60,
            "flight": 0.70,
            "accommodation": 0.70,
            "weather": 0.90,
            "destination_characteristics": 0.80
        },

        "interests": {
            "nature": 0.40,
            "sightseeing": 0.30,
            "water_coastal": 1.00,
            "wildlife": 0.20
        }
    },


    "Wildlife Traveler": {

        "preferences": {
            "budget": 0.50,
            "flight": 0.50,
            "accommodation": 0.50,
            "weather": 0.70,
            "destination_characteristics": 1.00
        },

        "interests": {
            "nature": 0.80,
            "sightseeing": 0.20,
            "water_coastal": 0.10,
            "wildlife": 1.00
        }
    }
}


# ---------------------------------------------------------
# Function for personalized preference scoring.
# ---------------------------------------------------------

def calculate_persona_preference_score(
    user_preferences
):

    weighted_total = pd.Series(
        0.0,
        index=normalized_features.index
    )

    active_total = pd.Series(
        0.0,
        index=normalized_features.index
    )


    for group_name, features in (
        preference_groups.items()
    ):

        group_weight = (
            user_preferences[
                group_name
            ]
        )

        group_total = pd.Series(
            0.0,
            index=normalized_features.index
        )

        group_active = pd.Series(
            0.0,
            index=normalized_features.index
        )


        for feature in features:

            values = (
                normalized_features[
                    feature
                ]
            )

            valid_mask = (
                values.notna()
            )


            group_total.loc[
                valid_mask
            ] += (
                values.loc[
                    valid_mask
                ]
                * group_weight
            )


            group_active.loc[
                valid_mask
            ] += group_weight


        valid_group = (
            group_active > 0
        )


        group_score = pd.Series(
            np.nan,
            index=normalized_features.index
        )


        group_score.loc[
            valid_group
        ] = (

            group_total.loc[
                valid_group
            ]
            /
            group_active.loc[
                valid_group
            ]
        )


        valid_mask = (
            group_score.notna()
        )


        weighted_total.loc[
            valid_mask
        ] += (

            group_score.loc[
                valid_mask
            ]
            * group_weight
        )


        active_total.loc[
            valid_mask
        ] += group_weight


    return (
        weighted_total
        /
        active_total
    )


# ---------------------------------------------------------
# Function for personalized interest scoring.
# ---------------------------------------------------------

def calculate_persona_interest_score(
    user_interests
):

    total_weight = sum(
        user_interests.values()
    )

    score = pd.Series(
        0.0,
        index=destination_profile_scores.index
    )


    for profile, weight in (
        user_interests.items()
    ):

        column = (
            f"{profile}_score"
        )

        score += (

            destination_profile_scores[
                column
            ].fillna(0)

            * weight
        )


    return (
        score
        /
        total_weight
    )


# ---------------------------------------------------------
# Calculate rankings for each persona.
# ---------------------------------------------------------

persona_rankings = {}


for persona_name, persona in (
    user_personas.items()
):

    preference_score = (
        calculate_persona_preference_score(
            persona[
                "preferences"
            ]
        )
    )


    interest_score = (
        calculate_persona_interest_score(
            persona[
                "interests"
            ]
        )
    )


    # -----------------------------------------------------
    # Combine travel preferences and interests.
    # -----------------------------------------------------

    travel_weight = (
        recommendation_config[
            "travel_factor_weight"
        ]
    )

    interest_weight = (
        recommendation_config[
            "interest_weight"
        ]
    )


    final_score = (

        preference_score
        * travel_weight

        +

        interest_score
        * interest_weight
    )


    ranking = pd.DataFrame({

        "destination":
            recommendation_destinations,

        "score":
            final_score.values
    })


    ranking = (
        ranking
        .sort_values(
            "score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    ranking.insert(
        0,
        "rank",
        range(
            1,
            len(ranking) + 1
        )
    )


    persona_rankings[
        persona_name
    ] = ranking


# ---------------------------------------------------------
# Display top 5 recommendations for each persona.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "TOP 5 RECOMMENDATIONS BY USER PERSONA"
)

print(
    "=" * 60
)


for persona_name, ranking in (
    persona_rankings.items()
):

    print(
        f"\n{persona_name}"
    )

    print(
        ranking[
            [
                "rank",
                "destination",
                "score"
            ]
        ]
        .head(5)
        .to_string(
            index=False
        )
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ PERSONALIZATION EVALUATION COMPLETED"
)

print(
    "=" * 60
)

PERSONALIZATION EVALUATION

TOP 5 RECOMMENDATIONS BY USER PERSONA

Budget Nature Traveler
 rank destination    score
    1      Mumbai 0.644650
    2      Manali 0.579520
    3   Bengaluru 0.557790
    4       Delhi 0.555292
    5      Jaipur 0.536027

City Sightseeing Traveler
 rank destination    score
    1      Mumbai 0.732322
    2   Bengaluru 0.697689
    3       Delhi 0.679851
    4     Chennai 0.633583
    5        Pune 0.623697

Beach Traveler
 rank destination    score
    1      Mumbai 0.587221
    2     Chennai 0.577412
    3       Delhi 0.554550
    4   Bengaluru 0.550725
    5       Kochi 0.539218

Wildlife Traveler
 rank destination    score
    1      Mumbai 0.630006
    2      Manali 0.560761
    3       Delhi 0.525434
    4   Bengaluru 0.515061
    5      Jaipur 0.512081

✓ PERSONALIZATION EVALUATION COMPLETED


In [9]:
# ---------------------------------------------------------
# PERSONALIZATION DIFFERENCE EVALUATION
#
# The previous test showed that different user profiles
# produce different rankings.
#
# Now we quantify that difference using:
#
#   1. Top-5 overlap
#   2. Top-10 overlap
#   3. Spearman rank correlation
#
# Lower overlap / lower correlation means stronger
# personalization differences.
# ---------------------------------------------------------

from itertools import combinations

from scipy.stats import spearmanr


print("=" * 60)
print("PERSONALIZATION DIFFERENCE EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Store persona names.
# ---------------------------------------------------------

persona_names = list(
    persona_rankings.keys()
)


# ---------------------------------------------------------
# Calculate pairwise comparison metrics.
# ---------------------------------------------------------

comparison_results = []


for persona_a, persona_b in combinations(
    persona_names,
    2
):

    ranking_a = (
        persona_rankings[
            persona_a
        ]
    )

    ranking_b = (
        persona_rankings[
            persona_b
        ]
    )


    # -----------------------------------------------------
    # Top-5 destination sets.
    # -----------------------------------------------------

    top5_a = set(
        ranking_a[
            "destination"
        ].head(5)
    )

    top5_b = set(
        ranking_b[
            "destination"
        ].head(5)
    )


    top5_overlap = len(
        top5_a.intersection(
            top5_b
        )
    )


    # -----------------------------------------------------
    # Top-10 destination sets.
    # -----------------------------------------------------

    top10_a = set(
        ranking_a[
            "destination"
        ].head(10)
    )

    top10_b = set(
        ranking_b[
            "destination"
        ].head(10)
    )


    top10_overlap = len(
        top10_a.intersection(
            top10_b
        )
    )


    # -----------------------------------------------------
    # Create destination → rank mappings.
    #
    # This lets us compare the complete ranking rather than
    # only the top destinations.
    # -----------------------------------------------------

    ranks_a = dict(
        zip(
            ranking_a[
                "destination"
            ],
            ranking_a[
                "rank"
            ]
        )
    )


    ranks_b = dict(
        zip(
            ranking_b[
                "destination"
            ],
            ranking_b[
                "rank"
            ]
        )
    )


    common_destinations = sorted(
        set(ranks_a)
        &
        set(ranks_b)
    )


    rank_values_a = [
        ranks_a[destination]
        for destination
        in common_destinations
    ]


    rank_values_b = [
        ranks_b[destination]
        for destination
        in common_destinations
    ]


    # -----------------------------------------------------
    # Spearman rank correlation.
    #
    # +1 → rankings are identical
    #  0 → little relationship
    # -1 → rankings are reversed
    # -----------------------------------------------------

    correlation, _ = spearmanr(
        rank_values_a,
        rank_values_b
    )


    comparison_results.append({

        "persona_a":
            persona_a,

        "persona_b":
            persona_b,

        "top5_overlap":
            top5_overlap,

        "top10_overlap":
            top10_overlap,

        "spearman_rank_correlation":
            correlation
    })


# ---------------------------------------------------------
# Convert comparisons to DataFrame.
# ---------------------------------------------------------

personalization_comparison_df = pd.DataFrame(
    comparison_results
)


# ---------------------------------------------------------
# Display pairwise results.
# ---------------------------------------------------------

print(
    "\nPairwise personalization comparison:"
)

print(
    personalization_comparison_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Calculate average metrics.
# ---------------------------------------------------------

average_top5_overlap = (
    personalization_comparison_df[
        "top5_overlap"
    ].mean()
)


average_top10_overlap = (
    personalization_comparison_df[
        "top10_overlap"
    ].mean()
)


average_correlation = (
    personalization_comparison_df[
        "spearman_rank_correlation"
    ].mean()
)


print(
    "\n" + "=" * 60
)

print(
    "PERSONALIZATION SUMMARY"
)

print(
    "=" * 60
)


print(
    f"\nAverage Top-5 overlap: "
    f"{average_top5_overlap:.2f} / 5"
)


print(
    f"Average Top-10 overlap: "
    f"{average_top10_overlap:.2f} / 10"
)


print(
    f"Average Spearman correlation: "
    f"{average_correlation:.4f}"
)


# ---------------------------------------------------------
# Check whether there is at least some ranking variation.
# ---------------------------------------------------------

unique_top5_sets = {

    tuple(
        ranking[
            "destination"
        ].head(5)
    )

    for ranking
    in persona_rankings.values()
}


if len(unique_top5_sets) <= 1:

    raise ValueError(
        "All personas produced identical Top-5 rankings."
    )


print(
    "\nNumber of unique Top-5 recommendation lists:",
    len(unique_top5_sets)
)


print(
    "\n" + "=" * 60
)

print(
    "✓ PERSONALIZATION DIFFERENCE EVALUATION PASSED"
)

print(
    "=" * 60
)

PERSONALIZATION DIFFERENCE EVALUATION

Pairwise personalization comparison:
                persona_a                 persona_b  top5_overlap  top10_overlap  spearman_rank_correlation
   Budget Nature Traveler City Sightseeing Traveler             3              8                   0.921152
   Budget Nature Traveler            Beach Traveler             3              8                   0.940648
   Budget Nature Traveler         Wildlife Traveler             5              9                   0.983385
City Sightseeing Traveler            Beach Traveler             4              9                   0.958703
City Sightseeing Traveler         Wildlife Traveler             3              8                   0.876879
           Beach Traveler         Wildlife Traveler             3              8                   0.915390

PERSONALIZATION SUMMARY

Average Top-5 overlap: 3.50 / 5
Average Top-10 overlap: 8.33 / 10
Average Spearman correlation: 0.9327

Number of unique Top-5 recommendation 

In [10]:
# ---------------------------------------------------------
# CONTROLLED PREFERENCE SENSITIVITY EVALUATION
#
# This test changes ONE user preference at a time while
# keeping all other preferences fixed.
#
# The purpose is to verify that the recommendation system
# responds to individual user preferences.
# ---------------------------------------------------------

print("=" * 60)
print("CONTROLLED PREFERENCE SENSITIVITY EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Base user profile.
#
# All sensitivity experiments start from this profile.
# ---------------------------------------------------------

base_preferences = {

    "budget": 0.60,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.60,

    "destination_characteristics": 0.60
}


base_interests = {

    "nature": 0.60,

    "sightseeing": 0.60,

    "water_coastal": 0.60,

    "wildlife": 0.60
}


# ---------------------------------------------------------
# Define controlled experiments.
#
# Each experiment changes only ONE variable.
# ---------------------------------------------------------

sensitivity_tests = {

    "Low Budget Importance": {

        "type": "preference",

        "feature": "budget",

        "value": 0.10
    },


    "High Budget Importance": {

        "type": "preference",

        "feature": "budget",

        "value": 1.00
    },


    "Low Nature Interest": {

        "type": "interest",

        "feature": "nature",

        "value": 0.10
    },


    "High Nature Interest": {

        "type": "interest",

        "feature": "nature",

        "value": 1.00
    },


    "Low Water Interest": {

        "type": "interest",

        "feature": "water_coastal",

        "value": 0.10
    },


    "High Water Interest": {

        "type": "interest",

        "feature": "water_coastal",

        "value": 1.00
    },


    "Low Sightseeing Interest": {

        "type": "interest",

        "feature": "sightseeing",

        "value": 0.10
    },


    "High Sightseeing Interest": {

        "type": "interest",

        "feature": "sightseeing",

        "value": 1.00
    },


    "Low Wildlife Interest": {

        "type": "interest",

        "feature": "wildlife",

        "value": 0.10
    },


    "High Wildlife Interest": {

        "type": "interest",

        "feature": "wildlife",

        "value": 1.00
    }
}


# ---------------------------------------------------------
# Function to generate a ranking for any user profile.
#
# This uses the same validated scoring logic from the
# personalization evaluation.
# ---------------------------------------------------------

def generate_profile_ranking(
    preferences,
    interests
):

    preference_score = (
        calculate_persona_preference_score(
            preferences
        )
    )


    interest_score = (
        calculate_persona_interest_score(
            interests
        )
    )


    final_score = (

        preference_score
        * recommendation_config[
            "travel_factor_weight"
        ]

        +

        interest_score
        * recommendation_config[
            "interest_weight"
        ]
    )


    ranking = pd.DataFrame({

        "destination":
            recommendation_destinations,

        "score":
            final_score.values
    })


    return (
        ranking
        .sort_values(
            "score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


# ---------------------------------------------------------
# Generate the baseline ranking.
# ---------------------------------------------------------

baseline_ranking = (
    generate_profile_ranking(
        base_preferences,
        base_interests
    )
)


# ---------------------------------------------------------
# Store all experiment rankings.
# ---------------------------------------------------------

sensitivity_rankings = {}


for test_name, test in (
    sensitivity_tests.items()
):

    test_preferences = (
        base_preferences.copy()
    )

    test_interests = (
        base_interests.copy()
    )


    # -----------------------------------------------------
    # Change only the selected variable.
    # -----------------------------------------------------

    if test["type"] == "preference":

        test_preferences[
            test["feature"]
        ] = test["value"]


    else:

        test_interests[
            test["feature"]
        ] = test["value"]


    ranking = (
        generate_profile_ranking(
            test_preferences,
            test_interests
        )
    )


    sensitivity_rankings[
        test_name
    ] = ranking


# ---------------------------------------------------------
# Display baseline ranking.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "BASELINE USER"
)

print(
    "=" * 60
)

print(
    baseline_ranking[
        [
            "destination",
            "score"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Display sensitivity experiment results.
# ---------------------------------------------------------

for test_name, ranking in (
    sensitivity_rankings.items()
):

    print(
        "\n" + "-" * 60
    )

    print(
        test_name
    )

    print(
        "-" * 60
    )

    print(
        ranking[
            [
                "destination",
                "score"
            ]
        ]
        .head(5)
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# Calculate Top-5 changes.
#
# This measures how many destinations changed when only one
# preference was modified.
# ---------------------------------------------------------

baseline_top5 = set(
    baseline_ranking[
        "destination"
    ].head(5)
)


sensitivity_results = []


for test_name, ranking in (
    sensitivity_rankings.items()
):

    test_top5 = set(
        ranking[
            "destination"
        ].head(5)
    )


    overlap = len(
        baseline_top5.intersection(
            test_top5
        )
    )


    changed_destinations = (
        5 - overlap
    )


    sensitivity_results.append({

        "test":
            test_name,

        "top5_overlap":
            overlap,

        "top5_destinations_changed":
            changed_destinations
    })


# ---------------------------------------------------------
# Create summary table.
# ---------------------------------------------------------

sensitivity_summary = pd.DataFrame(
    sensitivity_results
)


print(
    "\n" + "=" * 60
)

print(
    "SENSITIVITY SUMMARY"
)

print(
    "=" * 60
)

print(
    sensitivity_summary.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Verify that at least one controlled change actually
# changes the Top-5 recommendations.
# ---------------------------------------------------------

if (
    sensitivity_summary[
        "top5_destinations_changed"
    ].max()
    <= 0
):

    raise ValueError(
        "Changing individual preferences did not change "
        "any Top-5 recommendations."
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ CONTROLLED PREFERENCE SENSITIVITY PASSED"
)

print(
    "=" * 60
)

CONTROLLED PREFERENCE SENSITIVITY EVALUATION

BASELINE USER
       destination    score
            Mumbai 0.647865
             Delhi 0.586939
         Bengaluru 0.585826
           Chennai 0.570248
              Pune 0.547658
             Kochi 0.546749
           Kolkata 0.541913
            Jaipur 0.532828
            Manali 0.524101
Thiruvananthapuram 0.504213

------------------------------------------------------------
Low Budget Importance
------------------------------------------------------------
destination    score
     Mumbai 0.627847
      Delhi 0.568996
  Bengaluru 0.554014
    Chennai 0.527080
     Jaipur 0.510357

------------------------------------------------------------
High Budget Importance
------------------------------------------------------------
destination    score
     Mumbai 0.659641
  Bengaluru 0.604539
      Delhi 0.597494
    Chennai 0.593682
      Kochi 0.574255

------------------------------------------------------------
Low Nature Interest
-------

In [11]:
# ---------------------------------------------------------
# PREFERENCE INFLUENCE ANALYSIS
#
# This evaluation measures how strongly each individual
# user preference can change destination scores.
#
# For every tested preference:
#
#     baseline value = 0.60
#     changed value  = 1.00
#
# We then measure:
#
#     score change = changed score - baseline score
#
# A larger positive change means that the destination is
# more sensitive to that preference.
# ---------------------------------------------------------

print("=" * 60)
print("PREFERENCE INFLUENCE ANALYSIS")
print("=" * 60)


# ---------------------------------------------------------
# Define the baseline user.
# ---------------------------------------------------------

baseline_preferences = {

    "budget": 0.60,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.60,

    "destination_characteristics": 0.60
}


baseline_interests = {

    "nature": 0.60,

    "sightseeing": 0.60,

    "water_coastal": 0.60,

    "wildlife": 0.60
}


# ---------------------------------------------------------
# Calculate the baseline ranking and scores.
# ---------------------------------------------------------

baseline_ranking = (
    generate_profile_ranking(
        baseline_preferences,
        baseline_interests
    )
)


baseline_scores = baseline_ranking.set_index(
    "destination"
)[
    "score"
]


# ---------------------------------------------------------
# Preferences and interests to analyze.
# ---------------------------------------------------------

influence_tests = [

    ("preference", "budget"),

    ("preference", "flight"),

    ("preference", "accommodation"),

    ("preference", "weather"),

    ("preference", "destination_characteristics"),

    ("interest", "nature"),

    ("interest", "sightseeing"),

    ("interest", "water_coastal"),

    ("interest", "wildlife")
]


influence_results = []


# ---------------------------------------------------------
# Test each variable independently.
# ---------------------------------------------------------

for variable_type, feature in influence_tests:

    # -----------------------------------------------------
    # Copy the baseline profile.
    # -----------------------------------------------------

    test_preferences = (
        baseline_preferences.copy()
    )

    test_interests = (
        baseline_interests.copy()
    )


    # -----------------------------------------------------
    # Increase only the selected variable.
    # -----------------------------------------------------

    if variable_type == "preference":

        test_preferences[
            feature
        ] = 1.0

    else:

        test_interests[
            feature
        ] = 1.0


    # -----------------------------------------------------
    # Generate the changed-user ranking.
    # -----------------------------------------------------

    changed_ranking = (
        generate_profile_ranking(
            test_preferences,
            test_interests
        )
    )


    changed_scores = (
        changed_ranking
        .set_index(
            "destination"
        )[
            "score"
        ]
    )


    # -----------------------------------------------------
    # Calculate score changes.
    # -----------------------------------------------------

    score_change = (
        changed_scores
        -
        baseline_scores
    )


    # -----------------------------------------------------
    # Find the strongest positive effects.
    # -----------------------------------------------------

    strongest_positive = (
        score_change
        .sort_values(
            ascending=False
        )
        .head(5)
    )


    # -----------------------------------------------------
    # Calculate average absolute influence.
    #
    # This measures how strongly the entire destination
    # ranking responds to this variable.
    # -----------------------------------------------------

    mean_absolute_change = (
        score_change
        .abs()
        .mean()
    )


    maximum_change = (
        score_change
        .abs()
        .max()
    )


    influence_results.append({

        "type":
            variable_type,

        "feature":
            feature,

        "mean_absolute_score_change":
            mean_absolute_change,

        "maximum_score_change":
            maximum_change,

        "strongest_destination":
            strongest_positive.index[0],

        "strongest_positive_change":
            strongest_positive.iloc[0]
    })


    # -----------------------------------------------------
    # Display the strongest affected destinations.
    # -----------------------------------------------------

    print(
        "\n" + "-" * 60
    )

    print(
        f"{feature.upper()} "
        f"({variable_type})"
    )

    print(
        "-" * 60
    )

    print(
        "Top destinations positively affected:"
    )


    for destination, change in (
        strongest_positive.items()
    ):

        print(
            f"  {destination:25s}"
            f"{change:+.6f}"
        )


    print(
        f"\nMean absolute score change: "
        f"{mean_absolute_change:.6f}"
    )


    print(
        f"Maximum score change: "
        f"{maximum_change:.6f}"
    )


# ---------------------------------------------------------
# Create influence summary.
# ---------------------------------------------------------

influence_summary = pd.DataFrame(
    influence_results
)


# ---------------------------------------------------------
# Sort by overall influence.
# ---------------------------------------------------------

influence_summary = (
    influence_summary
    .sort_values(
        "mean_absolute_score_change",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n" + "=" * 60
)

print(
    "OVERALL PREFERENCE INFLUENCE"
)

print(
    "=" * 60
)


print(
    influence_summary.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Validation.
# ---------------------------------------------------------

if (
    influence_summary[
        "mean_absolute_score_change"
    ].max()
    <= 0
):

    raise ValueError(
        "No user preference produced any score change."
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ PREFERENCE INFLUENCE ANALYSIS PASSED"
)

print(
    "=" * 60
)

PREFERENCE INFLUENCE ANALYSIS

------------------------------------------------------------
BUDGET (preference)
------------------------------------------------------------
Top destinations positively affected:
  Bhopal                   +0.039387
  Nainital                 +0.039328
  Gangtok                  +0.037919
  Darjeeling               +0.035553
  Bhubaneswar              +0.035539

Mean absolute score change: 0.019949
Maximum score change: 0.039387

------------------------------------------------------------
FLIGHT (preference)
------------------------------------------------------------
Top destinations positively affected:
  Goa                      +0.011231
  Bengaluru                +0.010974
  Delhi                    +0.007878
  Mumbai                   +0.005778
  Jaipur                   +0.000288

Mean absolute score change: 0.001902
Maximum score change: 0.030047

------------------------------------------------------------
ACCOMMODATION (preference)
-----------

In [12]:
# ---------------------------------------------------------
# PREFERENCE DIRECTION & EXPLAINABILITY TEST
#
# Purpose:
# Verify that increasing a user's preference for a feature
# actually rewards destinations that score highly on that
# feature.
#
# This is a sanity check for the recommendation logic.
# ---------------------------------------------------------

print("=" * 60)
print("PREFERENCE DIRECTION & EXPLAINABILITY TEST")
print("=" * 60)


# ---------------------------------------------------------
# Use a neutral baseline user.
#
# Every preference and interest starts at 0.60.
# ---------------------------------------------------------

baseline_preferences = {

    "budget": 0.60,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.60,

    "destination_characteristics": 0.60
}


baseline_interests = {

    "nature": 0.60,

    "sightseeing": 0.60,

    "water_coastal": 0.60,

    "wildlife": 0.60
}


# ---------------------------------------------------------
# Calculate baseline scores.
# ---------------------------------------------------------

baseline_preference_score = (
    calculate_persona_preference_score(
        baseline_preferences
    )
)


baseline_interest_score = (
    calculate_persona_interest_score(
        baseline_interests
    )
)


baseline_final_score = (

    baseline_preference_score
    * recommendation_config[
        "travel_factor_weight"
    ]

    +

    baseline_interest_score
    * recommendation_config[
        "interest_weight"
    ]
)


# ---------------------------------------------------------
# Define the variables to test.
# ---------------------------------------------------------

variables_to_test = [

    ("preference", "budget"),

    ("preference", "flight"),

    ("preference", "accommodation"),

    ("preference", "weather"),

    (
        "preference",
        "destination_characteristics"
    ),

    ("interest", "nature"),

    ("interest", "sightseeing"),

    ("interest", "water_coastal"),

    ("interest", "wildlife")
]


# ---------------------------------------------------------
# Store results.
# ---------------------------------------------------------

direction_results = []


# ---------------------------------------------------------
# Test each variable.
# ---------------------------------------------------------

for variable_type, feature in variables_to_test:

    # -----------------------------------------------------
    # Create a copy of the neutral user.
    # -----------------------------------------------------

    test_preferences = (
        baseline_preferences.copy()
    )

    test_interests = (
        baseline_interests.copy()
    )


    # -----------------------------------------------------
    # Increase ONLY this variable.
    # -----------------------------------------------------

    if variable_type == "preference":

        test_preferences[
            feature
        ] = 1.00

    else:

        test_interests[
            feature
        ] = 1.00


    # -----------------------------------------------------
    # Calculate changed-user scores.
    # -----------------------------------------------------

    changed_preference_score = (
        calculate_persona_preference_score(
            test_preferences
        )
    )


    changed_interest_score = (
        calculate_persona_interest_score(
            test_interests
        )
    )


    changed_final_score = (

        changed_preference_score
        * recommendation_config[
            "travel_factor_weight"
        ]

        +

        changed_interest_score
        * recommendation_config[
            "interest_weight"
        ]
    )


    # -----------------------------------------------------
    # Calculate how much every destination changed.
    # -----------------------------------------------------

    score_change = (
        changed_final_score
        -
        baseline_final_score
    )


    # -----------------------------------------------------
    # Find the destinations receiving the largest positive
    # and negative changes.
    # -----------------------------------------------------

    largest_positive = (
        score_change
        .sort_values(
            ascending=False
        )
        .iloc[0]
    )


    largest_positive_destination = (
        score_change
        .sort_values(
            ascending=False
        )
        .index[0]
    )


    largest_negative = (
        score_change
        .sort_values(
            ascending=True
        )
        .iloc[0]
    )


    largest_negative_destination = (
        score_change
        .sort_values(
            ascending=True
        )
        .index[0]
    )


    # -----------------------------------------------------
    # Calculate overall influence.
    # -----------------------------------------------------

    mean_change = (
        score_change.mean()
    )


    mean_absolute_change = (
        score_change.abs().mean()
    )


    # -----------------------------------------------------
    # Store results.
    # -----------------------------------------------------

    direction_results.append({

        "type":
            variable_type,

        "feature":
            feature,

        "mean_score_change":
            mean_change,

        "mean_absolute_change":
            mean_absolute_change,

        "largest_positive_destination":
            largest_positive_destination,

        "largest_positive_change":
            largest_positive,

        "largest_negative_destination":
            largest_negative_destination,

        "largest_negative_change":
            largest_negative
    })


# ---------------------------------------------------------
# Create summary DataFrame.
# ---------------------------------------------------------

direction_summary = pd.DataFrame(
    direction_results
)


# ---------------------------------------------------------
# Display results.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "INDIVIDUAL PREFERENCE EFFECTS"
)

print(
    "=" * 60
)


for _, row in (
    direction_summary.iterrows()
):

    print(
        f"\n{row['feature'].upper()}"
        f" ({row['type']})"
    )

    print(
        f"  Mean score change: "
        f"{row['mean_score_change']:+.6f}"
    )

    print(
        f"  Mean absolute change: "
        f"{row['mean_absolute_change']:.6f}"
    )

    print(
        f"  Largest positive: "
        f"{row['largest_positive_destination']}"
        f" ({row['largest_positive_change']:+.6f})"
    )

    print(
        f"  Largest negative: "
        f"{row['largest_negative_destination']}"
        f" ({row['largest_negative_change']:+.6f})"
    )


# ---------------------------------------------------------
# Check that every tested variable has some influence.
# ---------------------------------------------------------

uninfluential_features = (
    direction_summary[
        "mean_absolute_change"
    ] <= 1e-12
)


if uninfluential_features.any():

    problematic = (
        direction_summary.loc[
            uninfluential_features,
            "feature"
        ].tolist()
    )

    raise ValueError(
        "These variables have no measurable influence: "
        + str(problematic)
    )


# ---------------------------------------------------------
# Check that changing the user profile changes the final
# recommendation score for every tested variable.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "INFLUENCE VALIDATION"
)

print(
    "=" * 60
)


for _, row in (
    direction_summary.iterrows()
):

    print(
        f"✓ {row['feature']}: "
        f"influence detected "
        f"({row['mean_absolute_change']:.6f})"
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ PREFERENCE DIRECTION & "
    "EXPLAINABILITY TEST PASSED"
)

print(
    "=" * 60
)

PREFERENCE DIRECTION & EXPLAINABILITY TEST

INDIVIDUAL PREFERENCE EFFECTS

BUDGET (preference)
  Mean score change: +0.019912
  Mean absolute change: 0.019949
  Largest positive: 6 (+0.039387)
  Largest negative: 41 (-0.000933)

FLIGHT (preference)
  Mean score change: -0.000456
  Mean absolute change: 0.001902
  Largest positive: 14 (+0.011231)
  Largest negative: 44 (-0.030047)

ACCOMMODATION (preference)
  Mean score change: -0.000402
  Mean absolute change: 0.003497
  Largest positive: 46 (+0.011181)
  Largest negative: 12 (-0.013099)

WEATHER (preference)
  Mean score change: +0.008649
  Mean absolute change: 0.012891
  Largest positive: 27 (+0.052105)
  Largest negative: 6 (-0.013990)

DESTINATION_CHARACTERISTICS (preference)
  Mean score change: -0.027702
  Mean absolute change: 0.027702
  Largest positive: 44 (-0.007277)
  Largest negative: 27 (-0.052105)

NATURE (interest)
  Mean score change: -0.000321
  Mean absolute change: 0.003305
  Largest positive: 29 (+0.013601)
  Larg

In [13]:
# ---------------------------------------------------------
# RECOMMENDATION QUALITY EVALUATION
#
# Purpose:
# Check whether the destinations ranked highly by the
# recommendation engine actually have stronger alignment
# with the user's preferences than the overall destination
# population.
#
# We compare:
#
#   Top-5 recommendations
#          VS
#   All 50 destinations
#
# across:
#
#   - travel preference score
#   - interest score
#   - combined personalized score
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION QUALITY EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Use the validated sample user profile.
# ---------------------------------------------------------

evaluation_preferences = {

    "budget": 0.90,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.80,

    "destination_characteristics": 0.80
}


evaluation_interests = {

    "nature": 1.00,

    "sightseeing": 0.50,

    "water_coastal": 0.20,

    "wildlife": 0.60
}


# ---------------------------------------------------------
# Generate personalized scores for this user.
# ---------------------------------------------------------

evaluation_preference_score = (
    calculate_persona_preference_score(
        evaluation_preferences
    )
)


evaluation_interest_score = (
    calculate_persona_interest_score(
        evaluation_interests
    )
)


# ---------------------------------------------------------
# Calculate the combined personalized score.
#
# These are the same weights used by the recommendation
# system:
#
#   Travel preferences = 60%
#   Interests          = 40%
# ---------------------------------------------------------

evaluation_base_score = (

    evaluation_preference_score
    * recommendation_config[
        "travel_factor_weight"
    ]

    +

    evaluation_interest_score
    * recommendation_config[
        "interest_weight"
    ]
)


# ---------------------------------------------------------
# Build the complete evaluation DataFrame.
# ---------------------------------------------------------

evaluation_df = pd.DataFrame({

    "destination":
        recommendation_destinations,

    "preference_score":
        evaluation_preference_score.values,

    "interest_score":
        evaluation_interest_score.values,

    "combined_score":
        evaluation_base_score.values
})


# ---------------------------------------------------------
# Rank all destinations.
# ---------------------------------------------------------

evaluation_df = (
    evaluation_df
    .sort_values(
        "combined_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


evaluation_df.insert(
    0,
    "rank",
    range(
        1,
        len(evaluation_df) + 1
    )
)


# ---------------------------------------------------------
# Select Top-5 recommendations.
# ---------------------------------------------------------

top5_recommendations = (
    evaluation_df
    .head(5)
    .copy()
)


# ---------------------------------------------------------
# Calculate averages for all destinations.
# ---------------------------------------------------------

all_preference_mean = (
    evaluation_df[
        "preference_score"
    ].mean()
)


all_interest_mean = (
    evaluation_df[
        "interest_score"
    ].mean()
)


all_combined_mean = (
    evaluation_df[
        "combined_score"
    ].mean()
)


# ---------------------------------------------------------
# Calculate averages for Top-5 destinations.
# ---------------------------------------------------------

top5_preference_mean = (
    top5_recommendations[
        "preference_score"
    ].mean()
)


top5_interest_mean = (
    top5_recommendations[
        "interest_score"
    ].mean()
)


top5_combined_mean = (
    top5_recommendations[
        "combined_score"
    ].mean()
)


# ---------------------------------------------------------
# Calculate improvement over the overall destination pool.
#
# Positive values indicate that the recommended Top-5 is
# stronger than the average destination.
# ---------------------------------------------------------

preference_improvement = (

    top5_preference_mean
    -
    all_preference_mean
)


interest_improvement = (

    top5_interest_mean
    -
    all_interest_mean
)


combined_improvement = (

    top5_combined_mean
    -
    all_combined_mean
)


# ---------------------------------------------------------
# Display the Top-5 recommendations.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "TOP 5 RECOMMENDATIONS"
)

print(
    "=" * 60
)

print(
    top5_recommendations[
        [
            "rank",
            "destination",
            "preference_score",
            "interest_score",
            "combined_score"
        ]
    ].to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Display comparison with all destinations.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "TOP-5 VS ALL DESTINATIONS"
)

print(
    "=" * 60
)


print(
    f"\nPreference score:"
)

print(
    f"  Top-5 average : "
    f"{top5_preference_mean:.6f}"
)

print(
    f"  All average   : "
    f"{all_preference_mean:.6f}"
)

print(
    f"  Improvement   : "
    f"{preference_improvement:+.6f}"
)


print(
    f"\nInterest score:"
)

print(
    f"  Top-5 average : "
    f"{top5_interest_mean:.6f}"
)

print(
    f"  All average   : "
    f"{all_interest_mean:.6f}"
)

print(
    f"  Improvement   : "
    f"{interest_improvement:+.6f}"
)


print(
    f"\nCombined personalized score:"
)

print(
    f"  Top-5 average : "
    f"{top5_combined_mean:.6f}"
)

print(
    f"  All average   : "
    f"{all_combined_mean:.6f}"
)

print(
    f"  Improvement   : "
    f"{combined_improvement:+.6f}"
)


# ---------------------------------------------------------
# Calculate percentile position of the Top-5 average.
#
# This tells us how strong the recommended destinations are
# relative to the complete destination pool.
# ---------------------------------------------------------

combined_scores = (
    evaluation_df[
        "combined_score"
    ]
)


top5_average_percentile = (
    (
        combined_scores
        <=
        top5_combined_mean
    ).mean()
    * 100
)


print(
    "\n" + "=" * 60
)

print(
    "RECOMMENDATION QUALITY SUMMARY"
)

print(
    "=" * 60
)


print(
    f"\nTop-5 average combined-score percentile: "
    f"{top5_average_percentile:.2f}%"
)


# ---------------------------------------------------------
# Validation.
#
# A recommendation system should ideally rank destinations
# whose personalized scores are above the overall average.
# ---------------------------------------------------------

if top5_combined_mean <= all_combined_mean:

    raise ValueError(
        "Top-5 recommendations are not better than "
        "the overall destination average."
    )


print(
    "\nTop-5 recommendations outperform "
    "the overall destination average."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ RECOMMENDATION QUALITY EVALUATION PASSED"
)

print(
    "=" * 60
)

RECOMMENDATION QUALITY EVALUATION

TOP 5 RECOMMENDATIONS
 rank destination  preference_score  interest_score  combined_score
    1      Mumbai          0.751593        0.522352        0.659896
    2   Bengaluru          0.713373        0.401584        0.588658
    3       Delhi          0.693414        0.408991        0.579645
    4      Manali          0.636394        0.487155        0.576698
    5     Chennai          0.692993        0.343195        0.553074

TOP-5 VS ALL DESTINATIONS

Preference score:
  Top-5 average : 0.697553
  All average   : 0.525442
  Improvement   : +0.172112

Interest score:
  Top-5 average : 0.432655
  All average   : 0.240157
  Improvement   : +0.192499

Combined personalized score:
  Top-5 average : 0.591594
  All average   : 0.411328
  Improvement   : +0.180267

RECOMMENDATION QUALITY SUMMARY

Top-5 average combined-score percentile: 98.00%

Top-5 recommendations outperform the overall destination average.

✓ RECOMMENDATION QUALITY EVALUATION PASSED


In [16]:
# ---------------------------------------------------------
# HYBRID MODEL ABLATION EVALUATION
#
# This version explicitly aligns every score by destination.
#
# This prevents errors caused by:
#   - different array lengths
#   - different indexes
#   - different destination ordering
#
# We compare:
#
#   1. Preference-only
#   2. Similarity-only
#   3. Hybrid
# ---------------------------------------------------------

print("=" * 60)
print("HYBRID MODEL ABLATION EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# USER PROFILE
# ---------------------------------------------------------

ablation_preferences = {

    "budget": 0.90,

    "flight": 0.60,

    "accommodation": 0.60,

    "weather": 0.80,

    "destination_characteristics": 0.80
}


ablation_interests = {

    "nature": 1.00,

    "sightseeing": 0.50,

    "water_coastal": 0.20,

    "wildlife": 0.60
}


# ---------------------------------------------------------
# BUILD A SINGLE, TRUSTED DESTINATION LIST
#
# We use the destination column from the PCA recommendation
# dataset because this is the same destination set used by
# the similarity model.
# ---------------------------------------------------------

evaluation_destinations = (
    recommendation_destinations
)


evaluation_destinations = list(
    evaluation_destinations
)


print(
    "\nNumber of evaluation destinations:",
    len(evaluation_destinations)
)


# ---------------------------------------------------------
# VALIDATE DESTINATION UNIQUENESS
# ---------------------------------------------------------

if len(
    evaluation_destinations
) != len(
    set(evaluation_destinations)
):

    raise ValueError(
        "Duplicate destinations detected."
    )


# ---------------------------------------------------------
# CALCULATE PERSONALIZED PREFERENCE SCORE
# ---------------------------------------------------------

preference_score = (
    calculate_persona_preference_score(
        ablation_preferences
    )
)


# ---------------------------------------------------------
# CALCULATE PERSONALIZED INTEREST SCORE
# ---------------------------------------------------------

interest_score = (
    calculate_persona_interest_score(
        ablation_interests
    )
)


# ---------------------------------------------------------
# ALIGN BOTH SCORE SERIES TO THE DESTINATION LIST
#
# This is the important correction.
# ---------------------------------------------------------

preference_score = (
    preference_score
    .copy()
)


interest_score = (
    interest_score
    .copy()
)


# ---------------------------------------------------------
# The scoring functions use dataframe indexes.
#
# Convert them into destination-indexed Series using the
# actual destination names from the processed dataset.
# ---------------------------------------------------------

score_destinations = list(
    processed_df[
        "destination"
    ]
)


if len(
    score_destinations
) != len(
    preference_score
):

    raise ValueError(
        "Preference score length does not match "
        "processed destination count."
    )


preference_score.index = (
    score_destinations
)


interest_score.index = (
    score_destinations
)


# ---------------------------------------------------------
# Reindex scores using the trusted evaluation destination
# order.
# ---------------------------------------------------------

preference_score = (
    preference_score
    .reindex(
        evaluation_destinations
    )
)


interest_score = (
    interest_score
    .reindex(
        evaluation_destinations
    )
)


# ---------------------------------------------------------
# Validate that no destinations were lost during alignment.
# ---------------------------------------------------------

if preference_score.isna().all():

    raise ValueError(
        "Preference scores could not be aligned "
        "with destinations."
    )


if interest_score.isna().all():

    raise ValueError(
        "Interest scores could not be aligned "
        "with destinations."
    )


# ---------------------------------------------------------
# COMBINE PERSONALIZED PREFERENCE + INTEREST
#
# Travel-factor weight = 0.60
# Interest weight      = 0.40
# ---------------------------------------------------------

personalized_score = (

    preference_score
    * recommendation_config[
        "travel_factor_weight"
    ]

    +

    interest_score
    * recommendation_config[
        "interest_weight"
    ]
)


# ---------------------------------------------------------
# SIMILARITY SCORE
#
# Use Mumbai as the reference destination because it has
# already been used in our previous similarity tests.
# ---------------------------------------------------------

reference_destination = "Mumbai"


# ---------------------------------------------------------
# Validate reference destination.
# ---------------------------------------------------------

if (
    reference_destination
    not in destination_similarity.index
):

    raise ValueError(
        f"Reference destination '{reference_destination}' "
        "not found in similarity matrix."
    )


# ---------------------------------------------------------
# Extract similarity values for the reference destination.
#
# If destination_similarity is a DataFrame, selecting one
# row gives similarity against every destination.
# ---------------------------------------------------------

similarity_row = (
    destination_similarity.loc[
        reference_destination
    ]
)


# ---------------------------------------------------------
# If the extracted row is itself a DataFrame/Series with
# destination labels, explicitly convert it to a Series.
# ---------------------------------------------------------

similarity_score = pd.Series(
    similarity_row
)


# ---------------------------------------------------------
# Align similarity scores to our destination order.
# ---------------------------------------------------------

similarity_score = (
    similarity_score
    .reindex(
        evaluation_destinations
    )
)


# ---------------------------------------------------------
# Check for missing similarity values.
# ---------------------------------------------------------

if similarity_score.isna().any():

    missing_similarity_destinations = (
        similarity_score[
            similarity_score.isna()
        ]
        .index
        .tolist()
    )

    raise ValueError(
        "Missing similarity values for destinations: "
        + str(
            missing_similarity_destinations
        )
    )


# ---------------------------------------------------------
# NORMALIZE SIMILARITY
#
# Raw cosine similarity can range from -1 to +1.
#
# Convert it to [0, 1] for combining with the personalized
# score.
# ---------------------------------------------------------

similarity_min = (
    similarity_score.min()
)


similarity_max = (
    similarity_score.max()
)


if (
    similarity_max
    ==
    similarity_min
):

    normalized_similarity = pd.Series(
        0.5,
        index=evaluation_destinations
    )

else:

    normalized_similarity = (

        similarity_score
        -
        similarity_min

    ) / (

        similarity_max
        -
        similarity_min
    )


# ---------------------------------------------------------
# HYBRID SCORE
#
# Hybrid preference weight = 0.70
# Hybrid similarity weight = 0.30
# ---------------------------------------------------------

hybrid_score = (

    personalized_score
    * recommendation_config[
        "hybrid_preference_weight"
    ]

    +

    normalized_similarity
    * recommendation_config[
        "hybrid_similarity_weight"
    ]
)


# ---------------------------------------------------------
# FINAL LENGTH VALIDATION
#
# Every component must contain exactly one value per
# destination.
# ---------------------------------------------------------

print(
    "\nScore dimensions:"
)

print(
    "  Destinations:",
    len(
        evaluation_destinations
    )
)

print(
    "  Preference scores:",
    len(
        preference_score
    )
)

print(
    "  Interest scores:",
    len(
        interest_score
    )
)

print(
    "  Similarity scores:",
    len(
        normalized_similarity
    )
)

print(
    "  Hybrid scores:",
    len(
        hybrid_score
    )
)


if not (
    len(evaluation_destinations)
    ==
    len(preference_score)
    ==
    len(interest_score)
    ==
    len(normalized_similarity)
    ==
    len(hybrid_score)
):

    raise ValueError(
        "Score dimensions are still inconsistent."
    )


# ---------------------------------------------------------
# CREATE THE ABLATION DATAFRAME
#
# Because everything has already been aligned by destination,
# DataFrame construction is now safe.
# ---------------------------------------------------------

ablation_df = pd.DataFrame({

    "destination":
        evaluation_destinations,

    "preference_score":
        preference_score.values,

    "similarity_score":
        normalized_similarity.values,

    "hybrid_score":
        hybrid_score.values
})


# ---------------------------------------------------------
# PREFERENCE-ONLY RANKING
# ---------------------------------------------------------

preference_ranking = (

    ablation_df[
        [
            "destination",
            "preference_score"
        ]
    ]

    .sort_values(
        "preference_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


preference_ranking.insert(
    0,
    "rank",
    range(
        1,
        len(preference_ranking) + 1
    )
)


# ---------------------------------------------------------
# SIMILARITY-ONLY RANKING
# ---------------------------------------------------------

similarity_ranking = (

    ablation_df[
        [
            "destination",
            "similarity_score"
        ]
    ]

    .sort_values(
        "similarity_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


similarity_ranking.insert(
    0,
    "rank",
    range(
        1,
        len(similarity_ranking) + 1
    )
)


# ---------------------------------------------------------
# HYBRID RANKING
# ---------------------------------------------------------

hybrid_ranking = (

    ablation_df[
        [
            "destination",
            "hybrid_score"
        ]
    ]

    .sort_values(
        "hybrid_score",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


hybrid_ranking.insert(
    0,
    "rank",
    range(
        1,
        len(hybrid_ranking) + 1
    )
)


# ---------------------------------------------------------
# DISPLAY PREFERENCE-ONLY RESULTS
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "PREFERENCE-ONLY TOP 10"
)

print(
    "=" * 60
)

print(
    preference_ranking
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# DISPLAY SIMILARITY-ONLY RESULTS
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    f"SIMILARITY-ONLY TOP 10 "
    f"(Reference: {reference_destination})"
)

print(
    "=" * 60
)

print(
    similarity_ranking
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# DISPLAY HYBRID RESULTS
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "HYBRID TOP 10"
)

print(
    "=" * 60
)

print(
    hybrid_ranking
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# CALCULATE TOP-5 SETS
# ---------------------------------------------------------

preference_top5 = set(
    preference_ranking
    .head(5)[
        "destination"
    ]
)


similarity_top5 = set(
    similarity_ranking
    .head(5)[
        "destination"
    ]
)


hybrid_top5 = set(
    hybrid_ranking
    .head(5)[
        "destination"
    ]
)


# ---------------------------------------------------------
# CALCULATE TOP-5 OVERLAP
# ---------------------------------------------------------

preference_similarity_overlap = len(
    preference_top5
    &
    similarity_top5
)


preference_hybrid_overlap = len(
    preference_top5
    &
    hybrid_top5
)


similarity_hybrid_overlap = len(
    similarity_top5
    &
    hybrid_top5
)


# ---------------------------------------------------------
# DISPLAY ABLATION SUMMARY
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "ABLATION COMPARISON"
)

print(
    "=" * 60
)


print(
    "\nTop-5 overlap:"
)


print(
    f"  Preference ↔ Similarity: "
    f"{preference_similarity_overlap} / 5"
)


print(
    f"  Preference ↔ Hybrid: "
    f"{preference_hybrid_overlap} / 5"
)


print(
    f"  Similarity ↔ Hybrid: "
    f"{similarity_hybrid_overlap} / 5"
)


# ---------------------------------------------------------
# Check whether the hybrid model incorporates information
# from the individual components.
# ---------------------------------------------------------

if (
    hybrid_top5 == preference_top5
    and
    hybrid_top5 == similarity_top5
):

    print(
        "\nWarning:"
    )

    print(
        "Hybrid Top-5 is identical to both individual "
        "components."
    )

else:

    print(
        "\n✓ Hybrid ranking differs from at least "
        "one individual component."
    )


# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "✓ HYBRID MODEL ABLATION EVALUATION COMPLETED"
)

print(
    "=" * 60
)

HYBRID MODEL ABLATION EVALUATION

Number of evaluation destinations: 50

Score dimensions:
  Destinations: 50
  Preference scores: 50
  Interest scores: 50
  Similarity scores: 50
  Hybrid scores: 50

PREFERENCE-ONLY TOP 10
 rank destination  preference_score
    1      Mumbai          0.751593
    2   Bengaluru          0.713373
    3       Delhi          0.693414
    4     Chennai          0.692993
    5        Pune          0.672854
    6     Kolkata          0.660363
    7   Hyderabad          0.653674
    8      Jaipur          0.641897
    9      Manali          0.636394
   10       Kochi          0.635862

SIMILARITY-ONLY TOP 10 (Reference: Mumbai)
 rank destination  similarity_score
    1      Mumbai          1.000000
    2   Bengaluru          0.835196
    3       Delhi          0.810307
    4         Goa          0.718312
    5      Jaipur          0.664577
    6   Hyderabad          0.633930
    7        Pune          0.610201
    8     Chennai          0.562768
    9    Sri

In [17]:
# ---------------------------------------------------------
# REFERENCE DESTINATION EXCLUSION TEST
#
# Purpose:
# A destination used as the reference should not recommend
# itself.
#
# Example:
#   Reference = Goa
#
# The output should contain alternative destinations only.
# ---------------------------------------------------------

print("=" * 60)
print("REFERENCE DESTINATION EXCLUSION TEST")
print("=" * 60)


# ---------------------------------------------------------
# Test destination.
# ---------------------------------------------------------

reference_destination = "Goa"


# ---------------------------------------------------------
# Validate that the destination exists.
# ---------------------------------------------------------

if reference_destination not in destination_similarity.index:

    raise ValueError(
        f"Reference destination '{reference_destination}' "
        "does not exist in the similarity matrix."
    )


# ---------------------------------------------------------
# Extract similarity scores for the reference destination.
# ---------------------------------------------------------

reference_similarity = (
    destination_similarity.loc[
        reference_destination
    ]
    .copy()
)


# ---------------------------------------------------------
# Remove the reference destination itself.
#
# This is important because:
#
# similarity(Goa, Goa) = 1.0
#
# which would otherwise always make Goa the #1 result.
# ---------------------------------------------------------

reference_similarity = (
    reference_similarity
    .drop(
        labels=reference_destination,
        errors="ignore"
    )
)


# ---------------------------------------------------------
# Sort remaining destinations by similarity.
# ---------------------------------------------------------

similar_destinations = (
    reference_similarity
    .sort_values(
        ascending=False
    )
)


# ---------------------------------------------------------
# Display Top-10 alternatives.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    f"TOP 10 DESTINATIONS SIMILAR TO "
    f"{reference_destination}"
)

print(
    "=" * 60
)


similarity_result = pd.DataFrame({

    "rank":
        range(
            1,
            min(
                10,
                len(similar_destinations)
            ) + 1
        ),

    "destination":
        similar_destinations
        .head(10)
        .index,

    "similarity_score":
        similar_destinations
        .head(10)
        .values
})


print(
    similarity_result.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Validate that the reference destination is not present.
# ---------------------------------------------------------

if (
    reference_destination
    in
    similarity_result[
        "destination"
    ].values
):

    raise ValueError(
        "Reference destination is still present "
        "in the recommendation results."
    )


print(
    "\n✓ Reference destination successfully excluded."
)


# ---------------------------------------------------------
# Also verify that the number of alternatives is correct.
#
# With 50 destinations and one reference destination,
# there should be 49 possible alternatives.
# ---------------------------------------------------------

expected_alternatives = (
    len(destination_similarity)
    - 1
)


actual_alternatives = (
    len(reference_similarity)
)


print(
    f"Possible alternatives: "
    f"{actual_alternatives}"
)


if (
    actual_alternatives
    !=
    expected_alternatives
):

    raise ValueError(
        "Unexpected number of alternative destinations."
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ REFERENCE DESTINATION EXCLUSION TEST PASSED"
)

print(
    "=" * 60
)

REFERENCE DESTINATION EXCLUSION TEST

TOP 10 DESTINATIONS SIMILAR TO Goa
 rank destination  similarity_score
    1      Mumbai          0.586488
    2   Alappuzha          0.355314
    3   Kaziranga          0.351321
    4       Coorg          0.225543
    5   Bengaluru          0.209090
    6       Delhi          0.201240
    7     Gokarna          0.195864
    8      Jaipur          0.172510
    9  Kodaikanal          0.112580
   10      Munnar          0.102729

✓ Reference destination successfully excluded.
Possible alternatives: 49

✓ REFERENCE DESTINATION EXCLUSION TEST PASSED


In [20]:
# ---------------------------------------------------------
# FINAL DYNAMIC RECOMMENDATION ENGINE
#
# Purpose:
# Create the final user-dependent recommendation function.
#
# Supported modes:
#
#   1. preference
#   2. similarity
#   3. hybrid
#
# Important behavior:
# If a reference destination is provided, that destination
# is ALWAYS excluded from the final recommendations.
# ---------------------------------------------------------


def get_recommendations(
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid",
    top_k=10
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate Top-K.
    # -----------------------------------------------------

    if top_k <= 0:

        raise ValueError(
            "top_k must be greater than zero."
        )


    # -----------------------------------------------------
    # Calculate personalized travel preference score.
    # -----------------------------------------------------

    preference_score = (
        calculate_persona_preference_score(
            user_preferences
        )
    )


    # -----------------------------------------------------
    # Calculate personalized interest score.
    # -----------------------------------------------------

    interest_score = (
        calculate_persona_interest_score(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Combine travel preferences and destination interests.
    #
    # Travel-factor weight:
    #     0.60
    #
    # Interest weight:
    #     0.40
    # -----------------------------------------------------

    personalized_score = (

        preference_score
        * recommendation_config[
            "travel_factor_weight"
        ]

        +

        interest_score
        * recommendation_config[
            "interest_weight"
        ]
    )


    # -----------------------------------------------------
    # Start with the complete destination list.
    # -----------------------------------------------------

    destinations = list(
        recommendation_destinations
    )


    # -----------------------------------------------------
    # Calculate similarity scores only when needed.
    # -----------------------------------------------------

    normalized_similarity = pd.Series(
        0.0,
        index=destinations,
        dtype=float
    )


    if mode in [
        "similarity",
        "hybrid"
    ]:

        # -------------------------------------------------
        # Similarity mode requires a reference destination.
        # -------------------------------------------------

        if reference_destination is None:

            raise ValueError(
                "reference_destination is required "
                "for similarity and hybrid modes."
            )


        # -------------------------------------------------
        # Validate reference destination.
        # -------------------------------------------------

        if (
            reference_destination
            not in destination_similarity.index
        ):

            raise ValueError(
                f"Reference destination "
                f"'{reference_destination}' "
                "not found."
            )


        # -------------------------------------------------
        # Extract similarity values.
        # -------------------------------------------------

        raw_similarity = (
            destination_similarity.loc[
                reference_destination
            ]
            .reindex(
                destinations
            )
        )


        # -------------------------------------------------
        # Normalize raw similarity to [0, 1].
        # -------------------------------------------------

        similarity_min = (
            raw_similarity.min()
        )

        similarity_max = (
            raw_similarity.max()
        )


        if similarity_max == similarity_min:

            normalized_similarity = pd.Series(
                0.5,
                index=destinations
            )

        else:

            normalized_similarity = (

                raw_similarity
                -
                similarity_min

            ) / (

                similarity_max
                -
                similarity_min
            )


    # -----------------------------------------------------
    # Select the requested recommendation mode.
    # -----------------------------------------------------

    if mode == "preference":

        final_score = (
            personalized_score
        )


    elif mode == "similarity":

        final_score = (
            normalized_similarity
        )


    else:

        # -------------------------------------------------
        # Hybrid recommendation:
        #
        # Personalized preference = 70%
        # Similarity              = 30%
        # -------------------------------------------------

        final_score = (

            personalized_score
            * recommendation_config[
                "hybrid_preference_weight"
            ]

            +

            normalized_similarity
            * recommendation_config[
                "hybrid_similarity_weight"
            ]
        )


    # -----------------------------------------------------
    # Create result DataFrame.
    # -----------------------------------------------------

    result = pd.DataFrame({

        "destination":
            destinations,

        "personalized_preference_score":
            personalized_score
            .reindex(
                destinations
            )
            .values,

        "personalized_interest_score":
            interest_score
            .reindex(
                destinations
            )
            .values,

        "similarity_score":
            normalized_similarity
            .reindex(
                destinations
            )
            .values,

        "final_recommendation_score":
            final_score
            .reindex(
                destinations
            )
            .values
    })


    # -----------------------------------------------------
    # Remove the reference destination.
    #
    # This prevents:
    #
    #     similarity(Goa, Goa) = 1.0
    #
    # from making Goa recommend itself.
    # -----------------------------------------------------

    if reference_destination is not None:

        result = result[
            result[
                "destination"
            ]
            != reference_destination
        ]


    # -----------------------------------------------------
    # Rank recommendations.
    # -----------------------------------------------------

    result = (
        result
        .sort_values(
            "final_recommendation_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    # -----------------------------------------------------
    # Add ranking number.
    # -----------------------------------------------------

    result.insert(
        0,
        "rank",
        range(
            1,
            len(result) + 1
        )
    )


    # -----------------------------------------------------
    # Return only requested number of recommendations.
    # -----------------------------------------------------

    return result.head(
        min(
            top_k,
            len(result)
        )
    )

In [28]:
# ---------------------------------------------------------
# RECREATE SAMPLE USER PROFILE
#
# These are the same validated sample preferences used
# throughout our previous recommendation evaluations.
# ---------------------------------------------------------

sample_user_preferences = {

    # Strong importance for keeping the trip affordable.
    "budget": 0.90,

    # Moderate importance of flight-related factors.
    "flight": 0.60,

    # Moderate importance of accommodation.
    "accommodation": 0.60,

    # High importance of weather.
    "weather": 0.80,

    # High importance of destination characteristics.
    "destination_characteristics": 0.80
}


# ---------------------------------------------------------
# Destination interests
# ---------------------------------------------------------

sample_user_interests = {

    # Strong preference for nature.
    "nature": 1.00,

    # Moderate preference for sightseeing.
    "sightseeing": 0.50,

    # Low preference for water/coastal destinations.
    "water_coastal": 0.20,

    # Moderate preference for wildlife.
    "wildlife": 0.60
}


# ---------------------------------------------------------
# Verify the profile.
# ---------------------------------------------------------

print("=" * 60)
print("SAMPLE USER PROFILE RESTORED")
print("=" * 60)


print("\nTravel preferences:")

for feature, value in sample_user_preferences.items():

    print(
        f"  {feature:<30} {value:.2f}"
    )


print("\nDestination interests:")

for feature, value in sample_user_interests.items():

    print(
        f"  {feature:<30} {value:.2f}"
    )


print("\n✓ Sample user profile restored.")

SAMPLE USER PROFILE RESTORED

Travel preferences:
  budget                         0.90
  flight                         0.60
  accommodation                  0.60
  weather                        0.80
  destination_characteristics    0.80

Destination interests:
  nature                         1.00
  sightseeing                    0.50
  water_coastal                  0.20
  wildlife                       0.60

✓ Sample user profile restored.


In [29]:
# ---------------------------------------------------------
# TEST THE FINAL ENGINE
# ---------------------------------------------------------

goa_recommendations = get_recommendations(

    user_preferences=sample_user_preferences,

    user_interests=sample_user_interests,

    reference_destination="Goa",

    mode="hybrid",

    top_k=10
)


print("=" * 60)
print("FINAL ENGINE TEST — SIMILAR TO GOA")
print("=" * 60)

print(
    goa_recommendations.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Verify that Goa does not appear in its own alternatives.
# ---------------------------------------------------------

if "Goa" in goa_recommendations[
    "destination"
].values:

    raise ValueError(
        "Reference destination was not excluded."
    )


print(
    "\n✓ Goa successfully excluded from recommendations."
)

print(
    "\n✓ FINAL DYNAMIC ENGINE TEST PASSED"
)

FINAL ENGINE TEST — SIMILAR TO GOA
 rank destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
    1      Mumbai                       0.751593                     0.522352          0.730195                    0.680986
    2   Bengaluru                       0.713373                     0.401584          0.483953                    0.557246
    3       Delhi                       0.693414                     0.408991          0.478832                    0.549401
    4      Jaipur                       0.641897                     0.410318          0.460086                    0.522512
    5      Manali                       0.636394                     0.487155          0.281043                    0.488002
    6        Pune                       0.672854                     0.342008          0.362247                    0.487035
    7   Alappuzha                       0.545384                     0.261628          0.579361  

In [30]:
# ---------------------------------------------------------
# FINAL MULTI-SCENARIO RECOMMENDATION EVALUATION
#
# Purpose:
# Test the final recommendation engine using several
# realistic user personas.
#
# This verifies that:
#
#   1. Different users can receive different results.
#   2. All recommendation scores are valid.
#   3. No duplicate destinations are returned.
#   4. Reference destinations are excluded.
#   5. The final engine works across multiple scenarios.
# ---------------------------------------------------------

print("=" * 60)
print("FINAL MULTI-SCENARIO RECOMMENDATION EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Define realistic user scenarios.
# ---------------------------------------------------------

scenarios = {

    "Budget Nature Traveler": {

        "preferences": {

            "budget": 1.00,
            "flight": 0.40,
            "accommodation": 0.70,
            "weather": 0.80,
            "destination_characteristics": 0.90
        },

        "interests": {

            "nature": 1.00,
            "sightseeing": 0.30,
            "water_coastal": 0.20,
            "wildlife": 0.70
        },

        "reference": None,

        "mode": "preference"
    },


    "City Sightseeing Traveler": {

        "preferences": {

            "budget": 0.50,
            "flight": 0.70,
            "accommodation": 0.60,
            "weather": 0.60,
            "destination_characteristics": 0.70
        },

        "interests": {

            "nature": 0.20,
            "sightseeing": 1.00,
            "water_coastal": 0.20,
            "wildlife": 0.10
        },

        "reference": None,

        "mode": "preference"
    },


    "Beach Traveler": {

        "preferences": {

            "budget": 0.60,
            "flight": 0.60,
            "accommodation": 0.60,
            "weather": 0.80,
            "destination_characteristics": 0.80
        },

        "interests": {

            "nature": 0.40,
            "sightseeing": 0.30,
            "water_coastal": 1.00,
            "wildlife": 0.20
        },

        "reference": None,

        "mode": "preference"
    },


    "Wildlife Traveler": {

        "preferences": {

            "budget": 0.60,
            "flight": 0.50,
            "accommodation": 0.60,
            "weather": 0.70,
            "destination_characteristics": 0.90
        },

        "interests": {

            "nature": 0.90,
            "sightseeing": 0.20,
            "water_coastal": 0.10,
            "wildlife": 1.00
        },

        "reference": None,

        "mode": "preference"
    },


    "Goa Alternative Traveler": {

        "preferences": {

            "budget": 0.90,
            "flight": 0.60,
            "accommodation": 0.60,
            "weather": 0.80,
            "destination_characteristics": 0.80
        },

        "interests": {

            "nature": 1.00,
            "sightseeing": 0.50,
            "water_coastal": 0.20,
            "wildlife": 0.60
        },

        "reference": "Goa",

        "mode": "hybrid"
    }
}


# ---------------------------------------------------------
# Store Top-5 results for comparison.
# ---------------------------------------------------------

scenario_results = {}


# ---------------------------------------------------------
# Evaluate every scenario.
# ---------------------------------------------------------

for scenario_name, scenario in scenarios.items():

    print(
        "\n" + "=" * 60
    )

    print(
        scenario_name
    )

    print(
        "=" * 60
    )


    # -----------------------------------------------------
    # Generate recommendations.
    # -----------------------------------------------------

    recommendations = get_recommendations(

        user_preferences=
            scenario[
                "preferences"
            ],

        user_interests=
            scenario[
                "interests"
            ],

        reference_destination=
            scenario[
                "reference"
            ],

        mode=
            scenario[
                "mode"
            ],

        top_k=5
    )


    # -----------------------------------------------------
    # Validate recommendation count.
    # -----------------------------------------------------

    if len(
        recommendations
    ) != 5:

        raise ValueError(
            f"{scenario_name}: "
            "Expected 5 recommendations."
        )


    # -----------------------------------------------------
    # Validate destination uniqueness.
    # -----------------------------------------------------

    if recommendations[
        "destination"
    ].duplicated().any():

        raise ValueError(
            f"{scenario_name}: "
            "Duplicate destinations detected."
        )


    # -----------------------------------------------------
    # Validate recommendation scores.
    # -----------------------------------------------------

    score_columns = [

        "personalized_preference_score",

        "personalized_interest_score",

        "similarity_score",

        "final_recommendation_score"
    ]


    if recommendations[
        score_columns
    ].isna().any().any():

        raise ValueError(
            f"{scenario_name}: "
            "Missing recommendation scores detected."
        )


    # -----------------------------------------------------
    # Validate reference destination exclusion.
    # -----------------------------------------------------

    reference = scenario[
        "reference"
    ]


    if reference is not None:

        if reference in recommendations[
            "destination"
        ].values:

            raise ValueError(
                f"{scenario_name}: "
                f"Reference destination '{reference}' "
                "was not excluded."
            )


    # -----------------------------------------------------
    # Store results.
    # -----------------------------------------------------

    scenario_results[
        scenario_name
    ] = recommendations


    # -----------------------------------------------------
    # Display Top-5.
    # -----------------------------------------------------

    print(
        recommendations[
            [
                "rank",
                "destination",
                "final_recommendation_score"
            ]
        ].to_string(
            index=False
        )
    )


    print(
        "\n✓ Scenario validation passed."
    )


# ---------------------------------------------------------
# Compare recommendation lists between scenarios.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "SCENARIO DIFFERENCE ANALYSIS"
)

print(
    "=" * 60
)


scenario_names = list(
    scenario_results.keys()
)


different_pairs = 0


for i in range(
    len(scenario_names)
):

    for j in range(
        i + 1,
        len(scenario_names)
    ):

        scenario_a = (
            scenario_names[i]
        )

        scenario_b = (
            scenario_names[j]
        )


        top5_a = set(
            scenario_results[
                scenario_a
            ][
                "destination"
            ]
        )


        top5_b = set(
            scenario_results[
                scenario_b
            ][
                "destination"
            ]
        )


        overlap = len(
            top5_a
            &
            top5_b
        )


        if top5_a != top5_b:

            different_pairs += 1


        print(
            f"\n{scenario_a}"
        )

        print(
            f"vs {scenario_b}"
        )

        print(
            f"Top-5 overlap: "
            f"{overlap} / 5"
        )


# ---------------------------------------------------------
# Final validation.
#
# At least two scenarios should produce different Top-5
# recommendation lists.
# ---------------------------------------------------------

if different_pairs == 0:

    raise ValueError(
        "All scenarios produced identical "
        "recommendation lists."
    )


print(
    "\n" + "=" * 60
)

print(
    "MULTI-SCENARIO SUMMARY"
)

print(
    "=" * 60
)


print(
    f"\nScenarios evaluated: "
    f"{len(scenarios)}"
)


print(
    f"Scenario pairs with different "
    f"Top-5 lists: {different_pairs}"
)


print(
    "\n✓ Different user scenarios produce "
    "different recommendation behavior."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ FINAL MULTI-SCENARIO "
    "RECOMMENDATION EVALUATION PASSED"
)

print(
    "=" * 60
)

FINAL MULTI-SCENARIO RECOMMENDATION EVALUATION

Budget Nature Traveler
 rank destination  final_recommendation_score
    1      Mumbai                    0.646192
    2      Manali                    0.583030
    3   Bengaluru                    0.563660
    4       Delhi                    0.559736
    5      Jaipur                    0.540950

✓ Scenario validation passed.

City Sightseeing Traveler
 rank destination  final_recommendation_score
    1      Mumbai                    0.713316
    2   Bengaluru                    0.676512
    3       Delhi                    0.663670
    4     Chennai                    0.631267
    5        Pune                    0.614084

✓ Scenario validation passed.

Beach Traveler
 rank destination  final_recommendation_score
    1      Mumbai                    0.585081
    2     Chennai                    0.576418
    3       Delhi                    0.550486
    4   Bengaluru                    0.548572
    5       Kochi                    0.540

In [31]:
# ---------------------------------------------------------
# RECOMMENDATION DIVERSITY & DESTINATION BIAS EVALUATION
#
# Purpose:
# Evaluate whether the recommendation engine is overly
# dominated by a small number of destinations.
#
# We measure:
#
#   1. Unique destinations recommended
#   2. Number of times each destination appears
#   3. Number of #1 recommendations
#   4. Top-5 recommendation concentration
#   5. Average pairwise Top-5 overlap
#   6. Diversity of the recommendation lists
#
# This helps identify whether personalization is producing
# meaningful variation instead of recommending the same
# destinations to every user.
# ---------------------------------------------------------

print("=" * 60)
print("RECOMMENDATION DIVERSITY & DESTINATION BIAS EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Collect all Top-5 recommendations from the scenarios
# evaluated previously.
# ---------------------------------------------------------

all_recommended_destinations = []

top1_destinations = []

scenario_top5 = {}


for scenario_name, recommendations in scenario_results.items():

    # -----------------------------------------------------
    # Extract Top-5 destinations.
    # -----------------------------------------------------

    top5 = (
        recommendations[
            "destination"
        ]
        .head(5)
        .tolist()
    )


    scenario_top5[
        scenario_name
    ] = top5


    # -----------------------------------------------------
    # Add Top-5 destinations to the global collection.
    # -----------------------------------------------------

    all_recommended_destinations.extend(
        top5
    )


    # -----------------------------------------------------
    # Store the #1 destination separately.
    # -----------------------------------------------------

    top1_destinations.append(
        top5[0]
    )


# ---------------------------------------------------------
# Count how frequently every destination appears in Top-5.
# ---------------------------------------------------------

destination_frequency = (
    pd.Series(
        all_recommended_destinations
    )
    .value_counts()
)


print(
    "\n" + "=" * 60
)

print(
    "TOP-5 DESTINATION FREQUENCY"
)

print(
    "=" * 60
)


frequency_table = pd.DataFrame({

    "destination":
        destination_frequency.index,

    "top5_appearances":
        destination_frequency.values
})


print(
    frequency_table.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Calculate number of unique destinations recommended.
#
# Maximum possible:
#
# 5 scenarios × 5 recommendations = 25
#
# But destinations may repeat between scenarios.
# ---------------------------------------------------------

unique_recommended = len(
    set(
        all_recommended_destinations
    )
)


total_recommendation_slots = len(
    all_recommended_destinations
)


print(
    "\n" + "=" * 60
)

print(
    "RECOMMENDATION COVERAGE"
)

print(
    "=" * 60
)


print(
    f"\nTotal Top-5 recommendation slots: "
    f"{total_recommendation_slots}"
)


print(
    f"Unique destinations recommended: "
    f"{unique_recommended}"
)


coverage_ratio = (
    unique_recommended
    /
    total_recommendation_slots
)


print(
    f"Unique-destination ratio: "
    f"{coverage_ratio:.4f}"
)


# ---------------------------------------------------------
# Analyze #1 recommendation concentration.
# ---------------------------------------------------------

top1_frequency = (
    pd.Series(
        top1_destinations
    )
    .value_counts()
)


print(
    "\n" + "=" * 60
)

print(
    "TOP-1 RECOMMENDATION CONCENTRATION"
)

print(
    "=" * 60
)


for destination, count in (
    top1_frequency.items()
):

    print(
        f"  {destination:<25} "
        f"{count} / {len(scenarios)}"
    )


max_top1_count = (
    top1_frequency.max()
)


max_top1_ratio = (
    max_top1_count
    /
    len(scenarios)
)


print(
    f"\nMaximum #1 concentration: "
    f"{max_top1_ratio:.2%}"
)


# ---------------------------------------------------------
# Calculate pairwise Top-5 overlap.
#
# Lower overlap means greater diversity.
# ---------------------------------------------------------

scenario_names = list(
    scenario_top5.keys()
)


pairwise_overlaps = []


print(
    "\n" + "=" * 60
)

print(
    "PAIRWISE TOP-5 OVERLAP"
)

print(
    "=" * 60
)


for i in range(
    len(scenario_names)
):

    for j in range(
        i + 1,
        len(scenario_names)
    ):

        scenario_a = (
            scenario_names[i]
        )

        scenario_b = (
            scenario_names[j]
        )


        set_a = set(
            scenario_top5[
                scenario_a
            ]
        )


        set_b = set(
            scenario_top5[
                scenario_b
            ]
        )


        overlap = len(
            set_a & set_b
        )


        overlap_ratio = (
            overlap / 5
        )


        pairwise_overlaps.append(
            overlap_ratio
        )


        print(
            f"{scenario_a} "
            f"↔ {scenario_b}: "
            f"{overlap} / 5"
        )


# ---------------------------------------------------------
# Average pairwise overlap.
# ---------------------------------------------------------

average_overlap = (
    np.mean(
        pairwise_overlaps
    )
)


print(
    f"\nAverage Top-5 overlap: "
    f"{average_overlap:.4f}"
)


# ---------------------------------------------------------
# Convert overlap into a simple diversity measure.
#
# If average overlap = 1:
#
#     All scenarios have identical Top-5 lists.
#
# If average overlap = 0:
#
#     No destinations are shared.
#
# Therefore:
#
#     diversity = 1 - average overlap
# ---------------------------------------------------------

diversity_score = (
    1
    -
    average_overlap
)


print(
    f"Recommendation diversity score: "
    f"{diversity_score:.4f}"
)


# ---------------------------------------------------------
# Calculate recommendation concentration.
#
# The most frequently recommended destination's share of
# all Top-5 recommendation slots.
# ---------------------------------------------------------

most_frequent_count = (
    destination_frequency.iloc[0]
)


most_frequent_destination = (
    destination_frequency.index[0]
)


concentration_ratio = (
    most_frequent_count
    /
    total_recommendation_slots
)


print(
    "\n" + "=" * 60
)

print(
    "DESTINATION CONCENTRATION"
)

print(
    "=" * 60
)


print(
    f"\nMost frequently recommended: "
    f"{most_frequent_destination}"
)


print(
    f"Appearances: "
    f"{most_frequent_count} / "
    f"{total_recommendation_slots}"
)


print(
    f"Recommendation concentration: "
    f"{concentration_ratio:.2%}"
)


# ---------------------------------------------------------
# Final diagnostic interpretation.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "DIVERSITY SUMMARY"
)

print(
    "=" * 60
)


print(
    f"\nUnique destinations: "
    f"{unique_recommended}"
    f" / "
    f"{total_recommendation_slots}"
)


print(
    f"Average Top-5 overlap: "
    f"{average_overlap:.4f}"
)


print(
    f"Diversity score: "
    f"{diversity_score:.4f}"
)


print(
    f"Maximum #1 concentration: "
    f"{max_top1_ratio:.2%}"
)


# ---------------------------------------------------------
# Basic validation.
#
# We do NOT demand extremely high diversity because a good
# recommender is allowed to recommend strong destinations
# to multiple users.
#
# We only verify that:
#
#   - More than one destination is recommended.
#   - At least two scenarios have different lists.
#   - The system isn't returning one destination everywhere.
# ---------------------------------------------------------

if unique_recommended <= 1:

    raise ValueError(
        "Severe recommendation concentration: "
        "only one destination was recommended."
    )


if diversity_score == 0:

    raise ValueError(
        "No recommendation diversity detected."
    )


if max_top1_ratio >= 1.0:

    raise ValueError(
        "The same destination is ranked #1 "
        "for every evaluated scenario."
    )


print(
    "\n✓ Multiple destinations are being recommended."
)

print(
    "✓ Recommendation lists show measurable diversity."
)

print(
    "✓ No single destination is ranked #1 "
    "for every scenario."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ RECOMMENDATION DIVERSITY "
    "EVALUATION COMPLETED"
)

print(
    "=" * 60
)

RECOMMENDATION DIVERSITY & DESTINATION BIAS EVALUATION

TOP-5 DESTINATION FREQUENCY
destination  top5_appearances
     Mumbai                 5
  Bengaluru                 5
      Delhi                 5
     Manali                 3
     Jaipur                 3
    Chennai                 2
       Pune                 1
      Kochi                 1

RECOMMENDATION COVERAGE

Total Top-5 recommendation slots: 25
Unique destinations recommended: 8
Unique-destination ratio: 0.3200

TOP-1 RECOMMENDATION CONCENTRATION
  Mumbai                    5 / 5

Maximum #1 concentration: 100.00%

PAIRWISE TOP-5 OVERLAP
Budget Nature Traveler ↔ City Sightseeing Traveler: 3 / 5
Budget Nature Traveler ↔ Beach Traveler: 3 / 5
Budget Nature Traveler ↔ Wildlife Traveler: 5 / 5
Budget Nature Traveler ↔ Goa Alternative Traveler: 5 / 5
City Sightseeing Traveler ↔ Beach Traveler: 4 / 5
City Sightseeing Traveler ↔ Wildlife Traveler: 3 / 5
City Sightseeing Traveler ↔ Goa Alternative Traveler: 3 / 5
Beach Trave

ValueError: The same destination is ranked #1 for every evaluated scenario.

In [32]:
# ---------------------------------------------------------
# MUMBAI DOMINANCE / SCORE DECOMPOSITION ANALYSIS
#
# Purpose:
# Investigate why Mumbai is ranked #1 across multiple
# user personas.
#
# We compare:
#
#   - personalized preference score
#   - personalized interest score
#   - combined personalized score
#   - final recommendation score
#
# This helps determine whether Mumbai is genuinely strong
# across the underlying features or whether the scoring
# system is causing excessive concentration.
# ---------------------------------------------------------


print("=" * 60)
print("MUMBAI DOMINANCE / SCORE DECOMPOSITION ANALYSIS")
print("=" * 60)


# ---------------------------------------------------------
# Destinations we want to compare against Mumbai.
#
# These are destinations that appeared frequently in our
# previous recommendation results.
# ---------------------------------------------------------

comparison_destinations = [

    "Mumbai",
    "Bengaluru",
    "Delhi",
    "Manali",
    "Jaipur",
    "Chennai",
    "Pune",
    "Kochi"
]


# ---------------------------------------------------------
# Store decomposition results.
# ---------------------------------------------------------

decomposition_results = []


# ---------------------------------------------------------
# Evaluate every scenario.
# ---------------------------------------------------------

for scenario_name, scenario in scenarios.items():

    # -----------------------------------------------------
    # Generate recommendations for ALL destinations.
    #
    # top_k=50 allows us to inspect the complete ranking
    # instead of only the Top-5.
    # -----------------------------------------------------

    recommendations = get_recommendations(

        user_preferences=
            scenario[
                "preferences"
            ],

        user_interests=
            scenario[
                "interests"
            ],

        reference_destination=
            scenario[
                "reference"
            ],

        mode=
            scenario[
                "mode"
            ],

        top_k=50
    )


    # -----------------------------------------------------
    # Extract selected destinations.
    # -----------------------------------------------------

    selected = recommendations[
        recommendations[
            "destination"
        ].isin(
            comparison_destinations
        )
    ].copy()


    # -----------------------------------------------------
    # Add scenario name.
    # -----------------------------------------------------

    selected[
        "scenario"
    ] = scenario_name


    # -----------------------------------------------------
    # Store results.
    # -----------------------------------------------------

    decomposition_results.append(
        selected
    )


# ---------------------------------------------------------
# Combine all scenarios.
# ---------------------------------------------------------

decomposition_df = pd.concat(
    decomposition_results,
    ignore_index=True
)


# ---------------------------------------------------------
# Display the decomposition.
# ---------------------------------------------------------

display_columns = [

    "scenario",

    "destination",

    "personalized_preference_score",

    "personalized_interest_score",

    "similarity_score",

    "final_recommendation_score"
]


print(
    "\n" + "=" * 60
)

print(
    "SCORE DECOMPOSITION"
)

print(
    "=" * 60
)


print(
    decomposition_df[
        display_columns
    ]
    .sort_values(
        [
            "scenario",
            "final_recommendation_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Calculate Mumbai's average score across scenarios.
# ---------------------------------------------------------

mumbai_scores = decomposition_df[
    decomposition_df[
        "destination"
    ] == "Mumbai"
]


print(
    "\n" + "=" * 60
)

print(
    "MUMBAI AVERAGE SCORE"
)

print(
    "=" * 60
)


print(
    "\nAverage personalized preference score:",
    f"{mumbai_scores['personalized_preference_score'].mean():.6f}"
)


print(
    "Average personalized interest score:",
    f"{mumbai_scores['personalized_interest_score'].mean():.6f}"
)


print(
    "Average similarity score:",
    f"{mumbai_scores['similarity_score'].mean():.6f}"
)


print(
    "Average final recommendation score:",
    f"{mumbai_scores['final_recommendation_score'].mean():.6f}"
)


# ---------------------------------------------------------
# Compare Mumbai against every other selected destination.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "AVERAGE DESTINATION SCORE COMPARISON"
)

print(
    "=" * 60
)


average_destination_scores = (

    decomposition_df

    .groupby(
        "destination"
    )[

        [
            "personalized_preference_score",
            "personalized_interest_score",
            "similarity_score",
            "final_recommendation_score"
        ]

    ]

    .mean()

    .sort_values(
        "final_recommendation_score",
        ascending=False
    )
)


print(
    average_destination_scores.to_string()
)


# ---------------------------------------------------------
# Determine how much higher Mumbai scores compared with
# the second-best destination.
# ---------------------------------------------------------

if "Mumbai" in average_destination_scores.index:

    ranked_scores = (
        average_destination_scores[
            "final_recommendation_score"
        ]
        .sort_values(
            ascending=False
        )
    )


    mumbai_rank = (
        ranked_scores.index
        .tolist()
        .index("Mumbai")
        + 1
    )


    mumbai_score = (
        ranked_scores[
            "Mumbai"
        ]
    )


    if len(ranked_scores) > 1:

        second_best_destination = (
            ranked_scores.index[1]
        )

        second_best_score = (
            ranked_scores.iloc[1]
        )

        score_gap = (
            mumbai_score
            -
            second_best_score
        )


        print(
            "\n" + "=" * 60
        )

        print(
            "MUMBAI VS SECOND-BEST DESTINATION"
        )

        print(
            "=" * 60
        )


        print(
            f"\nMumbai average score: "
            f"{mumbai_score:.6f}"
        )


        print(
            f"Second-best destination: "
            f"{second_best_destination}"
        )


        print(
            f"Second-best average score: "
            f"{second_best_score:.6f}"
        )


        print(
            f"Score gap: "
            f"{score_gap:.6f}"
        )


        print(
            f"Mumbai average rank: "
            f"{mumbai_rank}"
        )


print(
    "\n" + "=" * 60
)

print(
    "✓ MUMBAI DOMINANCE ANALYSIS COMPLETED"
)

print(
    "=" * 60
)

MUMBAI DOMINANCE / SCORE DECOMPOSITION ANALYSIS

SCORE DECOMPOSITION
                 scenario destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
           Beach Traveler      Mumbai                       0.736571                     0.357847          0.000000                    0.585081
           Beach Traveler     Chennai                       0.664444                     0.444379          0.000000                    0.576418
           Beach Traveler       Delhi                       0.679290                     0.357279          0.000000                    0.550486
           Beach Traveler   Bengaluru                       0.689638                     0.336973          0.000000                    0.548572
           Beach Traveler       Kochi                       0.602902                     0.446871          0.000000                    0.540489
           Beach Traveler        Pune                       0.64099

In [33]:
# ---------------------------------------------------------
# EXPLAINABLE RECOMMENDATION ENGINE
#
# Purpose:
# Decompose a recommendation score into understandable
# contributions from:
#
#   1. Travel preference groups
#   2. Destination interests
#   3. Similarity
#
# This allows the system to explain WHY a destination was
# recommended instead of returning only a final score.
# ---------------------------------------------------------


def explain_recommendation(
    user_preferences,
    user_interests,
    destination,
    reference_destination=None,
    mode="hybrid"
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in processed_df[
        "destination"
    ].values:

        raise ValueError(
            f"Destination '{destination}' "
            "not found."
        )


    # -----------------------------------------------------
    # Calculate the raw preference-group scores.
    #
    # These are calculated using the same scoring function
    # used by the recommendation engine.
    # -----------------------------------------------------

    preference_score = (
        calculate_persona_preference_score(
            user_preferences
        )
    )


    # -----------------------------------------------------
    # Calculate personalized interest scores.
    # -----------------------------------------------------

    interest_score = (
        calculate_persona_interest_score(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Convert RangeIndex scores into destination-indexed
    # Series, exactly as we fixed in the final engine.
    # -----------------------------------------------------

    destinations = list(
        processed_df[
            "destination"
        ]
    )


    preference_score = pd.Series(

        preference_score.values,

        index=destinations,

        dtype=float
    )


    interest_score = pd.Series(

        interest_score.values,

        index=destinations,

        dtype=float
    )


    # -----------------------------------------------------
    # Get the selected destination's scores.
    # -----------------------------------------------------

    destination_preference = (
        preference_score[
            destination
        ]
    )


    destination_interest = (
        interest_score[
            destination
        ]
    )


    # -----------------------------------------------------
    # Calculate the weighted contributions.
    #
    # Travel preference contribution:
    #
    #     preference × 0.60
    #
    # Interest contribution:
    #
    #     interest × 0.40
    # -----------------------------------------------------

    preference_contribution = (

        destination_preference
        *
        recommendation_config[
            "travel_factor_weight"
        ]
    )


    interest_contribution = (

        destination_interest
        *
        recommendation_config[
            "interest_weight"
        ]
    )


    # -----------------------------------------------------
    # Calculate similarity contribution.
    # -----------------------------------------------------

    similarity_score = 0.0


    if mode in [
        "similarity",
        "hybrid"
    ]:

        if reference_destination is None:

            raise ValueError(
                "reference_destination is required "
                "for similarity and hybrid modes."
            )


        if reference_destination not in (
            destination_similarity.index
        ):

            raise ValueError(
                f"Reference destination "
                f"'{reference_destination}' "
                "not found."
            )


        # -------------------------------------------------
        # Extract similarity scores for every destination.
        # -------------------------------------------------

        raw_similarity = (
            destination_similarity.loc[
                reference_destination
            ]
            .reindex(
                destinations
            )
        )


        # -------------------------------------------------
        # Normalize similarity to [0, 1].
        # -------------------------------------------------

        similarity_min = (
            raw_similarity.min()
        )

        similarity_max = (
            raw_similarity.max()
        )


        if similarity_max == similarity_min:

            normalized_similarity = pd.Series(
                0.5,
                index=destinations
            )

        else:

            normalized_similarity = (

                raw_similarity
                -
                similarity_min

            ) / (

                similarity_max
                -
                similarity_min
            )


        similarity_score = (
            normalized_similarity[
                destination
            ]
        )


    # -----------------------------------------------------
    # Calculate final score based on recommendation mode.
    # -----------------------------------------------------

    if mode == "preference":

        final_score = (
            preference_contribution
            +
            interest_contribution
        )

        similarity_contribution = 0.0


    elif mode == "similarity":

        final_score = similarity_score

        preference_contribution = 0.0
        interest_contribution = 0.0

        similarity_contribution = (
            similarity_score
        )


    else:

        # -------------------------------------------------
        # Hybrid contribution:
        #
        # Personalized score = preference + interest
        #
        # Hybrid:
        #
        #   70% personalized score
        #   30% similarity
        # -------------------------------------------------

        hybrid_preference_weight = (
            recommendation_config[
                "hybrid_preference_weight"
            ]
        )

        hybrid_similarity_weight = (
            recommendation_config[
                "hybrid_similarity_weight"
            ]
        )


        # -------------------------------------------------
        # Break the hybrid preference contribution down
        # into its two underlying components.
        # -------------------------------------------------

        preference_contribution = (

            destination_preference
            *
            recommendation_config[
                "travel_factor_weight"
            ]
            *
            hybrid_preference_weight
        )


        interest_contribution = (

            destination_interest
            *
            recommendation_config[
                "interest_weight"
            ]
            *
            hybrid_preference_weight
        )


        similarity_contribution = (

            similarity_score
            *
            hybrid_similarity_weight
        )


        final_score = (

            preference_contribution

            +

            interest_contribution

            +

            similarity_contribution
        )


    # -----------------------------------------------------
    # Build explanation table.
    # -----------------------------------------------------

    explanation = pd.DataFrame({

        "component": [

            "Travel preferences",

            "Destination interests",

            "Similarity"
        ],

        "contribution": [

            preference_contribution,

            interest_contribution,

            similarity_contribution
        ]
    })


    # -----------------------------------------------------
    # Calculate percentage contribution.
    #
    # This tells us how much each component contributed
    # relative to the final score.
    # -----------------------------------------------------

    if final_score != 0:

        explanation[
            "percentage"
        ] = (

            explanation[
                "contribution"
            ]
            /
            final_score
            *
            100
        )

    else:

        explanation[
            "percentage"
        ] = 0.0


    # -----------------------------------------------------
    # Validate mathematical consistency.
    # -----------------------------------------------------

    reconstructed_score = (
        explanation[
            "contribution"
        ].sum()
    )


    reconstruction_error = abs(
        final_score
        -
        reconstructed_score
    )


    if reconstruction_error > 1e-10:

        raise ValueError(
            "Explanation contributions do not "
            "reconstruct the final score."
        )


    # -----------------------------------------------------
    # Return all explanation information.
    # -----------------------------------------------------

    return {

        "destination":
            destination,

        "mode":
            mode,

        "preference_score":
            destination_preference,

        "interest_score":
            destination_interest,

        "similarity_score":
            similarity_score,

        "final_score":
            final_score,

        "explanation":
            explanation,

        "reconstruction_error":
            reconstruction_error
    }


print("=" * 60)
print("✓ EXPLAINABLE RECOMMENDATION ENGINE CREATED")
print("=" * 60)

✓ EXPLAINABLE RECOMMENDATION ENGINE CREATED


In [34]:
# ---------------------------------------------------------
# TEST EXPLANATION FOR MUMBAI
#
# User:
#   Current sample user
#
# Reference:
#   Goa
#
# Mode:
#   Hybrid
# ---------------------------------------------------------


mumbai_explanation = explain_recommendation(

    user_preferences=
        sample_user_preferences,

    user_interests=
        sample_user_interests,

    destination="Mumbai",

    reference_destination="Goa",

    mode="hybrid"
)


print("=" * 60)
print("WHY WAS MUMBAI RECOMMENDED?")
print("=" * 60)


print(
    f"\nDestination: "
    f"{mumbai_explanation['destination']}"
)


print(
    f"Mode: "
    f"{mumbai_explanation['mode']}"
)


print(
    f"\nPersonalized preference score: "
    f"{mumbai_explanation['preference_score']:.6f}"
)


print(
    f"Personalized interest score: "
    f"{mumbai_explanation['interest_score']:.6f}"
)


print(
    f"Similarity score: "
    f"{mumbai_explanation['similarity_score']:.6f}"
)


print(
    f"\nFinal recommendation score: "
    f"{mumbai_explanation['final_score']:.6f}"
)


print(
    "\n" + "=" * 60
)

print(
    "SCORE CONTRIBUTIONS"
)

print(
    "=" * 60
)


print(
    mumbai_explanation[
        "explanation"
    ].to_string(
        index=False
    )
)


print(
    f"\nReconstruction error: "
    f"{mumbai_explanation['reconstruction_error']:.12f}"
)


print(
    "\n✓ Explanation mathematically "
    "reconstructs the final score."
)

WHY WAS MUMBAI RECOMMENDED?

Destination: Mumbai
Mode: hybrid

Personalized preference score: 0.751593
Personalized interest score: 0.522352
Similarity score: 0.730195

Final recommendation score: 0.680986

SCORE CONTRIBUTIONS
            component  contribution  percentage
   Travel preferences      0.315669   46.354676
Destination interests      0.146259   21.477467
           Similarity      0.219059   32.167857

Reconstruction error: 0.000000000000

✓ Explanation mathematically reconstructs the final score.


In [35]:
# ---------------------------------------------------------
# INDIVIDUAL PREFERENCE EXPLAINABILITY
#
# Purpose:
# Break the travel-preference contribution down into the
# individual preference groups.
#
# We will show:
#
#   Budget
#   Flight
#   Accommodation
#   Weather
#   Destination characteristics
#
# We also verify that these contributions reconstruct the
# total personalized preference score.
# ---------------------------------------------------------


def explain_preference_groups(
    user_preferences,
    destination
):

    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in processed_df[
        "destination"
    ].values:

        raise ValueError(
            f"Destination '{destination}' "
            "not found."
        )


    # -----------------------------------------------------
    # Get the destination row.
    # -----------------------------------------------------

    destination_index = (
        processed_df[
            "destination"
        ]
        .tolist()
        .index(
            destination
        )
    )


    # -----------------------------------------------------
    # Store individual group contributions.
    # -----------------------------------------------------

    group_results = []


    # -----------------------------------------------------
    # Process every preference group.
    # -----------------------------------------------------

    for group_name, features in (
        preference_groups.items()
    ):

        # -------------------------------------------------
        # User's importance for this group.
        # -------------------------------------------------

        user_weight = (
            user_preferences[
                group_name
            ]
        )


        # -------------------------------------------------
        # Calculate the destination's average score across
        # the features belonging to this group.
        #
        # Missing features are ignored.
        # -------------------------------------------------

        available_scores = []


        for feature in features:

            if feature not in normalized_features.columns:

                continue


            value = (
                normalized_features.iloc[
                    destination_index
                ][
                    feature
                ]
            )


            if pd.notna(value):

                # -----------------------------------------
                # Direction-corrected values are already
                # oriented so higher = better.
                # -----------------------------------------

                available_scores.append(
                    value
                )


        # -------------------------------------------------
        # If no data is available for this group, the group
        # contributes zero.
        # -------------------------------------------------

        if len(
            available_scores
        ) == 0:

            destination_group_score = 0.0

        else:

            destination_group_score = (
                np.mean(
                    available_scores
                )
            )


        # -------------------------------------------------
        # Group contribution before global travel weight.
        # -------------------------------------------------

        raw_contribution = (

            destination_group_score
            *
            user_weight
        )


        group_results.append({

            "group":
                group_name,

            "destination_score":
                destination_group_score,

            "user_weight":
                user_weight,

            "raw_contribution":
                raw_contribution
        })


    # -----------------------------------------------------
    # Convert results to DataFrame.
    # -----------------------------------------------------

    explanation_df = pd.DataFrame(
        group_results
    )


    # -----------------------------------------------------
    # Normalize group contributions so that their weighted
    # sum equals the personalized preference score.
    #
    # This keeps the explanation consistent with the actual
    # scoring architecture.
    # -----------------------------------------------------

    total_raw_contribution = (
        explanation_df[
            "raw_contribution"
        ].sum()
    )


    # -----------------------------------------------------
    # Get the actual personalized preference score from
    # the production scoring function.
    # -----------------------------------------------------

    raw_preference_scores = (
        calculate_persona_preference_score(
            user_preferences
        )
    )


    destinations = list(
        processed_df[
            "destination"
        ]
    )


    actual_preference_scores = pd.Series(

        raw_preference_scores.values,

        index=destinations,

        dtype=float
    )


    actual_score = (
        actual_preference_scores[
            destination
        ]
    )


    # -----------------------------------------------------
    # Scale individual contributions so they reconstruct
    # the production preference score.
    # -----------------------------------------------------

    if total_raw_contribution > 0:

        explanation_df[
            "contribution"
        ] = (

            explanation_df[
                "raw_contribution"
            ]
            /
            total_raw_contribution
            *
            actual_score
        )

    else:

        explanation_df[
            "contribution"
        ] = 0.0


    # -----------------------------------------------------
    # Calculate percentage contribution.
    # -----------------------------------------------------

    if actual_score != 0:

        explanation_df[
            "percentage"
        ] = (

            explanation_df[
                "contribution"
            ]
            /
            actual_score
            *
            100
        )

    else:

        explanation_df[
            "percentage"
        ] = 0.0


    # -----------------------------------------------------
    # Validate reconstruction.
    # -----------------------------------------------------

    reconstructed_score = (
        explanation_df[
            "contribution"
        ].sum()
    )


    reconstruction_error = abs(

        actual_score
        -
        reconstructed_score
    )


    if reconstruction_error > 1e-10:

        raise ValueError(
            "Individual preference contributions "
            "do not reconstruct the actual score."
        )


    return {

        "destination":
            destination,

        "preference_score":
            actual_score,

        "explanation":
            explanation_df,

        "reconstruction_error":
            reconstruction_error
    }


print("=" * 60)
print("✓ INDIVIDUAL PREFERENCE EXPLANATION ENGINE CREATED")
print("=" * 60)

✓ INDIVIDUAL PREFERENCE EXPLANATION ENGINE CREATED


In [36]:
# ---------------------------------------------------------
# TEST INDIVIDUAL PREFERENCE EXPLANATION
# ---------------------------------------------------------

mumbai_preference_explanation = (
    explain_preference_groups(
        user_preferences=
            sample_user_preferences,

        destination="Mumbai"
    )
)


print("=" * 60)
print("WHY IS MUMBAI STRONG FOR THIS USER?")
print("=" * 60)


print(
    f"\nPersonalized preference score: "
    f"{mumbai_preference_explanation['preference_score']:.6f}"
)


print(
    "\nINDIVIDUAL GROUP CONTRIBUTIONS"
)

print(
    "=" * 60
)


print(
    mumbai_preference_explanation[
        "explanation"
    ][
        [
            "group",
            "destination_score",
            "user_weight",
            "contribution",
            "percentage"
        ]
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .to_string(
        index=False
    )
)


print(
    f"\nReconstruction error: "
    f"{mumbai_preference_explanation['reconstruction_error']:.12f}"
)


print(
    "\n✓ Individual preference contributions "
    "reconstruct the production score."
)

WHY IS MUMBAI STRONG FOR THIS USER?

Personalized preference score: 0.751593

INDIVIDUAL GROUP CONTRIBUTIONS
                      group  destination_score  user_weight  contribution  percentage
                     budget           0.921839          0.9      0.224231   29.834147
                    weather           0.684796          0.8      0.148064   19.700027
                     flight           0.836881          0.6      0.135710   18.056376
              accommodation           0.819973          0.6      0.132969   17.691570
destination_characteristics           0.511611          0.8      0.110618   14.717881

Reconstruction error: 0.000000000000

✓ Individual preference contributions reconstruct the production score.


In [37]:
# ---------------------------------------------------------
# INDIVIDUAL INTEREST EXPLAINABILITY
#
# Purpose:
# Break the personalized interest score into the individual
# destination-interest profiles:
#
#   - Nature
#   - Sightseeing
#   - Water / Coastal
#   - Wildlife
#
# The contributions will be calculated using the user's
# interest weights and the destination's profile scores.
#
# Finally, we verify that the contributions reconstruct the
# production personalized-interest score exactly.
# ---------------------------------------------------------


def explain_interest_profiles(
    user_interests,
    destination
):

    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in processed_df[
        "destination"
    ].values:

        raise ValueError(
            f"Destination '{destination}' "
            "not found."
        )


    # -----------------------------------------------------
    # Get the destination index.
    # -----------------------------------------------------

    destination_index = (
        processed_df[
            "destination"
        ]
        .tolist()
        .index(
            destination
        )
    )


    # -----------------------------------------------------
    # Profile score columns.
    #
    # These were created earlier from the destination
    # characteristics.
    # -----------------------------------------------------

    profile_columns = {

        "nature":
            "nature_score",

        "sightseeing":
            "sightseeing_score",

        "water_coastal":
            "water_coastal_score",

        "wildlife":
            "wildlife_score"
    }


    # -----------------------------------------------------
    # Store individual interest contributions.
    # -----------------------------------------------------

    interest_results = []


    # -----------------------------------------------------
    # Process every interest profile.
    # -----------------------------------------------------

    for profile, score_column in (
        profile_columns.items()
    ):

        # -------------------------------------------------
        # User's importance for this interest.
        # -------------------------------------------------

        user_weight = (
            user_interests[
                profile
            ]
        )


        # -------------------------------------------------
        # Get destination's profile score.
        # -------------------------------------------------

        destination_score = float(
            profile_scores_df.iloc[
                destination_index
            ][
                score_column
            ]
        )


        # -------------------------------------------------
        # Raw weighted contribution.
        # -------------------------------------------------

        raw_contribution = (

            destination_score
            *
            user_weight
        )


        interest_results.append({

            "interest":
                profile,

            "destination_score":
                destination_score,

            "user_weight":
                user_weight,

            "raw_contribution":
                raw_contribution
        })


    # -----------------------------------------------------
    # Convert to DataFrame.
    # -----------------------------------------------------

    explanation_df = pd.DataFrame(
        interest_results
    )


    # -----------------------------------------------------
    # Calculate total raw contribution.
    # -----------------------------------------------------

    total_raw_contribution = (
        explanation_df[
            "raw_contribution"
        ].sum()
    )


    # -----------------------------------------------------
    # Calculate the actual production interest score.
    #
    # This guarantees that our explanation corresponds to
    # the score used by the recommendation engine.
    # -----------------------------------------------------

    raw_interest_score = (
        calculate_persona_interest_score(
            user_interests
        )
    )


    destinations = list(
        processed_df[
            "destination"
        ]
    )


    actual_interest_scores = pd.Series(

        raw_interest_score.values,

        index=destinations,

        dtype=float
    )


    actual_score = (
        actual_interest_scores[
            destination
        ]
    )


    # -----------------------------------------------------
    # Scale individual contributions to reconstruct the
    # production personalized-interest score.
    # -----------------------------------------------------

    if total_raw_contribution > 0:

        explanation_df[
            "contribution"
        ] = (

            explanation_df[
                "raw_contribution"
            ]
            /
            total_raw_contribution
            *
            actual_score
        )

    else:

        explanation_df[
            "contribution"
        ] = 0.0


    # -----------------------------------------------------
    # Calculate percentage contribution.
    # -----------------------------------------------------

    if actual_score != 0:

        explanation_df[
            "percentage"
        ] = (

            explanation_df[
                "contribution"
            ]
            /
            actual_score
            *
            100
        )

    else:

        explanation_df[
            "percentage"
        ] = 0.0


    # -----------------------------------------------------
    # Verify mathematical reconstruction.
    # -----------------------------------------------------

    reconstructed_score = (
        explanation_df[
            "contribution"
        ].sum()
    )


    reconstruction_error = abs(

        actual_score
        -
        reconstructed_score
    )


    if reconstruction_error > 1e-10:

        raise ValueError(
            "Interest contributions do not "
            "reconstruct the production score."
        )


    # -----------------------------------------------------
    # Return the complete explanation.
    # -----------------------------------------------------

    return {

        "destination":
            destination,

        "interest_score":
            actual_score,

        "explanation":
            explanation_df,

        "reconstruction_error":
            reconstruction_error
    }


print("=" * 60)
print("✓ INDIVIDUAL INTEREST EXPLANATION ENGINE CREATED")
print("=" * 60)

✓ INDIVIDUAL INTEREST EXPLANATION ENGINE CREATED


In [39]:
# ---------------------------------------------------------
# FIND THE DESTINATION PROFILE SCORE DATAFRAME
#
# Purpose:
# Identify which existing DataFrame in the notebook contains
# the four destination-interest score columns.
# ---------------------------------------------------------

required_profile_columns = {

    "nature_score",
    "sightseeing_score",
    "water_coastal_score",
    "wildlife_score"
}


print("=" * 60)
print("SEARCHING FOR DESTINATION PROFILE SCORE DATAFRAME")
print("=" * 60)


candidate_dataframes = []


# ---------------------------------------------------------
# Inspect all currently defined notebook variables.
# ---------------------------------------------------------

for variable_name, variable_value in list(
    globals().items()
):

    # Only inspect pandas DataFrames.
    if isinstance(
        variable_value,
        pd.DataFrame
    ):

        dataframe_columns = set(
            variable_value.columns
        )


        # Check whether the DataFrame contains all four
        # required interest-profile score columns.
        if required_profile_columns.issubset(
            dataframe_columns
        ):

            candidate_dataframes.append(
                (
                    variable_name,
                    variable_value.shape
                )
            )


# ---------------------------------------------------------
# Display matching DataFrames.
# ---------------------------------------------------------

if len(candidate_dataframes) == 0:

    print(
        "\n✗ No matching profile-score DataFrame found."
    )

else:

    print(
        "\nMatching DataFrames:"
    )

    for name, shape in candidate_dataframes:

        print(
            f"  ✓ {name:<35} shape={shape}"
        )


print(
    "\n" + "=" * 60
)

SEARCHING FOR DESTINATION PROFILE SCORE DATAFRAME

Matching DataFrames:
  ✓ destination_profile_scores          shape=(50, 5)



In [40]:
# ---------------------------------------------------------
# FIXED INDIVIDUAL INTEREST EXPLANATION ENGINE
#
# The earlier function referenced:
#
#     profile_scores_df
#
# But the actual DataFrame in our notebook is:
#
#     destination_profile_scores
#
# This version uses the correct existing DataFrame.
# ---------------------------------------------------------


def explain_interest_profiles(
    user_interests,
    destination
):

    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in processed_df[
        "destination"
    ].values:

        raise ValueError(
            f"Destination '{destination}' not found."
        )


    # -----------------------------------------------------
    # Get the destination index.
    # -----------------------------------------------------

    destination_index = (
        processed_df[
            "destination"
        ]
        .tolist()
        .index(
            destination
        )
    )


    # -----------------------------------------------------
    # Map each user interest to its destination profile
    # score column.
    # -----------------------------------------------------

    profile_columns = {

        "nature":
            "nature_score",

        "sightseeing":
            "sightseeing_score",

        "water_coastal":
            "water_coastal_score",

        "wildlife":
            "wildlife_score"
    }


    # -----------------------------------------------------
    # Calculate the raw contribution of each interest.
    # -----------------------------------------------------

    interest_results = []


    for profile, score_column in (
        profile_columns.items()
    ):

        # -------------------------------------------------
        # User's importance for this interest.
        # -------------------------------------------------

        user_weight = (
            user_interests[
                profile
            ]
        )


        # -------------------------------------------------
        # Destination's score for this interest.
        #
        # IMPORTANT:
        # Use the actual DataFrame that exists in the
        # notebook: destination_profile_scores.
        # -------------------------------------------------

        destination_score = float(

            destination_profile_scores.iloc[
                destination_index
            ][
                score_column
            ]
        )


        # -------------------------------------------------
        # Raw weighted contribution.
        # -------------------------------------------------

        raw_contribution = (

            destination_score
            *
            user_weight
        )


        interest_results.append({

            "interest":
                profile,

            "destination_score":
                destination_score,

            "user_weight":
                user_weight,

            "raw_contribution":
                raw_contribution
        })


    # -----------------------------------------------------
    # Convert results into a DataFrame.
    # -----------------------------------------------------

    explanation_df = pd.DataFrame(
        interest_results
    )


    # -----------------------------------------------------
    # Calculate total raw contribution.
    # -----------------------------------------------------

    total_raw_contribution = (
        explanation_df[
            "raw_contribution"
        ].sum()
    )


    # -----------------------------------------------------
    # Calculate the production interest score using the
    # SAME function used by the recommendation engine.
    # -----------------------------------------------------

    raw_interest_score = (
        calculate_persona_interest_score(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Align the calculated scores with destination names.
    # -----------------------------------------------------

    destinations = list(
        processed_df[
            "destination"
        ]
    )


    actual_interest_scores = pd.Series(

        raw_interest_score.values,

        index=destinations,

        dtype=float
    )


    actual_score = (
        actual_interest_scores[
            destination
        ]
    )


    # -----------------------------------------------------
    # Scale the individual contributions so their sum
    # exactly reconstructs the production interest score.
    # -----------------------------------------------------

    if total_raw_contribution > 0:

        explanation_df[
            "contribution"
        ] = (

            explanation_df[
                "raw_contribution"
            ]
            /
            total_raw_contribution
            *
            actual_score
        )

    else:

        explanation_df[
            "contribution"
        ] = 0.0


    # -----------------------------------------------------
    # Calculate percentage contribution.
    # -----------------------------------------------------

    if actual_score != 0:

        explanation_df[
            "percentage"
        ] = (

            explanation_df[
                "contribution"
            ]
            /
            actual_score
            *
            100
        )

    else:

        explanation_df[
            "percentage"
        ] = 0.0


    # -----------------------------------------------------
    # Verify mathematical consistency.
    # -----------------------------------------------------

    reconstructed_score = (
        explanation_df[
            "contribution"
        ].sum()
    )


    reconstruction_error = abs(

        actual_score
        -
        reconstructed_score
    )


    if reconstruction_error > 1e-10:

        raise ValueError(
            "Interest contributions do not "
            "reconstruct the production score."
        )


    # -----------------------------------------------------
    # Return explanation information.
    # -----------------------------------------------------

    return {

        "destination":
            destination,

        "interest_score":
            actual_score,

        "explanation":
            explanation_df,

        "reconstruction_error":
            reconstruction_error
    }


print("=" * 60)
print("✓ INTEREST EXPLANATION ENGINE FIXED")
print("=" * 60)

✓ INTEREST EXPLANATION ENGINE FIXED


In [41]:
# ---------------------------------------------------------
# TEST INTEREST EXPLANATION FOR MUMBAI
# ---------------------------------------------------------

mumbai_interest_explanation = (
    explain_interest_profiles(

        user_interests=
            sample_user_interests,

        destination="Mumbai"
    )
)


print("=" * 60)
print("WHY DOES MUMBAI MATCH THIS USER'S INTERESTS?")
print("=" * 60)


print(
    f"\nPersonalized interest score: "
    f"{mumbai_interest_explanation['interest_score']:.6f}"
)


print(
    "\nINDIVIDUAL INTEREST CONTRIBUTIONS"
)

print(
    "=" * 60
)


print(
    mumbai_interest_explanation[
        "explanation"
    ][
        [
            "interest",
            "destination_score",
            "user_weight",
            "contribution",
            "percentage"
        ]
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .to_string(
        index=False
    )
)


print(
    f"\nReconstruction error: "
    f"{mumbai_interest_explanation['reconstruction_error']:.12f}"
)


print(
    "\n✓ Interest contributions reconstruct "
    "the production score."
)

WHY DOES MUMBAI MATCH THIS USER'S INTERESTS?

Personalized interest score: 0.522352

INDIVIDUAL INTEREST CONTRIBUTIONS
     interest  destination_score  user_weight  contribution  percentage
       nature           0.437710          1.0      0.190308   36.433000
  sightseeing           0.829316          0.5      0.180286   34.514304
     wildlife           0.531850          0.6      0.138743   26.561292
water_coastal           0.149660          0.2      0.013014    2.491405

Reconstruction error: 0.000000000000

✓ Interest contributions reconstruct the production score.


In [42]:
# ---------------------------------------------------------
# FINAL EXPLAINABLE RECOMMENDATION ENGINE
#
# Purpose:
# Combine:
#
#   1. Travel preference explanation
#   2. Interest explanation
#   3. Similarity explanation
#
# into one complete explanation for a recommendation.
#
# The engine also verifies that every contribution adds up
# exactly to the final recommendation score.
# ---------------------------------------------------------


def get_full_recommendation_explanation(
    user_preferences,
    user_interests,
    destination,
    reference_destination=None,
    mode="hybrid"
):

    # -----------------------------------------------------
    # Validate mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Get the high-level recommendation explanation.
    #
    # This provides:
    #
    #   - preference contribution
    #   - interest contribution
    #   - similarity contribution
    #   - final score
    # -----------------------------------------------------

    overall = explain_recommendation(

        user_preferences=
            user_preferences,

        user_interests=
            user_interests,

        destination=
            destination,

        reference_destination=
            reference_destination,

        mode=
            mode
    )


    # -----------------------------------------------------
    # Get individual travel-preference explanations.
    #
    # Only required for modes that use personalized
    # preferences.
    # -----------------------------------------------------

    if mode in [
        "preference",
        "hybrid"
    ]:

        preference_details = (
            explain_preference_groups(

                user_preferences=
                    user_preferences,

                destination=
                    destination
            )
        )

    else:

        preference_details = None


    # -----------------------------------------------------
    # Get individual interest explanations.
    # -----------------------------------------------------

    if mode in [
        "preference",
        "hybrid"
    ]:

        interest_details = (
            explain_interest_profiles(

                user_interests=
                    user_interests,

                destination=
                    destination
            )
        )

    else:

        interest_details = None


    # -----------------------------------------------------
    # Calculate the exact expected final score.
    # -----------------------------------------------------

    preference_component = 0.0
    interest_component = 0.0
    similarity_component = 0.0


    # -----------------------------------------------------
    # Extract components from the high-level explanation.
    # -----------------------------------------------------

    for _, row in overall[
        "explanation"
    ].iterrows():

        component = row[
            "component"
        ]

        contribution = row[
            "contribution"
        ]


        if component == "Travel preferences":

            preference_component = (
                contribution
            )


        elif component == "Destination interests":

            interest_component = (
                contribution
            )


        elif component == "Similarity":

            similarity_component = (
                contribution
            )


    # -----------------------------------------------------
    # Reconstruct final recommendation score.
    # -----------------------------------------------------

    reconstructed_final_score = (

        preference_component

        +

        interest_component

        +

        similarity_component
    )


    # -----------------------------------------------------
    # Calculate reconstruction error.
    # -----------------------------------------------------

    final_reconstruction_error = abs(

        overall[
            "final_score"
        ]

        -

        reconstructed_final_score
    )


    # -----------------------------------------------------
    # Validate the complete explanation.
    # -----------------------------------------------------

    if final_reconstruction_error > 1e-10:

        raise ValueError(
            "Full explanation does not reconstruct "
            "the final recommendation score."
        )


    # -----------------------------------------------------
    # Return the complete explainability object.
    # -----------------------------------------------------

    return {

        "destination":
            destination,

        "mode":
            mode,

        "final_score":
            overall[
                "final_score"
            ],

        "preference_score":
            overall[
                "preference_score"
            ],

        "interest_score":
            overall[
                "interest_score"
            ],

        "similarity_score":
            overall[
                "similarity_score"
            ],

        "preference_contribution":
            preference_component,

        "interest_contribution":
            interest_component,

        "similarity_contribution":
            similarity_component,

        "preference_details":
            preference_details,

        "interest_details":
            interest_details,

        "overall_explanation":
            overall[
                "explanation"
            ],

        "reconstruction_error":
            final_reconstruction_error
    }


print("=" * 60)
print("✓ FINAL EXPLAINABLE RECOMMENDATION ENGINE CREATED")
print("=" * 60)

✓ FINAL EXPLAINABLE RECOMMENDATION ENGINE CREATED


In [43]:
# ---------------------------------------------------------
# COMPLETE EXPLANATION TEST — MUMBAI
# ---------------------------------------------------------

mumbai_full_explanation = (
    get_full_recommendation_explanation(

        user_preferences=
            sample_user_preferences,

        user_interests=
            sample_user_interests,

        destination="Mumbai",

        reference_destination="Goa",

        mode="hybrid"
    )
)


print("=" * 60)
print("COMPLETE RECOMMENDATION EXPLANATION")
print("=" * 60)


print(
    f"\nDestination: "
    f"{mumbai_full_explanation['destination']}"
)


print(
    f"Mode: "
    f"{mumbai_full_explanation['mode']}"
)


print(
    f"\nFinal recommendation score: "
    f"{mumbai_full_explanation['final_score']:.6f}"
)


print(
    "\n" + "=" * 60
)

print(
    "HIGH-LEVEL SCORE CONTRIBUTIONS"
)

print(
    "=" * 60
)


print(
    mumbai_full_explanation[
        "overall_explanation"
    ].to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Travel preference details.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "TRAVEL PREFERENCE DETAILS"
)

print(
    "=" * 60
)


print(
    mumbai_full_explanation[
        "preference_details"
    ][
        "explanation"
    ][
        [
            "group",
            "destination_score",
            "user_weight",
            "contribution",
            "percentage"
        ]
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Interest details.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "INTEREST DETAILS"
)

print(
    "=" * 60
)


print(
    mumbai_full_explanation[
        "interest_details"
    ][
        "explanation"
    ][
        [
            "interest",
            "destination_score",
            "user_weight",
            "contribution",
            "percentage"
        ]
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Final mathematical validation.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "EXPLANATION VALIDATION"
)

print(
    "=" * 60
)


print(
    f"\nReconstructed final score: "
    f"{mumbai_full_explanation['final_score']:.6f}"
)


print(
    f"Reconstruction error: "
    f"{mumbai_full_explanation['reconstruction_error']:.12f}"
)


print(
    "\n✓ Complete recommendation explanation "
    "is mathematically consistent."
)

COMPLETE RECOMMENDATION EXPLANATION

Destination: Mumbai
Mode: hybrid

Final recommendation score: 0.680986

HIGH-LEVEL SCORE CONTRIBUTIONS
            component  contribution  percentage
   Travel preferences      0.315669   46.354676
Destination interests      0.146259   21.477467
           Similarity      0.219059   32.167857

TRAVEL PREFERENCE DETAILS
                      group  destination_score  user_weight  contribution  percentage
                     budget           0.921839          0.9      0.224231   29.834147
                    weather           0.684796          0.8      0.148064   19.700027
                     flight           0.836881          0.6      0.135710   18.056376
              accommodation           0.819973          0.6      0.132969   17.691570
destination_characteristics           0.511611          0.8      0.110618   14.717881

INTEREST DETAILS
     interest  destination_score  user_weight  contribution  percentage
       nature           0.437710   

In [44]:
# ---------------------------------------------------------
# MULTI-DESTINATION EXPLAINABILITY VALIDATION
#
# Purpose:
# Verify that the explainability engine works consistently
# across multiple destinations and not only for Mumbai.
#
# For every destination we validate:
#
#   1. Final recommendation score
#   2. Preference contribution
#   3. Interest contribution
#   4. Similarity contribution
#   5. Preference-level explanation
#   6. Interest-level explanation
#   7. Mathematical reconstruction
# ---------------------------------------------------------


print("=" * 60)
print("MULTI-DESTINATION EXPLAINABILITY VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Destinations selected from our previous recommendation
# and evaluation results.
# ---------------------------------------------------------

evaluation_destinations = [

    "Mumbai",
    "Bengaluru",
    "Delhi",
    "Manali",
    "Chennai",
    "Jaipur",
    "Kochi",
    "Pune"
]


# ---------------------------------------------------------
# Store validation results.
# ---------------------------------------------------------

validation_results = []


# ---------------------------------------------------------
# Test every destination.
# ---------------------------------------------------------

for destination in evaluation_destinations:

    # -----------------------------------------------------
    # Generate the complete explanation.
    # -----------------------------------------------------

    result = (
        get_full_recommendation_explanation(

            user_preferences=
                sample_user_preferences,

            user_interests=
                sample_user_interests,

            destination=
                destination,

            reference_destination=
                "Goa",

            mode=
                "hybrid"
        )
    )


    # -----------------------------------------------------
    # Extract preference explanation.
    # -----------------------------------------------------

    preference_details = (
        result[
            "preference_details"
        ][
            "explanation"
        ]
    )


    # -----------------------------------------------------
    # Extract interest explanation.
    # -----------------------------------------------------

    interest_details = (
        result[
            "interest_details"
        ][
            "explanation"
        ]
    )


    # -----------------------------------------------------
    # Validate expected number of components.
    # -----------------------------------------------------

    preference_component_count = len(
        preference_details
    )


    interest_component_count = len(
        interest_details
    )


    preference_components_valid = (
        preference_component_count == 5
    )


    interest_components_valid = (
        interest_component_count == 4
    )


    # -----------------------------------------------------
    # Validate that all contribution values are present.
    # -----------------------------------------------------

    contributions = [

        result[
            "preference_contribution"
        ],

        result[
            "interest_contribution"
        ],

        result[
            "similarity_contribution"
        ]
    ]


    contributions_valid = all(

        pd.notna(value)

        for value in contributions
    )


    # -----------------------------------------------------
    # Check mathematical reconstruction.
    # -----------------------------------------------------

    reconstruction_error = (
        result[
            "reconstruction_error"
        ]
    )


    reconstruction_valid = (
        reconstruction_error <= 1e-10
    )


    # -----------------------------------------------------
    # Store validation result.
    # -----------------------------------------------------

    validation_results.append({

        "destination":
            destination,

        "final_score":
            result[
                "final_score"
            ],

        "preference_contribution":
            result[
                "preference_contribution"
            ],

        "interest_contribution":
            result[
                "interest_contribution"
            ],

        "similarity_contribution":
            result[
                "similarity_contribution"
            ],

        "preference_components":
            preference_component_count,

        "interest_components":
            interest_component_count,

        "reconstruction_error":
            reconstruction_error,

        "valid":
            (
                preference_components_valid
                and
                interest_components_valid
                and
                contributions_valid
                and
                reconstruction_valid
            )
    })


# ---------------------------------------------------------
# Convert results to DataFrame.
# ---------------------------------------------------------

validation_df = pd.DataFrame(
    validation_results
)


# ---------------------------------------------------------
# Display validation results.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "EXPLAINABILITY VALIDATION RESULTS"
)

print(
    "=" * 60
)


print(
    validation_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Calculate summary statistics.
# ---------------------------------------------------------

total_destinations = len(
    validation_df
)


successful_destinations = (
    validation_df[
        "valid"
    ].sum()
)


maximum_error = (
    validation_df[
        "reconstruction_error"
    ].max()
)


print(
    "\n" + "=" * 60
)

print(
    "VALIDATION SUMMARY"
)

print(
    "=" * 60
)


print(
    f"\nDestinations tested: "
    f"{total_destinations}"
)


print(
    f"Successful explanations: "
    f"{successful_destinations}"
    f" / "
    f"{total_destinations}"
)


print(
    f"Maximum reconstruction error: "
    f"{maximum_error:.12f}"
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if successful_destinations != total_destinations:

    raise ValueError(
        "One or more destinations failed "
        "explainability validation."
    )


if maximum_error > 1e-10:

    raise ValueError(
        "Explainability reconstruction error "
        "exceeds tolerance."
    )


print(
    "\n✓ All destinations have valid "
    "explanations."
)


print(
    "✓ Preference explanations contain "
    "all 5 groups."
)


print(
    "✓ Interest explanations contain "
    "all 4 profiles."
)


print(
    "✓ Final scores are mathematically "
    "reconstructable."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ MULTI-DESTINATION EXPLAINABILITY "
    "VALIDATION PASSED"
)

print(
    "=" * 60
)

MULTI-DESTINATION EXPLAINABILITY VALIDATION

EXPLAINABILITY VALIDATION RESULTS
destination  final_score  preference_contribution  interest_contribution  similarity_contribution  preference_components  interest_components  reconstruction_error  valid
     Mumbai     0.680986                 0.315669               0.146259                 0.219059                      5                    4                   0.0   True
  Bengaluru     0.557246                 0.299617               0.112444                 0.145186                      5                    4                   0.0   True
      Delhi     0.549401                 0.291234               0.114517                 0.143650                      5                    4                   0.0   True
     Manali     0.488002                 0.267285               0.136403                 0.084313                      5                    4                   0.0   True
    Chennai     0.466498                 0.291057               0.

In [45]:
# ---------------------------------------------------------
# MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION
#
# Purpose:
# Test whether the recommendation engine behaves safely
# when some destination data is unavailable.
#
# We specifically validate:
#
#   1. No crashes
#   2. No NaN recommendation scores
#   3. No infinite scores
#   4. Missing data is handled through availability-aware
#      weighting
#   5. Destinations with different availability patterns
#      can still be ranked
#   6. Recommendations remain explainable
#
# Known dataset situation:
#
#   Flight          = 8 / 50 available
#   Accommodation   = 39 / 50 available
#   Weather         = 50 / 50 available
# ---------------------------------------------------------


print("=" * 60)
print("MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Check that the required availability columns exist.
# ---------------------------------------------------------

required_availability_columns = [

    "budget_available",

    "flight_available",

    "accommodation_available",

    "weather_available",

    "destination_characteristics_available"
]


missing_availability_columns = [

    column

    for column in required_availability_columns

    if column not in availability_df.columns
]


if missing_availability_columns:

    raise ValueError(
        "Missing availability columns: "
        f"{missing_availability_columns}"
    )


# ---------------------------------------------------------
# Display overall availability statistics.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "DATA AVAILABILITY SUMMARY"
)

print(
    "=" * 60
)


for column in required_availability_columns:

    available_count = int(
        availability_df[
            column
        ].sum()
    )

    total_count = len(
        availability_df
    )

    print(
        f"{column:<45}"
        f"{available_count:>3} / "
        f"{total_count}"
    )


# ---------------------------------------------------------
# Generate recommendations using the sample user.
#
# We use hybrid mode with Goa as the reference because this
# exercises both personalization and similarity.
# ---------------------------------------------------------

try:

    robustness_recommendations = (
        get_recommendations(

            user_preferences=
                sample_user_preferences,

            user_interests=
                sample_user_interests,

            reference_destination=
                "Goa",

            mode=
                "hybrid",

            top_k=50
        )
    )

    recommendation_generation_success = True

except Exception as error:

    recommendation_generation_success = False

    print(
        "\n✗ Recommendation generation failed:"
    )

    print(
        type(error).__name__,
        str(error)
    )


# ---------------------------------------------------------
# Stop if recommendation generation itself failed.
# ---------------------------------------------------------

if not recommendation_generation_success:

    raise ValueError(
        "Availability robustness test failed because "
        "the recommendation engine could not generate "
        "recommendations."
    )


# ---------------------------------------------------------
# Check recommendation score columns.
# ---------------------------------------------------------

required_score_columns = [

    "personalized_preference_score",

    "personalized_interest_score",

    "similarity_score",

    "final_recommendation_score"
]


missing_score_columns = [

    column

    for column in required_score_columns

    if column not in robustness_recommendations.columns
]


if missing_score_columns:

    raise ValueError(
        "Missing recommendation score columns: "
        f"{missing_score_columns}"
    )


# ---------------------------------------------------------
# Validate that no recommendation score contains NaN.
# ---------------------------------------------------------

nan_counts = (
    robustness_recommendations[
        required_score_columns
    ]
    .isna()
    .sum()
)


print(
    "\n" + "=" * 60
)

print(
    "MISSING SCORE VALIDATION"
)

print(
    "=" * 60
)


print(
    nan_counts.to_string()
)


if nan_counts.sum() > 0:

    raise ValueError(
        "NaN recommendation scores detected."
    )


print(
    "\n✓ No NaN recommendation scores."
)


# ---------------------------------------------------------
# Validate that no score is infinite.
# ---------------------------------------------------------

infinite_counts = {}

for column in required_score_columns:

    infinite_counts[
        column
    ] = int(
        np.isinf(
            robustness_recommendations[
                column
            ]
        ).sum()
    )


print(
    "\n" + "=" * 60
)

print(
    "INFINITE SCORE VALIDATION"
)

print(
    "=" * 60
)


for column, count in infinite_counts.items():

    print(
        f"{column:<40}{count}"
    )


if sum(
    infinite_counts.values()
) > 0:

    raise ValueError(
        "Infinite recommendation scores detected."
    )


print(
    "\n✓ No infinite recommendation scores."
)


# ---------------------------------------------------------
# Check that every destination has a valid final score.
# ---------------------------------------------------------

valid_destination_count = (
    robustness_recommendations[
        "destination"
    ]
    .nunique()
)


total_destination_count = (
    len(
        recommendation_destinations
    )
)


print(
    "\n" + "=" * 60
)

print(
    "DESTINATION COVERAGE"
)

print(
    "=" * 60
)


print(
    f"\nUnique destinations returned: "
    f"{valid_destination_count}"
)


print(
    f"Expected destinations: "
    f"{total_destination_count}"
)


if valid_destination_count != total_destination_count:

    raise ValueError(
        "Recommendation engine did not return "
        "all expected destinations."
    )


print(
    "\n✓ All destinations received valid scores."
)


# ---------------------------------------------------------
# Analyze destinations with the least available data.
#
# This is important because these destinations exercise
# the missing-data handling logic most strongly.
# ---------------------------------------------------------

availability_analysis = (
    availability_df[
        [
            "destination",

            "budget_available",

            "flight_available",

            "accommodation_available",

            "weather_available",

            "destination_characteristics_available",

            "total_active_weight"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Calculate number of active preference groups.
# ---------------------------------------------------------

availability_analysis[
    "active_groups"
] = (

    availability_analysis[
        required_availability_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Display destinations with the fewest active groups.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "DESTINATIONS WITH LOWEST DATA AVAILABILITY"
)

print(
    "=" * 60
)


low_availability = (
    availability_analysis
    .sort_values(
        [
            "active_groups",
            "total_active_weight"
        ],
        ascending=[
            True,
            True
        ]
    )
    .head(10)
)


print(
    low_availability.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Compare recommendation scores of low-availability
# destinations.
# ---------------------------------------------------------

low_availability_destinations = (
    low_availability[
        "destination"
    ]
    .tolist()
)


low_availability_scores = (
    robustness_recommendations[
        robustness_recommendations[
            "destination"
        ].isin(
            low_availability_destinations
        )
    ][
        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ]
    .sort_values(
        "final_recommendation_score",
        ascending=False
    )
)


print(
    "\n" + "=" * 60
)

print(
    "LOW-AVAILABILITY DESTINATION SCORES"
)

print(
    "=" * 60
)


print(
    low_availability_scores.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Test explainability on a low-availability destination.
#
# We select the destination with the fewest active groups.
# ---------------------------------------------------------

robustness_test_destination = (
    low_availability.iloc[0][
        "destination"
    ]
)


print(
    "\n" + "=" * 60
)

print(
    "LOW-AVAILABILITY EXPLAINABILITY TEST"
)

print(
    "=" * 60
)


print(
    f"\nTesting destination: "
    f"{robustness_test_destination}"
)


try:

    robustness_explanation = (
        get_full_recommendation_explanation(

            user_preferences=
                sample_user_preferences,

            user_interests=
                sample_user_interests,

            destination=
                robustness_test_destination,

            reference_destination=
                "Goa",

            mode=
                "hybrid"
        )
    )

except Exception as error:

    raise ValueError(
        "Explainability failed for a destination "
        "with limited data."
    ) from error


print(
    f"\nFinal score: "
    f"{robustness_explanation['final_score']:.6f}"
)


print(
    f"Reconstruction error: "
    f"{robustness_explanation['reconstruction_error']:.12f}"
)


# ---------------------------------------------------------
# Final robustness checks.
# ---------------------------------------------------------

if not np.isfinite(
    robustness_explanation[
        "final_score"
    ]
):

    raise ValueError(
        "Non-finite explanation score detected."
    )


if (
    robustness_explanation[
        "reconstruction_error"
    ]
    >
    1e-10
):

    raise ValueError(
        "Explainability reconstruction failed "
        "for low-availability destination."
    )


# ---------------------------------------------------------
# Final summary.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "ROBUSTNESS SUMMARY"
)

print(
    "=" * 60
)


print(
    "\n✓ Recommendation generation succeeded."
)


print(
    "✓ No NaN recommendation scores."
)


print(
    "✓ No infinite recommendation scores."
)


print(
    "✓ All destinations received scores."
)


print(
    "✓ Low-availability destinations were "
    "successfully ranked."
)


print(
    "✓ Low-availability destination remained "
    "explainable."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ MISSING-DATA & AVAILABILITY "
    "ROBUSTNESS EVALUATION PASSED"
)

print(
    "=" * 60
)

MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION


NameError: name 'availability_df' is not defined

In [47]:
# ---------------------------------------------------------
# FIND AVAILABILITY DATAFRAME
#
# Purpose:
# Locate the existing DataFrame containing the availability
# flags used by the recommendation system.
# ---------------------------------------------------------

required_availability_columns = {

    "budget_available",

    "flight_available",

    "accommodation_available",

    "weather_available",

    "destination_characteristics_available"
}


print("=" * 60)
print("SEARCHING FOR AVAILABILITY DATAFRAME")
print("=" * 60)


candidate_availability_dataframes = []


# ---------------------------------------------------------
# Search all currently defined pandas DataFrames.
# ---------------------------------------------------------

for variable_name, variable_value in list(
    globals().items()
):

    if isinstance(
        variable_value,
        pd.DataFrame
    ):

        dataframe_columns = set(
            variable_value.columns
        )


        # -------------------------------------------------
        # Check whether all required availability columns
        # exist in this DataFrame.
        # -------------------------------------------------

        if required_availability_columns.issubset(
            dataframe_columns
        ):

            candidate_availability_dataframes.append(
                (
                    variable_name,
                    variable_value.shape
                )
            )


# ---------------------------------------------------------
# Display the matching DataFrames.
# ---------------------------------------------------------

if len(
    candidate_availability_dataframes
) == 0:

    print(
        "\n✗ No availability DataFrame found."
    )

else:

    print(
        "\nMatching DataFrames:"
    )

    for name, shape in (
        candidate_availability_dataframes
    ):

        print(
            f"  ✓ {name:<40} "
            f"shape={shape}"
        )


print(
    "\n" + "=" * 60
)

SEARCHING FOR AVAILABILITY DATAFRAME

✗ No availability DataFrame found.



In [48]:
# ---------------------------------------------------------
# FIND EXISTING AVAILABILITY-RELATED VARIABLES
#
# Purpose:
# Search the current notebook namespace for variables whose
# names or contents are related to availability.
# ---------------------------------------------------------

print("=" * 60)
print("SEARCHING CURRENT NOTEBOOK FOR AVAILABILITY VARIABLES")
print("=" * 60)


# ---------------------------------------------------------
# First search variable names.
# ---------------------------------------------------------

print("\nVariables containing 'availability':")

for variable_name in sorted(
    globals().keys()
):

    if "availability" in variable_name.lower():

        print(
            f"  - {variable_name}"
        )


# ---------------------------------------------------------
# Search for variables containing the individual availability
# terms in their names.
# ---------------------------------------------------------

keywords = [
    "budget",
    "flight",
    "accommodation",
    "weather",
    "weight"
]


print(
    "\nPotential related variables:"
)


for variable_name in sorted(
    globals().keys()
):

    name_lower = variable_name.lower()


    if any(
        keyword in name_lower
        for keyword in keywords
    ):

        # -------------------------------------------------
        # Avoid printing Python/system internals.
        # -------------------------------------------------

        if not variable_name.startswith("_"):

            print(
                f"  - {variable_name}"
            )


print(
    "\n" + "=" * 60
)

SEARCHING CURRENT NOTEBOOK FOR AVAILABILITY VARIABLES

Variables containing 'availability':
  - candidate_availability_dataframes
  - required_availability_columns

Potential related variables:
  - interest_weight
  - travel_weight



In [49]:
# ---------------------------------------------------------
# REBUILD AVAILABILITY TABLE FOR ROBUSTNESS EVALUATION
#
# Purpose:
# Reconstruct the availability information from the
# validated processed dataset.
#
# We are NOT changing the recommendation model.
# This DataFrame is only used for evaluation.
# ---------------------------------------------------------


print("=" * 60)
print("REBUILDING AVAILABILITY DATA FOR EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# Verify that the source dataset exists.
# ---------------------------------------------------------

if "processed_df" not in globals():

    raise NameError(
        "processed_df is not available. "
        "Please recreate the processed dataset first."
    )


# ---------------------------------------------------------
# Required source columns.
# ---------------------------------------------------------

required_source_columns = [

    "destination",

    "flight_count",

    "hotel_count",

    "weather_available",

    "accommodation_available",

    "flight_available"
]


missing_source_columns = [

    column

    for column in required_source_columns

    if column not in processed_df.columns
]


if missing_source_columns:

    raise ValueError(
        "Required source columns are missing: "
        f"{missing_source_columns}"
    )


# ---------------------------------------------------------
# Create the evaluation availability DataFrame.
# ---------------------------------------------------------

availability_df = pd.DataFrame({

    "destination":
        processed_df[
            "destination"
        ].values,

    # -----------------------------------------------------
    # Budget availability.
    #
    # Budget information depends on hotel pricing in our
    # current scoring pipeline.
    # -----------------------------------------------------

    "budget_available": (

        processed_df[
            "hotel_count"
        ]
        .notna()
        .astype(int)
    ),

    # -----------------------------------------------------
    # Flight availability.
    # -----------------------------------------------------

    "flight_available": (

        processed_df[
            "flight_available"
        ]
        .fillna(0)
        .astype(int)
    ),

    # -----------------------------------------------------
    # Accommodation availability.
    # -----------------------------------------------------

    "accommodation_available": (

        processed_df[
            "accommodation_available"
        ]
        .fillna(0)
        .astype(int)
    ),

    # -----------------------------------------------------
    # Weather availability.
    # -----------------------------------------------------

    "weather_available": (

        processed_df[
            "weather_available"
        ]
        .fillna(0)
        .astype(int)
    ),

    # -----------------------------------------------------
    # Destination characteristics are available for every
    # destination because these are derived from the
    # destination feature data.
    # -----------------------------------------------------

    "destination_characteristics_available":
        1
})


# ---------------------------------------------------------
# Reconstruct the total active preference weight.
#
# These weights correspond to the current sample user.
# ---------------------------------------------------------

availability_df[
    "total_active_weight"
] = (

    availability_df[
        "budget_available"
    ]
    *
    sample_user_preferences[
        "budget"
    ]

    +

    availability_df[
        "flight_available"
    ]
    *
    sample_user_preferences[
        "flight"
    ]

    +

    availability_df[
        "accommodation_available"
    ]
    *
    sample_user_preferences[
        "accommodation"
    ]

    +

    availability_df[
        "weather_available"
    ]
    *
    sample_user_preferences[
        "weather"
    ]

    +

    availability_df[
        "destination_characteristics_available"
    ]
    *
    sample_user_preferences[
        "destination_characteristics"
    ]
)


# ---------------------------------------------------------
# Display reconstructed availability statistics.
# ---------------------------------------------------------

print(
    "\nAvailability counts:"
)


availability_columns = [

    "budget_available",

    "flight_available",

    "accommodation_available",

    "weather_available",

    "destination_characteristics_available"
]


for column in availability_columns:

    available = int(
        availability_df[
            column
        ].sum()
    )

    total = len(
        availability_df
    )

    print(
        f"{column:<40}"
        f"{available:>3} / {total}"
    )


# ---------------------------------------------------------
# Display first few records.
# ---------------------------------------------------------

print(
    "\nFirst 10 availability records:"
)


print(
    availability_df
    .head(10)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Basic validation.
# ---------------------------------------------------------

if len(
    availability_df
) != len(
    processed_df
):

    raise ValueError(
        "Availability table does not contain "
        "the same number of destinations."
    )


if availability_df[
    "destination"
].duplicated().any():

    raise ValueError(
        "Duplicate destinations found in "
        "availability table."
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ AVAILABILITY DATA REBUILT FOR EVALUATION"
)

print(
    "=" * 60
)

REBUILDING AVAILABILITY DATA FOR EVALUATION

Availability counts:
budget_available                         39 / 50
flight_available                          8 / 50
accommodation_available                  39 / 50
weather_available                        50 / 50
destination_characteristics_available    50 / 50

First 10 availability records:
destination  budget_available  flight_available  accommodation_available  weather_available  destination_characteristics_available  total_active_weight
       Agra                 1                 0                        1                  1                                      1                  3.1
  Ahmedabad                 1                 0                        1                  1                                      1                  3.1
  Alappuzha                 1                 0                        1                  1                                      1                  3.1
   Amritsar                 1                 0  

In [50]:
# ---------------------------------------------------------
# MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION
#
# Purpose:
# Verify that destinations with missing flight/hotel data
# can still receive valid recommendation scores and remain
# explainable.
# ---------------------------------------------------------


print("=" * 60)
print("MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION")
print("=" * 60)


# ---------------------------------------------------------
# 1. DISPLAY AVAILABILITY SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("DATA AVAILABILITY SUMMARY")
print("=" * 60)


availability_columns = [

    "budget_available",
    "flight_available",
    "accommodation_available",
    "weather_available",
    "destination_characteristics_available"
]


for column in availability_columns:

    available_count = int(
        availability_df[column].sum()
    )

    total_count = len(
        availability_df
    )

    print(
        f"{column:<45}"
        f"{available_count:>3} / {total_count}"
    )


# ---------------------------------------------------------
# 2. GENERATE FULL RECOMMENDATIONS
#
# We evaluate all 50 destinations rather than only the
# Top-10 recommendations.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("GENERATING RECOMMENDATIONS")
print("=" * 60)


try:

    robustness_recommendations = get_recommendations(

        user_preferences=
            sample_user_preferences,

        user_interests=
            sample_user_interests,

        reference_destination=
            "Goa",

        mode=
            "hybrid",

        top_k=50
    )

except Exception as error:

    raise ValueError(
        "Recommendation generation failed."
    ) from error


print(
    "\n✓ Recommendation generation succeeded."
)


# ---------------------------------------------------------
# 3. CHECK REQUIRED SCORE COLUMNS
# ---------------------------------------------------------

required_score_columns = [

    "personalized_preference_score",
    "personalized_interest_score",
    "similarity_score",
    "final_recommendation_score"
]


missing_score_columns = [

    column

    for column in required_score_columns

    if column not in robustness_recommendations.columns
]


if missing_score_columns:

    raise ValueError(
        "Missing recommendation columns: "
        f"{missing_score_columns}"
    )


# ---------------------------------------------------------
# 4. CHECK FOR NaN VALUES
# ---------------------------------------------------------

nan_counts = (

    robustness_recommendations[
        required_score_columns
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 60)
print("NaN SCORE VALIDATION")
print("=" * 60)


print(
    nan_counts.to_string()
)


if nan_counts.sum() > 0:

    raise ValueError(
        "NaN values detected in recommendation scores."
    )


print(
    "\n✓ No NaN recommendation scores."
)


# ---------------------------------------------------------
# 5. CHECK FOR INFINITE VALUES
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("INFINITE SCORE VALIDATION")
print("=" * 60)


infinite_counts = {}


for column in required_score_columns:

    infinite_counts[column] = int(

        np.isinf(
            robustness_recommendations[
                column
            ]
        ).sum()

    )


for column, count in infinite_counts.items():

    print(
        f"{column:<40}{count}"
    )


if sum(
    infinite_counts.values()
) > 0:

    raise ValueError(
        "Infinite recommendation scores detected."
    )


print(
    "\n✓ No infinite recommendation scores."
)


# ---------------------------------------------------------
# 6. CHECK DESTINATION COVERAGE
# ---------------------------------------------------------

returned_destinations = set(

    robustness_recommendations[
        "destination"
    ]
)


expected_destinations = set(

    recommendation_destinations
)


print("\n" + "=" * 60)
print("DESTINATION COVERAGE")
print("=" * 60)


print(
    f"\nReturned destinations: "
    f"{len(returned_destinations)}"
)


print(
    f"Expected destinations: "
    f"{len(expected_destinations)}"
)


missing_destinations = (
    expected_destinations
    -
    returned_destinations
)


if missing_destinations:

    raise ValueError(
        "Destinations missing from recommendations: "
        f"{sorted(missing_destinations)}"
    )


print(
    "\n✓ All 50 destinations received scores."
)


# ---------------------------------------------------------
# 7. FIND DESTINATIONS WITH LOW DATA AVAILABILITY
#
# These are the most important cases for this test.
# ---------------------------------------------------------

availability_analysis = (
    availability_df.copy()
)


availability_analysis[
    "active_groups"
] = (

    availability_analysis[
        availability_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Sort by number of available groups.
# ---------------------------------------------------------

low_availability = (

    availability_analysis

    .sort_values(
        [
            "active_groups",
            "total_active_weight"
        ],
        ascending=[
            True,
            True
        ]
    )

    .head(10)
)


print("\n" + "=" * 60)
print("DESTINATIONS WITH LOWEST DATA AVAILABILITY")
print("=" * 60)


print(
    low_availability.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 8. JOIN AVAILABILITY WITH RECOMMENDATION SCORES
# ---------------------------------------------------------

low_availability_destinations = (

    low_availability[
        "destination"
    ]
    .tolist()
)


low_availability_scores = (

    robustness_recommendations[

        robustness_recommendations[
            "destination"
        ].isin(
            low_availability_destinations
        )

    ][

        [
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "similarity_score",
            "final_recommendation_score"
        ]

    ]

    .sort_values(
        "final_recommendation_score",
        ascending=False
    )
)


print("\n" + "=" * 60)
print("LOW-AVAILABILITY DESTINATION SCORES")
print("=" * 60)


print(
    low_availability_scores.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 9. TEST EXPLAINABILITY ON THE LOWEST-AVAILABILITY
#    DESTINATION
# ---------------------------------------------------------

robustness_test_destination = (

    low_availability.iloc[0][
        "destination"
    ]
)


print("\n" + "=" * 60)
print("LOW-AVAILABILITY EXPLAINABILITY TEST")
print("=" * 60)


print(
    f"\nTesting destination: "
    f"{robustness_test_destination}"
)


try:

    robustness_explanation = (
        get_full_recommendation_explanation(

            user_preferences=
                sample_user_preferences,

            user_interests=
                sample_user_interests,

            destination=
                robustness_test_destination,

            reference_destination=
                "Goa",

            mode=
                "hybrid"
        )
    )

except Exception as error:

    raise ValueError(
        "Explainability failed for low-availability "
        "destination."
    ) from error


print(
    f"\nFinal recommendation score: "
    f"{robustness_explanation['final_score']:.6f}"
)


print(
    f"Reconstruction error: "
    f"{robustness_explanation['reconstruction_error']:.12f}"
)


# ---------------------------------------------------------
# 10. FINAL VALIDATION
# ---------------------------------------------------------

if not np.isfinite(

    robustness_explanation[
        "final_score"
    ]

):

    raise ValueError(
        "Final explanation score is not finite."
    )


if (

    robustness_explanation[
        "reconstruction_error"
    ]

    >

    1e-10

):

    raise ValueError(
        "Explainability reconstruction failed."
    )


# ---------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)


print(
    "\n✓ Recommendation generation succeeded."
)


print(
    "✓ No NaN recommendation scores."
)


print(
    "✓ No infinite recommendation scores."
)


print(
    "✓ All destinations received valid scores."
)


print(
    "✓ Low-availability destinations were ranked."
)


print(
    "✓ Low-availability destination remained explainable."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ MISSING-DATA & AVAILABILITY "
    "ROBUSTNESS EVALUATION PASSED"
)

print(
    "=" * 60
)

MISSING-DATA & AVAILABILITY ROBUSTNESS EVALUATION

DATA AVAILABILITY SUMMARY
budget_available                              39 / 50
flight_available                               8 / 50
accommodation_available                       39 / 50
weather_available                             50 / 50
destination_characteristics_available         50 / 50

GENERATING RECOMMENDATIONS

✓ Recommendation generation succeeded.

NaN SCORE VALIDATION
personalized_preference_score    0
personalized_interest_score      0
similarity_score                 0
final_recommendation_score       0

✓ No NaN recommendation scores.

INFINITE SCORE VALIDATION
personalized_preference_score           0
personalized_interest_score             0
similarity_score                        0
final_recommendation_score              0

✓ No infinite recommendation scores.

DESTINATION COVERAGE

Returned destinations: 49
Expected destinations: 50


ValueError: Destinations missing from recommendations: ['Goa']

In [51]:
# ---------------------------------------------------------
# CORRECTED DESTINATION COVERAGE VALIDATION
#
# Goa is intentionally excluded because it is being used as
# the reference destination for similarity recommendations.
#
# Therefore:
#
#   Total destinations       = 50
#   Reference destination    = Goa
#   Eligible recommendations = 49
# ---------------------------------------------------------


print("=" * 60)
print("CORRECTED DESTINATION COVERAGE VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Get destinations actually returned by the engine.
# ---------------------------------------------------------

returned_destinations = set(

    robustness_recommendations[
        "destination"
    ]
)


# ---------------------------------------------------------
# Start with all known destinations.
# ---------------------------------------------------------

expected_destinations = set(

    recommendation_destinations
)


# ---------------------------------------------------------
# Remove the reference destination.
#
# Goa must NOT appear because it is the destination the
# user asked to find alternatives to.
# ---------------------------------------------------------

reference_destination = "Goa"

expected_destinations.discard(
    reference_destination
)


# ---------------------------------------------------------
# Find any genuinely missing destinations.
# ---------------------------------------------------------

missing_destinations = (

    expected_destinations
    -
    returned_destinations
)


# ---------------------------------------------------------
# Find unexpected destinations.
# ---------------------------------------------------------

unexpected_destinations = (

    returned_destinations
    -
    expected_destinations
)


# ---------------------------------------------------------
# Display coverage.
# ---------------------------------------------------------

print(
    f"\nTotal model destinations: "
    f"{len(recommendation_destinations)}"
)


print(
    f"Reference destination: "
    f"{reference_destination}"
)


print(
    f"Expected recommendation destinations: "
    f"{len(expected_destinations)}"
)


print(
    f"Returned recommendation destinations: "
    f"{len(returned_destinations)}"
)


print(
    f"Excluded reference destination present: "
    f"{reference_destination in returned_destinations}"
)


# ---------------------------------------------------------
# Validate that Goa was excluded.
# ---------------------------------------------------------

if reference_destination in returned_destinations:

    raise ValueError(
        "Reference destination Goa was incorrectly "
        "included in recommendations."
    )


# ---------------------------------------------------------
# Validate that no eligible destination is missing.
# ---------------------------------------------------------

if missing_destinations:

    raise ValueError(
        "Eligible destinations are missing: "
        f"{sorted(missing_destinations)}"
    )


# ---------------------------------------------------------
# Validate that no unexpected destination was returned.
# ---------------------------------------------------------

if unexpected_destinations:

    raise ValueError(
        "Unexpected destinations returned: "
        f"{sorted(unexpected_destinations)}"
    )


# ---------------------------------------------------------
# Final result.
# ---------------------------------------------------------

print(
    "\n✓ 49 eligible destinations returned."
)


print(
    "✓ Goa correctly excluded as reference destination."
)


print(
    "✓ No eligible destinations missing."
)


print(
    "✓ No unexpected destinations returned."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ DESTINATION COVERAGE VALIDATION PASSED"
)

print(
    "=" * 60
)

CORRECTED DESTINATION COVERAGE VALIDATION

Total model destinations: 50
Reference destination: Goa
Expected recommendation destinations: 49
Returned recommendation destinations: 49
Excluded reference destination present: False

✓ 49 eligible destinations returned.
✓ Goa correctly excluded as reference destination.
✓ No eligible destinations missing.
✓ No unexpected destinations returned.

✓ DESTINATION COVERAGE VALIDATION PASSED


In [52]:
# ---------------------------------------------------------
# LOW-AVAILABILITY ROBUSTNESS ANALYSIS
#
# Purpose:
# Identify destinations with the least available travel
# data and verify that the recommendation engine can still
# score and explain them correctly.
# ---------------------------------------------------------


print("=" * 60)
print("LOW-AVAILABILITY ROBUSTNESS ANALYSIS")
print("=" * 60)


# ---------------------------------------------------------
# Availability columns used to determine how much data is
# available for each destination.
# ---------------------------------------------------------

availability_columns = [

    "budget_available",
    "flight_available",
    "accommodation_available",
    "weather_available",
    "destination_characteristics_available"
]


# ---------------------------------------------------------
# Copy the availability table so that the original table
# remains unchanged.
# ---------------------------------------------------------

availability_analysis = (
    availability_df.copy()
)


# ---------------------------------------------------------
# Count how many preference groups have available data.
# ---------------------------------------------------------

availability_analysis[
    "active_groups"
] = (

    availability_analysis[
        availability_columns
    ]
    .sum(axis=1)
)


# ---------------------------------------------------------
# Sort destinations from lowest data availability to
# highest data availability.
#
# total_active_weight is used as a secondary sorting
# criterion.
# ---------------------------------------------------------

low_availability = (

    availability_analysis

    .sort_values(
        [
            "active_groups",
            "total_active_weight"
        ],

        ascending=[
            True,
            True
        ]
    )

    .head(10)
)


print(
    "\n" + "=" * 60
)

print(
    "DESTINATIONS WITH LOWEST DATA AVAILABILITY"
)

print(
    "=" * 60
)


print(
    low_availability.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Obtain the destination names.
# ---------------------------------------------------------

low_availability_destinations = (

    low_availability[
        "destination"
    ]
    .tolist()
)


# ---------------------------------------------------------
# Join availability information with recommendation
# scores.
#
# This lets us directly check whether low-data destinations
# still receive valid recommendation scores.
# ---------------------------------------------------------

low_availability_scores = (

    robustness_recommendations[

        robustness_recommendations[
            "destination"
        ].isin(
            low_availability_destinations
        )

    ][

        [
            "destination",

            "personalized_preference_score",

            "personalized_interest_score",

            "similarity_score",

            "final_recommendation_score"
        ]

    ]

    .sort_values(
        "final_recommendation_score",
        ascending=False
    )
)


print(
    "\n" + "=" * 60
)

print(
    "LOW-AVAILABILITY DESTINATION SCORES"
)

print(
    "=" * 60
)


print(
    low_availability_scores.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Validate that all low-availability destinations received
# recommendation scores.
# ---------------------------------------------------------

missing_low_availability = (

    set(
        low_availability_destinations
    )

    -

    set(
        low_availability_scores[
            "destination"
        ]
    )
)


if missing_low_availability:

    raise ValueError(
        "Low-availability destinations did not receive "
        "recommendation scores: "
        f"{sorted(missing_low_availability)}"
    )


# ---------------------------------------------------------
# Validate that recommendation scores are finite.
# ---------------------------------------------------------

if not np.isfinite(

    low_availability_scores[
        "final_recommendation_score"
    ]

).all():

    raise ValueError(
        "Non-finite scores detected for "
        "low-availability destinations."
    )


print(
    "\n✓ All low-availability destinations "
    "received finite recommendation scores."
)


# ---------------------------------------------------------
# Select the destination with the least available data.
# ---------------------------------------------------------

robustness_test_destination = (

    low_availability.iloc[0][
        "destination"
    ]
)


print(
    "\n" + "=" * 60
)

print(
    "LOW-AVAILABILITY EXPLAINABILITY TEST"
)

print(
    "=" * 60
)


print(
    f"\nTesting destination: "
    f"{robustness_test_destination}"
)


# ---------------------------------------------------------
# Generate a complete explanation for this destination.
# ---------------------------------------------------------

try:

    robustness_explanation = (

        get_full_recommendation_explanation(

            user_preferences=
                sample_user_preferences,

            user_interests=
                sample_user_interests,

            destination=
                robustness_test_destination,

            reference_destination=
                "Goa",

            mode=
                "hybrid"
        )
    )

except Exception as error:

    raise ValueError(
        "Explainability failed for the "
        "low-availability destination."
    ) from error


# ---------------------------------------------------------
# Display explanation result.
# ---------------------------------------------------------

print(
    f"\nFinal recommendation score: "
    f"{robustness_explanation['final_score']:.6f}"
)


print(
    f"Preference contribution: "
    f"{robustness_explanation['preference_contribution']:.6f}"
)


print(
    f"Interest contribution: "
    f"{robustness_explanation['interest_contribution']:.6f}"
)


print(
    f"Similarity contribution: "
    f"{robustness_explanation['similarity_contribution']:.6f}"
)


print(
    f"Reconstruction error: "
    f"{robustness_explanation['reconstruction_error']:.12f}"
)


# ---------------------------------------------------------
# Validate explanation score.
# ---------------------------------------------------------

if not np.isfinite(

    robustness_explanation[
        "final_score"
    ]

):

    raise ValueError(
        "Final explanation score is not finite."
    )


# ---------------------------------------------------------
# Validate mathematical reconstruction.
# ---------------------------------------------------------

if (

    robustness_explanation[
        "reconstruction_error"
    ]

    >

    1e-10

):

    raise ValueError(
        "Explanation reconstruction failed for "
        "low-availability destination."
    )


print(
    "\n✓ Low-availability recommendation "
    "is explainable."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ LOW-AVAILABILITY ROBUSTNESS TEST PASSED"
)

print(
    "=" * 60
)

LOW-AVAILABILITY ROBUSTNESS ANALYSIS

DESTINATIONS WITH LOWEST DATA AVAILABILITY
  destination  budget_available  flight_available  accommodation_available  weather_available  destination_characteristics_available  total_active_weight  active_groups
      Andaman                 0                 0                        0                  1                                      1                  1.6              2
        Coorg                 0                 0                        0                  1                                      1                  1.6              2
  Jim Corbett                 0                 0                        0                  1                                      1                  1.6              2
    Kaziranga                 0                 0                        0                  1                                      1                  1.6              2
   Kodaikanal                 0                 0                        0

In [56]:
# ---------------------------------------------------------
# USER PROFILE EDGE-CASE ROBUSTNESS TEST
#
# Purpose:
# Verify that the recommendation engine remains stable when
# users provide extreme or unusual preference profiles.
#
# Tests:
#
#   1. All-zero profile
#   2. Budget-only profile
#   3. Nature-only profile
#   4. All-max profile
#
# We validate:
#
#   - No crashes
#   - No NaN values
#   - No infinite values
#   - Recommendations are generated
#   - Scores remain finite
# ---------------------------------------------------------


print("=" * 60)
print("USER PROFILE EDGE-CASE ROBUSTNESS TEST")
print("=" * 60)


# ---------------------------------------------------------
# Define test profiles.
# ---------------------------------------------------------

edge_case_profiles = {

    "All Zero":

    {
        "preferences": {

            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {

            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Budget Only":

    {
        "preferences": {

            "budget": 1.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {

            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Nature Only":

    {
        "preferences": {

            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {

            "nature": 1.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "All Maximum":

    {
        "preferences": {

            "budget": 1.0,
            "flight": 1.0,
            "accommodation": 1.0,
            "weather": 1.0,
            "destination_characteristics": 1.0
        },

        "interests": {

            "nature": 1.0,
            "sightseeing": 1.0,
            "water_coastal": 1.0,
            "wildlife": 1.0
        }
    }
}


# ---------------------------------------------------------
# Store validation results.
# ---------------------------------------------------------

edge_case_results = []


# ---------------------------------------------------------
# Evaluate each profile.
# ---------------------------------------------------------

for profile_name, profile in (
    edge_case_profiles.items()
):

    print(
        "\n" + "=" * 60
    )

    print(
        profile_name
    )

    print(
        "=" * 60
    )


    try:

        results = get_recommendations(

            user_preferences=
                profile[
                    "preferences"
                ],

            user_interests=
                profile[
                    "interests"
                ],

            reference_destination=
                None,

            mode=
                "preference",

            top_k=10
        )


        # -------------------------------------------------
        # Check recommendation count.
        # -------------------------------------------------

        recommendation_count = len(
            results
        )


        # -------------------------------------------------
        # Check score validity.
        # -------------------------------------------------

        score_column = (
            "final_recommendation_score"
        )


        has_nan = (
            results[
                score_column
            ]
            .isna()
            .any()
        )


        has_infinite = (
            np.isinf(
                results[
                    score_column
                ]
            )
            .any()
        )


        # -------------------------------------------------
        # Store results.
        # -------------------------------------------------

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                recommendation_count,

            "nan_scores":
                has_nan,

            "infinite_scores":
                has_infinite,

            "valid":
                (
                    recommendation_count > 0
                    and
                    not has_nan
                    and
                    not has_infinite
                )
        })


        # -------------------------------------------------
        # Display top 5.
        # -------------------------------------------------

        print(
            "\nTop 5:"
        )


        print(
            results[
                [
                    "destination",
                    score_column
                ]
            ]
            .head(5)
            .to_string(
                index=False
            )
        )


        print(
            "\n✓ Profile evaluated successfully."
        )


    except Exception as error:

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                0,

            "nan_scores":
                True,

            "infinite_scores":
                True,

            "valid":
                False
        })


        print(
            "\n✗ Profile failed:"
        )

        print(
            type(error).__name__,
            str(error)
        )


# ---------------------------------------------------------
# Convert results to DataFrame.
# ---------------------------------------------------------

edge_case_results_df = pd.DataFrame(
    edge_case_results
)


# ---------------------------------------------------------
# Display summary.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "EDGE-CASE SUMMARY"
)

print(
    "=" * 60
)


print(
    edge_case_results_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if not edge_case_results_df[
    "valid"
].all():

    raise ValueError(
        "One or more user profile edge cases failed."
    )


print(
    "\n✓ All user profile edge cases "
    "produced valid recommendations."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ USER PROFILE EDGE-CASE "
    "ROBUSTNESS TEST PASSED"
)

print(
    "=" * 60
)

USER PROFILE EDGE-CASE ROBUSTNESS TEST

All Zero

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

✓ Profile evaluated successfully.

Budget Only

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

✓ Profile evaluated successfully.

Nature Only

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

✓ Profile evaluated successfully.

All Maximum

Top 5:
destination  final_recommendation_score
     Mumbai    

ValueError: One or more user profile edge cases failed.

In [55]:
# ---------------------------------------------------------
# FINAL ZERO-SAFE PREFERENCE SCORING FUNCTION
# ---------------------------------------------------------

def calculate_preference_scores(
    user_preferences
):

    # -----------------------------------------------------
    # Initialize accumulators.
    # -----------------------------------------------------

    weighted_total = pd.Series(
        0.0,
        index=normalized_features.index
    )

    active_total = pd.Series(
        0.0,
        index=normalized_features.index
    )


    # -----------------------------------------------------
    # Process every user preference.
    # -----------------------------------------------------

    for group, user_weight in (
        user_preferences.items()
    ):

        user_weight = float(
            user_weight
        )


        # -------------------------------------------------
        # Zero-weight preferences have no influence.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        if group not in preference_groups:

            raise ValueError(
                f"Unknown preference group: {group}"
            )


        group_features = (
            preference_groups[
                group
            ]
        )


        feature_scores = []


        # -------------------------------------------------
        # Process features in this preference group.
        # -------------------------------------------------

        for feature in group_features:

            if feature not in normalized_features.columns:

                continue


            # -------------------------------------------------
            # Preserve the original missing-value information.
            # -------------------------------------------------

            original_values = (
                normalized_features[
                    feature
                ]
            )


            available_mask = (
                original_values.notna()
            )


            # -------------------------------------------------
            # Fill missing values only for score calculation.
            # -------------------------------------------------

            feature_values = (
                original_values
                .fillna(0.0)
            )


            # -------------------------------------------------
            # Apply preference direction.
            #
            # Lower-is-better features are reversed so that
            # higher score always means better.
            # -------------------------------------------------

            direction = (
                preference_direction_map.get(
                    feature,
                    1
                )
            )


            if direction == -1:

                feature_values = (
                    1.0
                    -
                    feature_values
                )


            feature_scores.append(
                (
                    feature_values,
                    available_mask
                )
            )


        # -------------------------------------------------
        # Skip groups with no usable features.
        # -------------------------------------------------

        if not feature_scores:

            continue


        # -------------------------------------------------
        # Combine feature scores.
        # -------------------------------------------------

        score_matrix = pd.concat(

            [
                values

                for values, _ in feature_scores
            ],

            axis=1
        )


        availability_matrix = pd.concat(

            [
                available

                for _, available in feature_scores
            ],

            axis=1
        )


        # -------------------------------------------------
        # Calculate the group score.
        # -------------------------------------------------

        group_score = (
            score_matrix
            .mean(axis=1)
        )


        # -------------------------------------------------
        # A group is considered available for a destination
        # if at least one underlying feature exists.
        # -------------------------------------------------

        group_available = (
            availability_matrix
            .any(axis=1)
        )


        # -------------------------------------------------
        # Add weighted contribution.
        # -------------------------------------------------

        weighted_total += (

            group_score
            *
            user_weight
            *
            group_available.astype(float)
        )


        # -------------------------------------------------
        # Add active user weight.
        # -------------------------------------------------

        active_total += (

            user_weight
            *
            group_available.astype(float)
        )


    # -----------------------------------------------------
    # Create safe output.
    # -----------------------------------------------------

    preference_scores = pd.Series(
        0.0,
        index=normalized_features.index
    )


    # -----------------------------------------------------
    # Only divide where active weight exists.
    #
    # This is the key fix.
    # -----------------------------------------------------

    valid_mask = (
        active_total > 0
    )


    preference_scores.loc[
        valid_mask
    ] = (

        weighted_total.loc[
            valid_mask
        ]

        /

        active_total.loc[
            valid_mask
        ]
    )


    # -----------------------------------------------------
    # Final safety cleanup.
    # -----------------------------------------------------

    preference_scores = (

        preference_scores

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    return preference_scores


print("=" * 60)
print("✓ ZERO-SAFE PREFERENCE SCORING FUNCTION CREATED")
print("=" * 60)

✓ ZERO-SAFE PREFERENCE SCORING FUNCTION CREATED


In [57]:
# ---------------------------------------------------------
# TRACE NaN THROUGH THE RECOMMENDATION PIPELINE
#
# Purpose:
# Determine exactly which scoring component becomes NaN
# for an extreme user profile.
#
# We will test:
#   - preference score
#   - interest score
#   - base personalized score
#   - similarity score
#   - final recommendation score
# ---------------------------------------------------------


print("=" * 60)
print("TRACING NaN THROUGH RECOMMENDATION PIPELINE")
print("=" * 60)


# ---------------------------------------------------------
# Use the Budget Only profile because it previously produced
# NaN values.
# ---------------------------------------------------------

test_preferences = {

    "budget": 1.0,
    "flight": 0.0,
    "accommodation": 0.0,
    "weather": 0.0,
    "destination_characteristics": 0.0
}


test_interests = {

    "nature": 0.0,
    "sightseeing": 0.0,
    "water_coastal": 0.0,
    "wildlife": 0.0
}


# ---------------------------------------------------------
# STEP 1 — Preference score
# ---------------------------------------------------------

preference_test = (
    calculate_preference_scores(
        test_preferences
    )
)


print(
    "\n" + "=" * 60
)

print(
    "STEP 1 — PREFERENCE SCORE"
)

print(
    "=" * 60
)


print(
    f"NaN count: "
    f"{preference_test.isna().sum()}"
)


print(
    f"Infinity count: "
    f"{np.isinf(preference_test).sum()}"
)


print(
    f"Minimum: "
    f"{preference_test.min()}"
)


print(
    f"Maximum: "
    f"{preference_test.max()}"
)


print(
    "\nFirst 10 preference scores:"
)

print(
    preference_test
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# STEP 2 — Interest score
#
# We need to determine the actual interest-scoring function
# available in the notebook.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "STEP 2 — SEARCHING FOR INTEREST SCORE FUNCTION"
)

print(
    "=" * 60
)


interest_function_candidates = [

    name

    for name in globals()

    if (
        "interest" in name.lower()
        and
        callable(
            globals()[name]
        )
    )
]


print(
    "Possible interest functions:"
)


for name in interest_function_candidates:

    print(
        f"  - {name}"
    )


# ---------------------------------------------------------
# Stop here intentionally.
#
# We don't want to guess the interest function name.
# The output will tell us which function is currently used.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "PREFERENCE TRACE COMPLETED"
)

print(
    "=" * 60
)

TRACING NaN THROUGH RECOMMENDATION PIPELINE

STEP 1 — PREFERENCE SCORE
NaN count: 0
Infinity count: 0
Minimum: 0.0
Maximum: 0.982376789352474

First 10 preference scores:
0    0.749985
1    0.715571
2    0.640005
3    0.645654
4    0.000000
5    0.982377
6    0.714927
7    0.718637
8    0.719583
9    0.000000

STEP 2 — SEARCHING FOR INTEREST SCORE FUNCTION
Possible interest functions:
  - calculate_interest_scores
  - calculate_persona_interest_score
  - explain_interest_profiles

PREFERENCE TRACE COMPLETED


In [58]:
# ---------------------------------------------------------
# TRACE INTEREST SCORE FOR EDGE-CASE PROFILE
#
# Purpose:
# Determine whether the interest-scoring stage introduces
# NaN values when all interest weights are zero.
# ---------------------------------------------------------


print("=" * 60)
print("STEP 2 — INTEREST SCORE TRACE")
print("=" * 60)


# ---------------------------------------------------------
# Budget-only profile has no active interests.
# ---------------------------------------------------------

test_interests = {

    "nature": 0.0,

    "sightseeing": 0.0,

    "water_coastal": 0.0,

    "wildlife": 0.0
}


# ---------------------------------------------------------
# Calculate interest scores.
# ---------------------------------------------------------

interest_test = (
    calculate_interest_scores(
        test_interests
    )
)


# ---------------------------------------------------------
# Display basic information.
# ---------------------------------------------------------

print(
    f"\nObject type: "
    f"{type(interest_test)}"
)


# ---------------------------------------------------------
# Handle Series output.
# ---------------------------------------------------------

if isinstance(
    interest_test,
    pd.Series
):

    print(
        f"Number of scores: "
        f"{len(interest_test)}"
    )

    print(
        f"NaN count: "
        f"{interest_test.isna().sum()}"
    )

    print(
        f"Infinity count: "
        f"{np.isinf(interest_test).sum()}"
    )

    print(
        f"Minimum: "
        f"{interest_test.min()}"
    )

    print(
        f"Maximum: "
        f"{interest_test.max()}"
    )

    print(
        "\nFirst 10 interest scores:"
    )

    print(
        interest_test
        .head(10)
        .to_string()
    )


# ---------------------------------------------------------
# Handle DataFrame output.
# ---------------------------------------------------------

elif isinstance(
    interest_test,
    pd.DataFrame
):

    print(
        "\nInterest score columns:"
    )

    print(
        interest_test.columns.tolist()
    )

    print(
        "\nNaN counts:"
    )

    print(
        interest_test.isna().sum()
    )


# ---------------------------------------------------------
# Handle scalar output.
# ---------------------------------------------------------

else:

    print(
        "\nInterest score result:"
    )

    print(
        interest_test
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ INTEREST SCORE TRACE COMPLETED"
)

print(
    "=" * 60
)

STEP 2 — INTEREST SCORE TRACE

Object type: <class 'pandas.Series'>
Number of scores: 50
NaN count: 50
Infinity count: 0
Minimum: nan
Maximum: nan

First 10 interest scores:
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN

✓ INTEREST SCORE TRACE COMPLETED


In [59]:
# ---------------------------------------------------------
# ZERO-SAFE PERSONALIZED INTEREST SCORING
#
# Purpose:
# Prevent NaN values when:
#
#   1. All user interest weights are zero.
#   2. Some interest profiles have no usable information.
#
# The function preserves the existing weighted-average
# behavior for normal user profiles.
# ---------------------------------------------------------


def calculate_interest_scores(
    user_interests
):

    # -----------------------------------------------------
    # Initialize weighted score and active weight for every
    # destination.
    # -----------------------------------------------------

    weighted_total = pd.Series(
        0.0,
        index=profile_scores_df.index
    )

    active_total = pd.Series(
        0.0,
        index=profile_scores_df.index
    )


    # -----------------------------------------------------
    # Process each interest profile.
    # -----------------------------------------------------

    for interest, user_weight in (
        user_interests.items()
    ):

        user_weight = float(
            user_weight
        )


        # -------------------------------------------------
        # Zero-weight interests have no influence.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        # -------------------------------------------------
        # Verify that the interest profile exists.
        # -------------------------------------------------

        if interest not in interest_profiles:

            raise ValueError(
                f"Unknown interest profile: {interest}"
            )


        # -------------------------------------------------
        # Determine the score column corresponding to this
        # interest.
        #
        # Example:
        #
        # nature
        #     →
        # nature_score
        # -------------------------------------------------

        score_column = (
            f"{interest}_score"
        )


        if score_column not in profile_scores_df.columns:

            raise ValueError(
                f"Missing interest score column: "
                f"{score_column}"
            )


        # -------------------------------------------------
        # Preserve original missing-value information.
        # -------------------------------------------------

        original_values = (
            profile_scores_df[
                score_column
            ]
        )


        available_mask = (
            original_values.notna()
        )


        # -------------------------------------------------
        # Replace missing values with zero for calculation.
        # -------------------------------------------------

        interest_scores = (
            original_values
            .fillna(0.0)
        )


        # -------------------------------------------------
        # Add weighted contribution.
        # -------------------------------------------------

        weighted_total += (

            interest_scores
            *
            user_weight
            *
            available_mask.astype(float)
        )


        # -------------------------------------------------
        # Track active user weight.
        # -------------------------------------------------

        active_total += (

            user_weight
            *
            available_mask.astype(float)
        )


    # -----------------------------------------------------
    # Create a safe output initialized to zero.
    # -----------------------------------------------------

    interest_scores = pd.Series(
        0.0,
        index=profile_scores_df.index
    )


    # -----------------------------------------------------
    # Only divide where an active interest weight exists.
    #
    # This prevents:
    #
    #     0 / 0 = NaN
    #
    # from occurring.
    # -----------------------------------------------------

    valid_mask = (
        active_total > 0
    )


    interest_scores.loc[
        valid_mask
    ] = (

        weighted_total.loc[
            valid_mask
        ]

        /

        active_total.loc[
            valid_mask
        ]
    )


    # -----------------------------------------------------
    # Final numerical safety.
    # -----------------------------------------------------

    interest_scores = (

        interest_scores

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    return interest_scores


print("=" * 60)
print("✓ ZERO-SAFE INTEREST SCORING FUNCTION CREATED")
print("=" * 60)

✓ ZERO-SAFE INTEREST SCORING FUNCTION CREATED


In [61]:
# ---------------------------------------------------------
# RESTORE INTEREST PROFILE SCORE DATAFRAME
#
# The notebook contains the destination interest profile
# data under `destination_profile_scores`.
#
# The scoring function expects the name `profile_scores_df`,
# so we create that reference here.
# ---------------------------------------------------------

profile_scores_df = destination_profile_scores.copy()


print("=" * 60)
print("INTEREST PROFILE DATAFRAME RESTORED")
print("=" * 60)

print(
    f"\nShape: {profile_scores_df.shape}"
)

print(
    "\nColumns:"
)

print(
    profile_scores_df.columns.tolist()
)


# ---------------------------------------------------------
# Basic validation.
# ---------------------------------------------------------

if "destination" not in profile_scores_df.columns:

    raise ValueError(
        "Destination column is missing from "
        "profile_scores_df."
    )


if len(profile_scores_df) != 50:

    raise ValueError(
        "Expected 50 destination profiles, "
        f"found {len(profile_scores_df)}."
    )


print(
    "\n✓ Interest profile dataframe is ready."
)

INTEREST PROFILE DATAFRAME RESTORED

Shape: (50, 5)

Columns:
['destination', 'nature_score', 'sightseeing_score', 'water_coastal_score', 'wildlife_score']

✓ Interest profile dataframe is ready.


In [62]:
# ---------------------------------------------------------
# VERIFY ZERO-SAFE INTEREST SCORING
# ---------------------------------------------------------


print("=" * 60)
print("VERIFYING ZERO-SAFE INTEREST SCORING")
print("=" * 60)


zero_interest_profile = {

    "nature": 0.0,

    "sightseeing": 0.0,

    "water_coastal": 0.0,

    "wildlife": 0.0
}


zero_interest_scores = (
    calculate_interest_scores(
        zero_interest_profile
    )
)


print(
    f"\nNumber of destinations: "
    f"{len(zero_interest_scores)}"
)


print(
    f"NaN count: "
    f"{zero_interest_scores.isna().sum()}"
)


print(
    f"Infinity count: "
    f"{np.isinf(zero_interest_scores).sum()}"
)


print(
    f"Minimum score: "
    f"{zero_interest_scores.min()}"
)


print(
    f"Maximum score: "
    f"{zero_interest_scores.max()}"
)


print(
    "\nFirst 10 scores:"
)


print(
    zero_interest_scores
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# Validation.
# ---------------------------------------------------------

if zero_interest_scores.isna().any():

    raise ValueError(
        "NaN values still exist in zero-interest scoring."
    )


if np.isinf(
    zero_interest_scores
).any():

    raise ValueError(
        "Infinite values still exist in zero-interest scoring."
    )


print(
    "\n✓ Zero-interest profile produces finite scores."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ INTEREST ZERO-WEIGHT FIX VERIFIED"
)

print(
    "=" * 60
)

VERIFYING ZERO-SAFE INTEREST SCORING

Number of destinations: 50
NaN count: 0
Infinity count: 0
Minimum score: 0.0
Maximum score: 0.0

First 10 scores:
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0

✓ Zero-interest profile produces finite scores.

✓ INTEREST ZERO-WEIGHT FIX VERIFIED


In [64]:
# ---------------------------------------------------------
# USER PROFILE EDGE-CASE ROBUSTNESS TEST
#
# Purpose:
# Verify that the recommendation engine remains stable for
# extreme or unusual user preference profiles.
#
# Test cases:
#   1. All Zero
#   2. Budget Only
#   3. Nature Only
#   4. All Maximum
#
# We validate:
#   - Recommendations are generated
#   - No NaN scores
#   - No infinite scores
#   - Scores remain finite
# ---------------------------------------------------------


print("=" * 60)
print("USER PROFILE EDGE-CASE ROBUSTNESS TEST")
print("=" * 60)


# ---------------------------------------------------------
# Define extreme user profiles.
# ---------------------------------------------------------

edge_case_profiles = {

    "All Zero": {

        "preferences": {
            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Budget Only": {

        "preferences": {
            "budget": 1.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Nature Only": {

        "preferences": {
            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 1.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "All Maximum": {

        "preferences": {
            "budget": 1.0,
            "flight": 1.0,
            "accommodation": 1.0,
            "weather": 1.0,
            "destination_characteristics": 1.0
        },

        "interests": {
            "nature": 1.0,
            "sightseeing": 1.0,
            "water_coastal": 1.0,
            "wildlife": 1.0
        }
    }
}


# ---------------------------------------------------------
# Store results from every edge-case test.
# ---------------------------------------------------------

edge_case_results = []


# ---------------------------------------------------------
# Evaluate every profile.
# ---------------------------------------------------------

for profile_name, profile in edge_case_profiles.items():

    print("\n" + "=" * 60)
    print(profile_name)
    print("=" * 60)


    try:

        # -------------------------------------------------
        # Generate recommendations.
        #
        # No reference destination is used because these
        # tests evaluate the user-profile behavior itself.
        # -------------------------------------------------

        results = get_recommendations(

            user_preferences=profile["preferences"],

            user_interests=profile["interests"],

            reference_destination=None,

            mode="preference",

            top_k=10
        )


        # -------------------------------------------------
        # Check recommendation count.
        # -------------------------------------------------

        recommendation_count = len(results)


        # -------------------------------------------------
        # Determine the score column returned by the engine.
        # -------------------------------------------------

        if "final_recommendation_score" in results.columns:

            score_column = (
                "final_recommendation_score"
            )

        elif "combined_score" in results.columns:

            score_column = (
                "combined_score"
            )

        elif "personalized_preference_score" in results.columns:

            score_column = (
                "personalized_preference_score"
            )

        else:

            raise ValueError(
                "No recognized recommendation score "
                "column found."
            )


        # -------------------------------------------------
        # Check for NaN values.
        # -------------------------------------------------

        nan_scores = (

            results[score_column]
            .isna()
            .sum()
        )


        # -------------------------------------------------
        # Check for infinite values.
        # -------------------------------------------------

        infinite_scores = (

            np.isinf(
                results[score_column]
            )
            .sum()
        )


        # -------------------------------------------------
        # Determine whether this profile passed.
        # -------------------------------------------------

        valid = (

            recommendation_count > 0

            and

            nan_scores == 0

            and

            infinite_scores == 0
        )


        # -------------------------------------------------
        # Store the validation result.
        # -------------------------------------------------

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                recommendation_count,

            "nan_scores":
                nan_scores,

            "infinite_scores":
                infinite_scores,

            "valid":
                valid
        })


        # -------------------------------------------------
        # Display top five recommendations.
        # -------------------------------------------------

        print("\nTop 5:")

        print(

            results[
                [
                    "destination",
                    score_column
                ]
            ]
            .head(5)
            .to_string(
                index=False
            )
        )


        # -------------------------------------------------
        # Display score validation.
        # -------------------------------------------------

        print(
            f"\nNaN scores: {nan_scores}"
        )

        print(
            f"Infinite scores: {infinite_scores}"
        )


        if valid:

            print(
                "\n✓ Profile evaluated successfully."
            )

        else:

            print(
                "\n✗ Profile validation failed."
            )


    except Exception as error:

        # -------------------------------------------------
        # Record failed profiles instead of stopping
        # immediately, so all edge cases can be inspected.
        # -------------------------------------------------

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                0,

            "nan_scores":
                -1,

            "infinite_scores":
                -1,

            "valid":
                False
        })


        print(
            "\n✗ Profile failed:"
        )

        print(
            f"{type(error).__name__}: {error}"
        )


# ---------------------------------------------------------
# Convert validation results into a DataFrame.
# ---------------------------------------------------------

edge_case_results_df = pd.DataFrame(
    edge_case_results
)


# ---------------------------------------------------------
# Display final summary.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("EDGE-CASE SUMMARY")
print("=" * 60)


print(
    edge_case_results_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Final validation.
#
# Every edge case must:
#
#   - produce recommendations
#   - contain zero NaN scores
#   - contain zero infinite scores
# ---------------------------------------------------------

if not edge_case_results_df["valid"].all():

    failed_profiles = (

        edge_case_results_df.loc[
            ~edge_case_results_df["valid"],
            "profile"
        ]
        .tolist()
    )


    raise ValueError(
        "One or more user profile edge cases failed: "
        f"{failed_profiles}"
    )


# ---------------------------------------------------------
# Final success message.
# ---------------------------------------------------------

print(
    "\n✓ All user profile edge cases "
    "produced valid recommendations."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ USER PROFILE EDGE-CASE "
    "ROBUSTNESS TEST PASSED"
)

print(
    "=" * 60
)

USER PROFILE EDGE-CASE ROBUSTNESS TEST

All Zero

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

NaN scores: 10
Infinite scores: 0

✗ Profile validation failed.

Budget Only

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

NaN scores: 10
Infinite scores: 0

✗ Profile validation failed.

Nature Only

Top 5:
destination  final_recommendation_score
       Agra                         NaN
  Ahmedabad                         NaN
  Alappuzha                         NaN
   Amritsar                         NaN
    Andaman                         NaN

NaN scores: 10
Infinite scores: 0

✗ Profile valid

ValueError: One or more user profile edge cases failed: ['All Zero', 'Budget Only', 'Nature Only']

In [65]:
# ---------------------------------------------------------
# FINAL SCORE PIPELINE DIAGNOSTIC
#
# Purpose:
# Find exactly where NaN is introduced after the individual
# preference and interest scores have already been calculated.
# ---------------------------------------------------------


print("=" * 60)
print("FINAL SCORE PIPELINE DIAGNOSTIC")
print("=" * 60)


# ---------------------------------------------------------
# Use the same All Zero profile that is currently failing.
# ---------------------------------------------------------

diagnostic_preferences = {

    "budget": 0.0,
    "flight": 0.0,
    "accommodation": 0.0,
    "weather": 0.0,
    "destination_characteristics": 0.0
}


diagnostic_interests = {

    "nature": 0.0,
    "sightseeing": 0.0,
    "water_coastal": 0.0,
    "wildlife": 0.0
}


# ---------------------------------------------------------
# STEP 1 — Preference scores
# ---------------------------------------------------------

diagnostic_preference_scores = (
    calculate_preference_scores(
        diagnostic_preferences
    )
)


print("\nSTEP 1 — PREFERENCE SCORES")

print(
    f"NaN: "
    f"{diagnostic_preference_scores.isna().sum()}"
)

print(
    f"Index type: "
    f"{type(diagnostic_preference_scores.index)}"
)

print(
    f"Index sample: "
    f"{diagnostic_preference_scores.index[:5].tolist()}"
)


# ---------------------------------------------------------
# STEP 2 — Interest scores
# ---------------------------------------------------------

diagnostic_interest_scores = (
    calculate_interest_scores(
        diagnostic_interests
    )
)


print("\nSTEP 2 — INTEREST SCORES")

print(
    f"NaN: "
    f"{diagnostic_interest_scores.isna().sum()}"
)

print(
    f"Index type: "
    f"{type(diagnostic_interest_scores.index)}"
)

print(
    f"Index sample: "
    f"{diagnostic_interest_scores.index[:5].tolist()}"
)


# ---------------------------------------------------------
# STEP 3 — Convert both score Series to explicit
# destination-indexed Series.
#
# This is the critical diagnostic.
# ---------------------------------------------------------

print(
    "\nSTEP 3 — DESTINATION INDEX CHECK"
)


# Preference scores currently use numerical indexes.
# Build a destination-indexed version using the processed
# destination ordering.

diagnostic_preference_by_destination = pd.Series(

    diagnostic_preference_scores.to_numpy(),

    index=processed_df[
        "destination"
    ].astype(str).values
)


# Interest scores use the profile dataframe ordering.
# Convert them to destination-indexed form as well.

diagnostic_interest_by_destination = pd.Series(

    diagnostic_interest_scores.to_numpy(),

    index=profile_scores_df[
        "destination"
    ].astype(str).values
)


print(
    "\nPreference destination index sample:"
)

print(
    diagnostic_preference_by_destination
    .head()
)


print(
    "\nInterest destination index sample:"
)

print(
    diagnostic_interest_by_destination
    .head()
)


# ---------------------------------------------------------
# STEP 4 — Check whether the destination names align.
# ---------------------------------------------------------

preference_destinations = set(
    diagnostic_preference_by_destination.index
)

interest_destinations = set(
    diagnostic_interest_by_destination.index
)


print(
    "\nPreference destinations:",
    len(preference_destinations)
)

print(
    "Interest destinations:",
    len(interest_destinations)
)


print(
    "Common destinations:",
    len(
        preference_destinations
        &
        interest_destinations
    )
)


# ---------------------------------------------------------
# STEP 5 — Explicitly align the two scores.
# ---------------------------------------------------------

common_destinations = sorted(

    preference_destinations
    &
    interest_destinations
)


aligned_preference = (
    diagnostic_preference_by_destination
    .reindex(common_destinations)
)


aligned_interest = (
    diagnostic_interest_by_destination
    .reindex(common_destinations)
)


print(
    "\nSTEP 5 — AFTER DESTINATION ALIGNMENT"
)


print(
    f"Preference NaN: "
    f"{aligned_preference.isna().sum()}"
)

print(
    f"Interest NaN: "
    f"{aligned_interest.isna().sum()}"
)


# ---------------------------------------------------------
# STEP 6 — Reproduce the base personalized score.
#
# Existing production weights:
#
#   travel preference = 0.6
#   interest          = 0.4
# ---------------------------------------------------------

diagnostic_base_score = (

    0.6
    *
    aligned_preference

    +

    0.4
    *
    aligned_interest
)


print(
    "\nSTEP 6 — BASE PERSONALIZED SCORE"
)


print(
    f"NaN: "
    f"{diagnostic_base_score.isna().sum()}"
)

print(
    f"Infinity: "
    f"{np.isinf(diagnostic_base_score).sum()}"
)

print(
    "\nFirst 10:"
)

print(
    diagnostic_base_score
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# STEP 7 — Test the hybrid calculation.
#
# Hybrid weights:
#
#   personalized score = 0.7
#   similarity          = 0.3
#
# For this diagnostic we don't need a reference destination;
# we only want to see whether the weighted combination itself
# can create NaN.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "PIPELINE DIAGNOSTIC SUMMARY"
)

print(
    "=" * 60
)


print(
    f"Preference NaN: "
    f"{diagnostic_preference_scores.isna().sum()}"
)

print(
    f"Interest NaN: "
    f"{diagnostic_interest_scores.isna().sum()}"
)

print(
    f"Aligned preference NaN: "
    f"{aligned_preference.isna().sum()}"
)

print(
    f"Aligned interest NaN: "
    f"{aligned_interest.isna().sum()}"
)

print(
    f"Base score NaN: "
    f"{diagnostic_base_score.isna().sum()}"
)


print(
    "\n✓ Diagnostic completed."
)

FINAL SCORE PIPELINE DIAGNOSTIC

STEP 1 — PREFERENCE SCORES
NaN: 0
Index type: <class 'pandas.RangeIndex'>
Index sample: [0, 1, 2, 3, 4]

STEP 2 — INTEREST SCORES
NaN: 0
Index type: <class 'pandas.RangeIndex'>
Index sample: [0, 1, 2, 3, 4]

STEP 3 — DESTINATION INDEX CHECK

Preference destination index sample:
Agra         0.0
Ahmedabad    0.0
Alappuzha    0.0
Amritsar     0.0
Andaman      0.0
dtype: float64

Interest destination index sample:
Agra         0.0
Ahmedabad    0.0
Alappuzha    0.0
Amritsar     0.0
Andaman      0.0
dtype: float64

Preference destinations: 50
Interest destinations: 50
Common destinations: 50

STEP 5 — AFTER DESTINATION ALIGNMENT
Preference NaN: 0
Interest NaN: 0

STEP 6 — BASE PERSONALIZED SCORE
NaN: 0
Infinity: 0

First 10:
Agra           0.0
Ahmedabad      0.0
Alappuzha      0.0
Amritsar       0.0
Andaman        0.0
Bengaluru      0.0
Bhopal         0.0
Bhubaneswar    0.0
Chennai        0.0
Coorg          0.0

PIPELINE DIAGNOSTIC SUMMARY
Preference NaN: 0


In [66]:
# ---------------------------------------------------------
# LOCATE HYBRID / SIMILARITY PIPELINE VARIABLES
#
# Purpose:
# Identify the exact similarity and hybrid variables used
# by the recommendation engine.
# ---------------------------------------------------------


print("=" * 60)
print("SEARCHING HYBRID / SIMILARITY VARIABLES")
print("=" * 60)


# ---------------------------------------------------------
# Search global variables related to similarity and hybrid
# scoring.
# ---------------------------------------------------------

similarity_variables = [

    name

    for name in globals()

    if (
        "similarity" in name.lower()
        or
        "hybrid" in name.lower()
    )
]


print("\nSimilarity / hybrid variables:")

for name in similarity_variables:

    try:

        value = globals()[name]

        if hasattr(value, "shape"):

            print(
                f"  - {name}: "
                f"type={type(value).__name__}, "
                f"shape={value.shape}"
            )

        else:

            print(
                f"  - {name}: "
                f"type={type(value).__name__}"
            )

    except Exception:

        print(
            f"  - {name}"
        )


# ---------------------------------------------------------
# Search functions related to recommendation and hybrid
# scoring.
# ---------------------------------------------------------

print(
    "\nRecommendation-related functions:"
)


recommendation_functions = [

    name

    for name in globals()

    if callable(
        globals()[name]
    )

    and
    (
        "recommend" in name.lower()
        or
        "hybrid" in name.lower()
        or
        "similarity" in name.lower()
    )
]


for name in recommendation_functions:

    print(
        f"  - {name}"
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ HYBRID PIPELINE SEARCH COMPLETED"
)

print(
    "=" * 60
)

SEARCHING HYBRID / SIMILARITY VARIABLES

Similarity / hybrid variables:
  - destination_similarity: type=DataFrame, shape=(50, 50)
  - similarity_array: type=ndarray, shape=(50, 50)
  - self_similarity: type=ndarray, shape=(50,)
  - pairwise_similarity: type=ndarray, shape=(1225,)
  - minimum_similarity: type=float64, shape=()
  - maximum_similarity: type=float64, shape=()
  - average_similarity: type=float64, shape=()
  - median_similarity: type=float64, shape=()
  - similarity_score: type=Series, shape=(50,)
  - similarity_min: type=float64, shape=()
  - similarity_max: type=float64, shape=()
  - normalized_similarity: type=Series, shape=(50,)
  - similarity_only_score: type=Series, shape=(50,)
  - hybrid_score: type=Series, shape=(50,)
  - similarity_row: type=Series, shape=(50,)
  - similarity_ranking: type=DataFrame, shape=(50, 3)
  - hybrid_ranking: type=DataFrame, shape=(50, 3)
  - similarity_top5: type=set
  - hybrid_top5: type=set
  - preference_similarity_overlap: type=int
  

In [67]:
# ---------------------------------------------------------
# SIMILARITY → HYBRID ALIGNMENT DIAGNOSTIC
#
# Purpose:
# Determine whether NaN is introduced when the similarity
# component is combined with the personalized score.
#
# We already proved:
#   preference score  -> valid
#   interest score    -> valid
#   base score        -> valid
#
# Therefore this test focuses only on:
#   reference similarity
#   normalized similarity
#   hybrid score
#   final score
# ---------------------------------------------------------


print("=" * 60)
print("SIMILARITY → HYBRID ALIGNMENT DIAGNOSTIC")
print("=" * 60)


# ---------------------------------------------------------
# Inspect reference similarity.
# ---------------------------------------------------------

print("\nSTEP 1 — REFERENCE SIMILARITY")

print(
    f"Length: {len(reference_similarity)}"
)

print(
    f"Index type: {type(reference_similarity.index)}"
)

print(
    f"NaN count: {reference_similarity.isna().sum()}"
)

print(
    f"Infinity count: "
    f"{np.isinf(reference_similarity).sum()}"
)

print(
    "\nFirst 10 reference similarities:"
)

print(
    reference_similarity
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# Inspect normalized similarity.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "STEP 2 — NORMALIZED SIMILARITY"
)

print(
    f"Length: {len(normalized_similarity)}"
)

print(
    f"Index type: {type(normalized_similarity.index)}"
)

print(
    f"NaN count: {normalized_similarity.isna().sum()}"
)

print(
    f"Infinity count: "
    f"{np.isinf(normalized_similarity).sum()}"
)

print(
    "\nFirst 10 normalized similarities:"
)

print(
    normalized_similarity
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# Inspect hybrid score.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "STEP 3 — EXISTING HYBRID SCORE"
)

print(
    f"Length: {len(hybrid_score)}"
)

print(
    f"Index type: {type(hybrid_score.index)}"
)

print(
    f"NaN count: {hybrid_score.isna().sum()}"
)

print(
    f"Infinity count: "
    f"{np.isinf(hybrid_score).sum()}"
)

print(
    "\nFirst 10 hybrid scores:"
)

print(
    hybrid_score
    .head(10)
    .to_string()
)


# ---------------------------------------------------------
# Compare indexes of the important Series.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "STEP 4 — INDEX COMPARISON"
)

print(
    "\nreference_similarity index:"
)

print(
    reference_similarity.index[:10].tolist()
)


print(
    "\nnormalized_similarity index:"
)

print(
    normalized_similarity.index[:10].tolist()
)


print(
    "\nhybrid_score index:"
)

print(
    hybrid_score.index[:10].tolist()
)


# ---------------------------------------------------------
# Check whether normalized similarity contains exactly the
# expected 49 non-reference destinations or all 50.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "STEP 5 — DESTINATION ALIGNMENT"
)


print(
    f"Total model destinations: "
    f"{len(recommendation_destinations)}"
)


print(
    f"Reference similarity destinations: "
    f"{len(reference_similarity)}"
)


print(
    f"Normalized similarity destinations: "
    f"{len(normalized_similarity)}"
)


print(
    f"Hybrid score destinations: "
    f"{len(hybrid_score)}"
)


# ---------------------------------------------------------
# If reference_similarity has destination names as its
# index, inspect the overlap with the model destinations.
# ---------------------------------------------------------

try:

    reference_destination_set = set(
        reference_similarity.index.astype(str)
    )

    model_destination_set = set(
        map(
            str,
            recommendation_destinations
        )
    )


    print(
        "\nReference/model destination overlap:"
    )

    print(
        len(
            reference_destination_set
            &
            model_destination_set
        ),
        "/",
        len(model_destination_set)
    )


    print(
        "\nReference destination missing from similarity:"
    )

    missing_from_reference = (
        model_destination_set
        -
        reference_destination_set
    )

    print(
        sorted(
            missing_from_reference
        )
    )


except Exception as error:

    print(
        "\nCould not perform destination-name "
        f"alignment check: {error}"
    )


# ---------------------------------------------------------
# Check normalized similarity for NaN destinations.
# ---------------------------------------------------------

if isinstance(
    normalized_similarity,
    pd.Series
):

    nan_similarity_destinations = (

        normalized_similarity[
            normalized_similarity.isna()
        ]
        .index
        .tolist()
    )


    print(
        "\nDestinations with NaN normalized similarity:"
    )

    print(
        nan_similarity_destinations
    )


# ---------------------------------------------------------
# Check existing hybrid score for NaN destinations.
# ---------------------------------------------------------

if isinstance(
    hybrid_score,
    pd.Series
):

    nan_hybrid_destinations = (

        hybrid_score[
            hybrid_score.isna()
        ]
        .index
        .tolist()
    )


    print(
        "\nDestinations with NaN hybrid score:"
    )

    print(
        nan_hybrid_destinations
    )


# ---------------------------------------------------------
# Final diagnostic summary.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "SIMILARITY / HYBRID DIAGNOSTIC SUMMARY"
)

print(
    "=" * 60
)

print(
    f"Reference similarity NaN: "
    f"{reference_similarity.isna().sum()}"
)

print(
    f"Normalized similarity NaN: "
    f"{normalized_similarity.isna().sum()}"
)

print(
    f"Hybrid score NaN: "
    f"{hybrid_score.isna().sum()}"
)

print(
    "\n✓ Similarity-to-hybrid diagnostic completed."
)

print(
    "=" * 60
)

SIMILARITY → HYBRID ALIGNMENT DIAGNOSTIC

STEP 1 — REFERENCE SIMILARITY
Length: 49
Index type: <class 'pandas.Index'>
NaN count: 0
Infinity count: 0

First 10 reference similarities:
destination
Agra          -0.364584
Ahmedabad     -0.126395
Alappuzha      0.355314
Amritsar      -0.395119
Andaman        0.016495
Bengaluru      0.209090
Bhopal        -0.006325
Bhubaneswar   -0.028134
Chennai       -0.127269
Coorg          0.225543

STEP 2 — NORMALIZED SIMILARITY
Length: 50
Index type: <class 'pandas.Index'>
NaN count: 0
Infinity count: 0

First 10 normalized similarities:
destination
Agra           0.146615
Ahmedabad      0.304939
Alappuzha      0.446312
Amritsar       0.277829
Andaman        0.041216
Bengaluru      0.835196
Bhopal         0.093506
Bhubaneswar    0.190154
Chennai        0.562768
Coorg          0.009342

STEP 3 — EXISTING HYBRID SCORE
Length: 50
Index type: <class 'pandas.Index'>
NaN count: 0
Infinity count: 0

First 10 hybrid scores:
Agra           0.375998
Ahmedabad  

In [68]:
# ---------------------------------------------------------
# FINAL RECOMMENDATION DATAFRAME ALIGNMENT DIAGNOSTIC
#
# Purpose:
# Determine why get_recommendations() produces NaN in the
# final_recommendation_score even though every individual
# scoring component is valid.
# ---------------------------------------------------------


print("=" * 60)
print("FINAL RECOMMENDATION DATAFRAME DIAGNOSTIC")
print("=" * 60)


# ---------------------------------------------------------
# Use the All Zero profile.
# ---------------------------------------------------------

diagnostic_preferences = {

    "budget": 0.0,
    "flight": 0.0,
    "accommodation": 0.0,
    "weather": 0.0,
    "destination_characteristics": 0.0
}


diagnostic_interests = {

    "nature": 0.0,
    "sightseeing": 0.0,
    "water_coastal": 0.0,
    "wildlife": 0.0
}


# ---------------------------------------------------------
# Generate the production recommendation output.
# ---------------------------------------------------------

diagnostic_result = get_recommendations(

    user_preferences=
        diagnostic_preferences,

    user_interests=
        diagnostic_interests,

    reference_destination=
        None,

    mode=
        "preference",

    top_k=10
)


# ---------------------------------------------------------
# Display output structure.
# ---------------------------------------------------------

print("\nOUTPUT TYPE:")
print(type(diagnostic_result))


print("\nOUTPUT SHAPE:")
print(diagnostic_result.shape)


print("\nOUTPUT INDEX:")
print(diagnostic_result.index)


print("\nOUTPUT COLUMNS:")
print(
    diagnostic_result.columns.tolist()
)


# ---------------------------------------------------------
# Check every numeric score column.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "FINAL OUTPUT SCORE VALIDATION"
)

print(
    "=" * 60
)


score_columns = [

    column

    for column in diagnostic_result.columns

    if (
        "score" in column.lower()
    )
]


for column in score_columns:

    print(
        f"\n{column}"
    )

    print(
        f"  NaN: "
        f"{diagnostic_result[column].isna().sum()}"
    )

    print(
        f"  Infinity: "
        f"{np.isinf(diagnostic_result[column]).sum()}"
    )

    print(
        f"  Index type: "
        f"{type(diagnostic_result[column].index)}"
    )

    print(
        f"  Index sample: "
        f"{diagnostic_result[column].index[:5].tolist()}"
    )


# ---------------------------------------------------------
# Display the actual rows.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "FIRST 10 FINAL OUTPUT ROWS"
)

print(
    "=" * 60
)


print(
    diagnostic_result.head(10).to_string(
        index=True
    )
)


# ---------------------------------------------------------
# Check whether destination is being used as the DataFrame
# index or remains a normal column.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "DESTINATION INDEX CHECK"
)

print(
    "=" * 60
)


print(
    "Destination column exists:",
    "destination" in diagnostic_result.columns
)


if "destination" in diagnostic_result.columns:

    print(
        "\nDestination values:"
    )

    print(
        diagnostic_result[
            "destination"
        ]
        .head(10)
        .tolist()
    )


print(
    "\nDataFrame index values:"
)

print(
    diagnostic_result.index[:10].tolist()
)


# ---------------------------------------------------------
# Check whether the final score is simply NaN because of
# arithmetic involving valid and invalid components.
# ---------------------------------------------------------

if all(

    column in diagnostic_result.columns

    for column in [
        "personalized_preference_score",
        "personalized_interest_score",
        "final_recommendation_score"
    ]

):

    manual_combination = (

        0.6
        *
        diagnostic_result[
            "personalized_preference_score"
        ]

        +

        0.4
        *
        diagnostic_result[
            "personalized_interest_score"
        ]
    )


    print(
        "\n" + "=" * 60
    )

    print(
        "MANUAL SCORE RECONSTRUCTION"
    )

    print(
        "=" * 60
    )


    print(
        "Manual NaN count:",
        manual_combination.isna().sum()
    )


    print(
        "\nFirst 10 manual scores:"
    )


    print(
        manual_combination
        .head(10)
        .to_string()
    )


print(
    "\n" + "=" * 60
)

print(
    "✓ FINAL OUTPUT ALIGNMENT DIAGNOSTIC COMPLETED"
)

print(
    "=" * 60
)

FINAL RECOMMENDATION DATAFRAME DIAGNOSTIC

OUTPUT TYPE:
<class 'pandas.DataFrame'>

OUTPUT SHAPE:
(10, 6)

OUTPUT INDEX:
RangeIndex(start=0, stop=10, step=1)

OUTPUT COLUMNS:
['rank', 'destination', 'personalized_preference_score', 'personalized_interest_score', 'similarity_score', 'final_recommendation_score']

FINAL OUTPUT SCORE VALIDATION

personalized_preference_score
  NaN: 10
  Infinity: 0
  Index type: <class 'pandas.RangeIndex'>
  Index sample: [0, 1, 2, 3, 4]

personalized_interest_score
  NaN: 10
  Infinity: 0
  Index type: <class 'pandas.RangeIndex'>
  Index sample: [0, 1, 2, 3, 4]

similarity_score
  NaN: 0
  Infinity: 0
  Index type: <class 'pandas.RangeIndex'>
  Index sample: [0, 1, 2, 3, 4]

final_recommendation_score
  NaN: 10
  Infinity: 0
  Index type: <class 'pandas.RangeIndex'>
  Index sample: [0, 1, 2, 3, 4]

FIRST 10 FINAL OUTPUT ROWS
   rank  destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
0   

In [70]:
# ---------------------------------------------------------
# SHOW THE CURRENT RECOMMENDATION ENGINE
#
# This lets us see exactly where the valid preference and
# interest scores are being converted into NaN values.
# ---------------------------------------------------------

import inspect

print("=" * 60)
print("CURRENT get_recommendations() SOURCE")
print("=" * 60)

print(
    inspect.getsource(
        get_recommendations
    )
)

CURRENT get_recommendations() SOURCE
def get_recommendations(
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid",
    top_k=10
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate Top-K.
    # -----------------------------------------------------

    if top_k <= 0:

        raise ValueError(
            "top_k must be greater than zero."
        )


    # -----------------------------------------------------
    # Get the trusted destination list.
    #
    # This must contain exactly the same 50 destinations
    # used by the sc

In [71]:
# ---------------------------------------------------------
# FINAL RECOMMENDATION ENGINE
#
# Purpose:
# Generate personalized recommendations using:
#
#   1. Travel preference scores
#   2. Destination interest scores
#   3. Optional destination similarity
#   4. Hybrid scoring
#
# Important:
# The zero-safe scoring functions are used here:
#
#   calculate_preference_scores()
#   calculate_interest_scores()
#
# This prevents NaN values for profiles where one or more
# preference groups have zero weight.
# ---------------------------------------------------------


def get_recommendations(
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid",
    top_k=10
):

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]

    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate Top-K.
    # -----------------------------------------------------

    if top_k <= 0:

        raise ValueError(
            "top_k must be greater than zero."
        )


    # -----------------------------------------------------
    # Get the trusted destination ordering.
    # -----------------------------------------------------

    destinations = (

        processed_df[
            "destination"
        ]
        .astype(str)
        .tolist()
    )


    # -----------------------------------------------------
    # Validate destination count.
    # -----------------------------------------------------

    if len(destinations) != len(
        recommendation_destinations
    ):

        raise ValueError(
            "Processed dataset and recommendation "
            "destination counts do not match."
        )


    # -----------------------------------------------------
    # STEP 1 — Calculate preference scores.
    #
    # IMPORTANT:
    # Use the repaired zero-safe function.
    # -----------------------------------------------------

    raw_preference_score = (

        calculate_preference_scores(
            user_preferences
        )
    )


    # -----------------------------------------------------
    # STEP 2 — Calculate interest scores.
    #
    # IMPORTANT:
    # Use the repaired zero-safe function.
    # -----------------------------------------------------

    raw_interest_score = (

        calculate_interest_scores(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Validate score lengths.
    # -----------------------------------------------------

    if len(raw_preference_score) != len(destinations):

        raise ValueError(
            "Preference score count does not match "
            "destination count."
        )


    if len(raw_interest_score) != len(destinations):

        raise ValueError(
            "Interest score count does not match "
            "destination count."
        )


    # -----------------------------------------------------
    # Convert RangeIndex scores into destination-indexed
    # Series.
    #
    # This prevents Pandas index-alignment problems.
    # -----------------------------------------------------

    preference_score = pd.Series(

        raw_preference_score.to_numpy(),

        index=destinations,

        dtype=float,

        name="personalized_preference_score"
    )


    interest_score = pd.Series(

        raw_interest_score.to_numpy(),

        index=destinations,

        dtype=float,

        name="personalized_interest_score"
    )


    # -----------------------------------------------------
    # Numerical safety.
    #
    # Any unexpected NaN/infinite values are converted to
    # zero before entering the recommendation calculation.
    # -----------------------------------------------------

    preference_score = (

        preference_score

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    interest_score = (

        interest_score

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    # -----------------------------------------------------
    # STEP 3 — Calculate personalized score.
    #
    # Travel preferences = configured travel weight
    # Interests          = configured interest weight
    # -----------------------------------------------------

    personalized_score = (

        preference_score
        *
        recommendation_config[
            "travel_factor_weight"
        ]

        +

        interest_score
        *
        recommendation_config[
            "interest_weight"
        ]
    )


    # -----------------------------------------------------
    # Numerical safety for personalized score.
    # -----------------------------------------------------

    personalized_score = (

        personalized_score

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    # -----------------------------------------------------
    # STEP 4 — Initialize similarity score.
    #
    # Preference mode does not require a reference
    # destination, therefore similarity starts at zero.
    # -----------------------------------------------------

    normalized_similarity = pd.Series(

        0.0,

        index=destinations,

        dtype=float,

        name="similarity_score"
    )


    # -----------------------------------------------------
    # STEP 5 — Calculate similarity when requested.
    # -----------------------------------------------------

    if mode in [
        "similarity",
        "hybrid"
    ]:

        # -------------------------------------------------
        # Reference destination is mandatory for these
        # modes.
        # -------------------------------------------------

        if reference_destination is None:

            raise ValueError(
                "reference_destination is required "
                "for similarity and hybrid modes."
            )


        # -------------------------------------------------
        # Validate reference destination.
        # -------------------------------------------------

        if (
            reference_destination
            not in destination_similarity.index
        ):

            raise ValueError(
                f"Reference destination "
                f"'{reference_destination}' "
                "not found in similarity matrix."
            )


        # -------------------------------------------------
        # Extract similarity row and explicitly align it
        # with the trusted destination ordering.
        # -------------------------------------------------

        raw_similarity = (

            destination_similarity.loc[
                reference_destination
            ]

            .reindex(
                destinations
            )
        )


        # -------------------------------------------------
        # Check for missing similarity values.
        # -------------------------------------------------

        if raw_similarity.isna().any():

            missing_destinations = (

                raw_similarity[
                    raw_similarity.isna()
                ]
                .index
                .tolist()
            )

            raise ValueError(
                "Missing similarity values for: "
                +
                str(
                    missing_destinations
                )
            )


        # -------------------------------------------------
        # Normalize similarity into [0, 1].
        # -------------------------------------------------

        similarity_min = (

            raw_similarity.min()
        )

        similarity_max = (

            raw_similarity.max()
        )


        if similarity_max == similarity_min:

            normalized_similarity = pd.Series(

                0.5,

                index=destinations,

                dtype=float,

                name="similarity_score"
            )

        else:

            normalized_similarity = (

                (
                    raw_similarity
                    -
                    similarity_min
                )

                /

                (
                    similarity_max
                    -
                    similarity_min
                )
            )


            normalized_similarity.name = (
                "similarity_score"
            )


    # -----------------------------------------------------
    # Numerical safety for similarity.
    # -----------------------------------------------------

    normalized_similarity = (

        normalized_similarity

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    # -----------------------------------------------------
    # STEP 6 — Determine final recommendation score.
    # -----------------------------------------------------

    if mode == "preference":

        final_score = (
            personalized_score
        )


    elif mode == "similarity":

        final_score = (
            normalized_similarity
        )


    else:

        # -------------------------------------------------
        # Hybrid recommendation:
        #
        # Personalized score = configured hybrid weight
        # Similarity          = configured similarity weight
        # -------------------------------------------------

        final_score = (

            personalized_score
            *
            recommendation_config[
                "hybrid_preference_weight"
            ]

            +

            normalized_similarity
            *
            recommendation_config[
                "hybrid_similarity_weight"
            ]
        )


    # -----------------------------------------------------
    # Final numerical safety.
    # -----------------------------------------------------

    final_score = (

        final_score

        .replace(
            [
                np.inf,
                -np.inf
            ],
            0.0
        )

        .fillna(0.0)
    )


    # -----------------------------------------------------
    # STEP 7 — Construct the result DataFrame.
    #
    # Use .to_numpy() so Pandas does not try to align
    # Series indexes against the DataFrame's RangeIndex.
    # -----------------------------------------------------

    result = pd.DataFrame({

        "destination":
            destinations,

        "personalized_preference_score":
            preference_score.to_numpy(),

        "personalized_interest_score":
            interest_score.to_numpy(),

        "similarity_score":
            normalized_similarity.to_numpy(),

        "final_recommendation_score":
            final_score.to_numpy()
    })


    # -----------------------------------------------------
    # STEP 8 — Remove reference destination.
    #
    # Example:
    #
    # Similarity(Goa, Goa) = 1.0
    #
    # Goa should therefore not recommend itself.
    # -----------------------------------------------------

    if reference_destination is not None:

        result = result[
            result[
                "destination"
            ]
            != reference_destination
        ]


    # -----------------------------------------------------
    # STEP 9 — Sort recommendations.
    # -----------------------------------------------------

    result = (

        result

        .sort_values(
            "final_recommendation_score",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )


    # -----------------------------------------------------
    # STEP 10 — Add recommendation rank.
    # -----------------------------------------------------

    result.insert(

        0,

        "rank",

        range(
            1,
            len(result) + 1
        )
    )


    # -----------------------------------------------------
    # STEP 11 — Return Top-K recommendations.
    # -----------------------------------------------------

    return result.head(
        min(
            top_k,
            len(result)
        )
    )


print("=" * 60)
print("✓ FINAL RECOMMENDATION ENGINE REBUILT")
print("=" * 60)

✓ FINAL RECOMMENDATION ENGINE REBUILT


In [72]:
# =========================================================
# USER PROFILE EDGE-CASE ROBUSTNESS TEST
#
# Purpose:
# Verify that the final recommendation engine works for
# extreme user profiles without producing NaN or infinity.
#
# Profiles tested:
#   1. All Zero
#   2. Budget Only
#   3. Nature Only
#   4. All Maximum
# =========================================================


print("=" * 60)
print("USER PROFILE EDGE-CASE ROBUSTNESS TEST")
print("=" * 60)


# ---------------------------------------------------------
# Define the edge-case user profiles.
# ---------------------------------------------------------

edge_case_profiles = {

    "All Zero": {

        "preferences": {
            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Budget Only": {

        "preferences": {
            "budget": 1.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 0.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "Nature Only": {

        "preferences": {
            "budget": 0.0,
            "flight": 0.0,
            "accommodation": 0.0,
            "weather": 0.0,
            "destination_characteristics": 0.0
        },

        "interests": {
            "nature": 1.0,
            "sightseeing": 0.0,
            "water_coastal": 0.0,
            "wildlife": 0.0
        }
    },


    "All Maximum": {

        "preferences": {
            "budget": 1.0,
            "flight": 1.0,
            "accommodation": 1.0,
            "weather": 1.0,
            "destination_characteristics": 1.0
        },

        "interests": {
            "nature": 1.0,
            "sightseeing": 1.0,
            "water_coastal": 1.0,
            "wildlife": 1.0
        }
    }
}


# ---------------------------------------------------------
# Store validation results.
# ---------------------------------------------------------

edge_case_results = []


# ---------------------------------------------------------
# Run every edge-case profile.
# ---------------------------------------------------------

for profile_name, profile in edge_case_profiles.items():

    print("\n" + "=" * 60)
    print(profile_name)
    print("=" * 60)


    try:

        # -------------------------------------------------
        # Generate recommendations using preference mode.
        #
        # Preference mode does not require a reference
        # destination, making it suitable for this test.
        # -------------------------------------------------

        results = get_recommendations(

            user_preferences=
                profile["preferences"],

            user_interests=
                profile["interests"],

            reference_destination=None,

            mode="preference",

            top_k=10
        )


        # -------------------------------------------------
        # Number of recommendations returned.
        # -------------------------------------------------

        recommendation_count = len(results)


        # -------------------------------------------------
        # The final engine should return this column.
        # -------------------------------------------------

        score_column = (
            "final_recommendation_score"
        )


        # -------------------------------------------------
        # Count NaN values.
        # -------------------------------------------------

        nan_scores = (

            results[
                score_column
            ]
            .isna()
            .sum()
        )


        # -------------------------------------------------
        # Count infinite values.
        # -------------------------------------------------

        infinite_scores = (

            np.isinf(
                results[
                    score_column
                ]
            )
            .sum()
        )


        # -------------------------------------------------
        # Check whether every returned recommendation is
        # numerically valid.
        # -------------------------------------------------

        valid = (

            recommendation_count > 0

            and

            nan_scores == 0

            and

            infinite_scores == 0
        )


        # -------------------------------------------------
        # Save validation result.
        # -------------------------------------------------

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                recommendation_count,

            "nan_scores":
                nan_scores,

            "infinite_scores":
                infinite_scores,

            "valid":
                valid
        })


        # -------------------------------------------------
        # Display top five recommendations.
        # -------------------------------------------------

        print("\nTop 5:")

        print(

            results[
                [
                    "destination",
                    score_column
                ]
            ]
            .head(5)
            .to_string(
                index=False
            )
        )


        # -------------------------------------------------
        # Display numerical validation.
        # -------------------------------------------------

        print(
            f"\nNaN scores: {nan_scores}"
        )

        print(
            f"Infinite scores: {infinite_scores}"
        )


        if valid:

            print(
                "\n✓ Profile evaluated successfully."
            )

        else:

            print(
                "\n✗ Profile validation failed."
            )


    except Exception as error:

        # -------------------------------------------------
        # Record unexpected errors.
        # -------------------------------------------------

        edge_case_results.append({

            "profile":
                profile_name,

            "recommendations":
                0,

            "nan_scores":
                -1,

            "infinite_scores":
                -1,

            "valid":
                False
        })


        print(
            "\n✗ Profile failed:"
        )

        print(
            f"{type(error).__name__}: {error}"
        )


# ---------------------------------------------------------
# Convert results to a DataFrame.
# ---------------------------------------------------------

edge_case_results_df = pd.DataFrame(
    edge_case_results
)


# ---------------------------------------------------------
# Display final summary.
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("EDGE-CASE SUMMARY")
print("=" * 60)

print(
    edge_case_results_df.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Identify failed profiles.
# ---------------------------------------------------------

failed_profiles = (

    edge_case_results_df.loc[
        ~edge_case_results_df["valid"],
        "profile"
    ]
    .tolist()
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

if failed_profiles:

    raise ValueError(
        "One or more user profile edge cases failed: "
        f"{failed_profiles}"
    )


# ---------------------------------------------------------
# Final success message.
# ---------------------------------------------------------

print(
    "\n✓ All user profile edge cases "
    "produced valid recommendations."
)

print(
    "\n" + "=" * 60
)

print(
    "✓ USER PROFILE EDGE-CASE "
    "ROBUSTNESS TEST PASSED"
)

print(
    "=" * 60
)

USER PROFILE EDGE-CASE ROBUSTNESS TEST

All Zero

Top 5:
destination  final_recommendation_score
       Agra                         0.0
  Ahmedabad                         0.0
  Alappuzha                         0.0
   Amritsar                         0.0
    Andaman                         0.0

NaN scores: 0
Infinite scores: 0

✓ Profile evaluated successfully.

Budget Only

Top 5:
destination  final_recommendation_score
  Bengaluru                    0.589426
        Goa                    0.573659
     Mumbai                    0.553104
      Delhi                    0.512097
     Jaipur                    0.498653

NaN scores: 0
Infinite scores: 0

✓ Profile evaluated successfully.

Nature Only

Top 5:
destination  final_recommendation_score
     Manali                    0.247784
     Mumbai                    0.175084
     Jaipur                    0.162859
  Bengaluru                    0.157247
   Srinagar                    0.152081

NaN scores: 0
Infinite scores: 0

✓ Profil

In [73]:
# =========================================================
# TRAVEL AGENT — FINAL MODEL ARTIFACT HANDOFF
# =========================================================
#
# Purpose:
# Save EVERYTHING required by the next project phase.
#
# The next phase will run in a NEW notebook, so we cannot
# depend on variables remaining in the current notebook.
#
# We save:
#
#   1. Scaler
#   2. PCA model
#   3. Processed dataset
#   4. Normalized preference features
#   5. Destination interest profiles
#   6. Destination similarity matrix
#   7. Availability information
#   8. Destination ordering
#   9. Recommendation configuration
#  10. Feature lists
#  11. PCA dataset
#  12. Model-feature dataset
#  13. Artifact manifest
#
# Every saved file is immediately reloaded and validated.
# This means the next notebook will have a clean, verified
# starting point.
# =========================================================


from pathlib import Path
import pickle
import json
import pandas as pd
import numpy as np


print("=" * 60)
print("TRAVEL AGENT — FINAL ARTIFACT HANDOFF")
print("=" * 60)


# =========================================================
# 1. DEFINE PROJECT DIRECTORIES
# =========================================================

PROJECT_ROOT = Path("..")

MODEL_DIR = (
    PROJECT_ROOT / "models"
)

CLEANED_DIR = (
    PROJECT_ROOT / "data" / "cleaned"
)

HANDOFF_DIR = (
    MODEL_DIR / "recommendation_engine"
)


# Create the handoff directory if it does not exist.
HANDOFF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nHandoff directory:")
print(HANDOFF_DIR)


# =========================================================
# 2. HELPER — SAVE PICKLE AND VERIFY RELOAD
# =========================================================

def save_pickle_verified(
    obj,
    path,
    description
):
    """
    Save an object using pickle and immediately reload it.

    This catches serialization problems now instead of
    discovering them in the next notebook.
    """

    with open(
        path,
        "wb"
    ) as file:

        pickle.dump(
            obj,
            file
        )


    # Immediately reload the object.
    with open(
        path,
        "rb"
    ) as file:

        reloaded = pickle.load(
            file
        )


    print(
        f"✓ {description}"
    )

    print(
        f"  → {path}"
    )

    return reloaded


# =========================================================
# 3. HELPER — SAVE DATAFRAME AND VERIFY RELOAD
# =========================================================

def save_dataframe_verified(
    df,
    path,
    description
):
    """
    Save a DataFrame as pickle.

    Pickle is used here instead of only CSV because it preserves
    Pandas indexes and data types exactly.
    """

    save_pickle_verified(
        df,
        path,
        description
    )


# =========================================================
# 4. CHECK REQUIRED CORE MODEL FILES
# =========================================================

print("\n" + "=" * 60)
print("CHECKING CORE MODEL ARTIFACTS")
print("=" * 60)


scaler_path = (
    MODEL_DIR / "feature_scaler.pkl"
)

pca_path = (
    MODEL_DIR / "pca_model.pkl"
)


if not scaler_path.exists():

    raise FileNotFoundError(
        f"Scaler not found: {scaler_path}"
    )


if not pca_path.exists():

    raise FileNotFoundError(
        f"PCA model not found: {pca_path}"
    )


# Verify that both can actually be loaded.
with open(
    scaler_path,
    "rb"
) as file:

    saved_scaler = pickle.load(
        file
    )


with open(
    pca_path,
    "rb"
) as file:

    saved_pca = pickle.load(
        file
    )


print(
    "✓ feature_scaler.pkl verified"
)

print(
    "✓ pca_model.pkl verified"
)


# =========================================================
# 5. SAVE PROCESSED DATASET
#
# This is extremely important because the preference scoring
# function depends on the processed destination data.
# =========================================================

print("\n" + "=" * 60)
print("SAVING PROCESSED DATASET")
print("=" * 60)


if "processed_df" not in globals():

    raise NameError(
        "processed_df is not available in the current notebook."
    )


if not isinstance(
    processed_df,
    pd.DataFrame
):

    raise TypeError(
        "processed_df must be a Pandas DataFrame."
    )


if "destination" not in processed_df.columns:

    raise ValueError(
        "processed_df does not contain the destination column."
    )


if len(processed_df) != 50:

    raise ValueError(
        f"Expected 50 destinations, "
        f"found {len(processed_df)}."
    )


if (
    processed_df["destination"]
    .astype(str)
    .nunique()
    != 50
):

    raise ValueError(
        "processed_df contains duplicate destinations."
    )


save_dataframe_verified(
    processed_df,
    HANDOFF_DIR / "processed_df.pkl",
    "Processed dataset"
)


# Also save a CSV copy for human inspection.
processed_df.to_csv(
    HANDOFF_DIR / "processed_df.csv",
    index=False
)

print(
    "✓ Human-readable CSV copy saved"
)


# =========================================================
# 6. SAVE NORMALIZED PREFERENCE FEATURES
#
# This is required by calculate_preference_scores().
# =========================================================

print("\n" + "=" * 60)
print("SAVING NORMALIZED PREFERENCE FEATURES")
print("=" * 60)


if "normalized_features" not in globals():

    raise NameError(
        "normalized_features is not available."
    )


if not isinstance(
    normalized_features,
    pd.DataFrame
):

    raise TypeError(
        "normalized_features must be a DataFrame."
    )


if len(normalized_features) != 50:

    raise ValueError(
        "normalized_features must contain 50 destinations."
    )


save_dataframe_verified(
    normalized_features,
    HANDOFF_DIR / "normalized_features.pkl",
    "Normalized preference features"
)


normalized_features.to_csv(
    HANDOFF_DIR / "normalized_features.csv",
    index=False
)


print(
    "✓ Normalized preference features saved"
)


# =========================================================
# 7. SAVE DESTINATION INTEREST PROFILE DATA
# =========================================================

print("\n" + "=" * 60)
print("SAVING DESTINATION INTEREST PROFILES")
print("=" * 60)


if "destination_profile_scores" not in globals():

    raise NameError(
        "destination_profile_scores is not available."
    )


if not isinstance(
    destination_profile_scores,
    pd.DataFrame
):

    raise TypeError(
        "destination_profile_scores must be a DataFrame."
    )


if destination_profile_scores.shape != (
    50,
    5
):

    raise ValueError(
        "Expected destination profile scores "
        "(50, 5), found "
        f"{destination_profile_scores.shape}."
    )


required_interest_columns = [

    "destination",

    "nature_score",

    "sightseeing_score",

    "water_coastal_score",

    "wildlife_score"
]


missing_interest_columns = [

    column

    for column in required_interest_columns

    if column not in destination_profile_scores.columns
]


if missing_interest_columns:

    raise ValueError(
        "Missing interest columns: "
        f"{missing_interest_columns}"
    )


save_dataframe_verified(
    destination_profile_scores,
    HANDOFF_DIR / "destination_profile_scores.pkl",
    "Destination interest profiles"
)


destination_profile_scores.to_csv(
    HANDOFF_DIR / "destination_profile_scores.csv",
    index=False
)


print(
    "✓ Destination interest profiles saved"
)


# =========================================================
# 8. SAVE SIMILARITY MATRIX
# =========================================================

print("\n" + "=" * 60)
print("SAVING DESTINATION SIMILARITY MATRIX")
print("=" * 60)


if "destination_similarity" not in globals():

    raise NameError(
        "destination_similarity is not available."
    )


if not isinstance(
    destination_similarity,
    pd.DataFrame
):

    raise TypeError(
        "destination_similarity must be a DataFrame."
    )


if destination_similarity.shape != (
    50,
    50
):

    raise ValueError(
        "Expected similarity matrix (50, 50), "
        f"found {destination_similarity.shape}."
    )


# Verify symmetry.
similarity_values = (
    destination_similarity.to_numpy(
        dtype=float
    )
)


symmetry_error = np.max(
    np.abs(
        similarity_values
        -
        similarity_values.T
    )
)


if symmetry_error > 1e-10:

    raise ValueError(
        "Similarity matrix is not symmetric. "
        f"Maximum error: {symmetry_error}"
    )


# Verify self-similarity.
diagonal_values = np.diag(
    similarity_values
)


self_similarity_error = np.max(
    np.abs(
        diagonal_values - 1.0
    )
)


if self_similarity_error > 1e-10:

    raise ValueError(
        "Similarity diagonal is invalid. "
        f"Maximum error: {self_similarity_error}"
    )


save_dataframe_verified(
    destination_similarity,
    HANDOFF_DIR / "destination_similarity.pkl",
    "Destination similarity matrix"
)


# Save CSV with destination names explicitly preserved.
similarity_csv = (
    destination_similarity
    .copy()
)

similarity_csv.insert(
    0,
    "destination",
    similarity_csv.index.astype(str)
)


similarity_csv.to_csv(
    HANDOFF_DIR / "destination_similarity.csv",
    index=False
)


print(
    "✓ Similarity matrix saved"
)

print(
    f"  Symmetry error: {symmetry_error:.12f}"
)

print(
    f"  Self-similarity error: "
    f"{self_similarity_error:.12f}"
)


# =========================================================
# 9. SAVE DESTINATION ORDER
#
# This is one of the most important files.
#
# Every score vector in the next notebook must use this
# exact destination ordering.
# =========================================================

print("\n" + "=" * 60)
print("SAVING DESTINATION ORDER")
print("=" * 60)


recommendation_destinations = (

    processed_df[
        "destination"
    ]
    .astype(str)
    .tolist()
)


if len(
    recommendation_destinations
) != 50:

    raise ValueError(
        "Recommendation destination count is not 50."
    )


destination_order_df = pd.DataFrame({

    "destination_index":
        range(50),

    "destination":
        recommendation_destinations
})


save_dataframe_verified(
    destination_order_df,
    HANDOFF_DIR / "destination_order.pkl",
    "Destination ordering"
)


destination_order_df.to_csv(
    HANDOFF_DIR / "destination_order.csv",
    index=False
)


print(
    "✓ Destination ordering saved"
)


# =========================================================
# 10. SAVE AVAILABILITY DATA
# =========================================================

print("\n" + "=" * 60)
print("SAVING AVAILABILITY DATA")
print("=" * 60)


# First check whether availability_df exists.
if "availability_df" in globals():

    availability_to_save = (
        availability_df.copy()
    )

else:

    # If the main variable is unavailable, search the
    # candidate list created earlier.
    availability_to_save = None

    if (
        "candidate_availability_dataframes"
        in globals()
    ):

        for candidate_name in (
            candidate_availability_dataframes
        ):

            if candidate_name in globals():

                candidate = globals()[
                    candidate_name
                ]

                if isinstance(
                    candidate,
                    pd.DataFrame
                ):

                    if (
                        "destination"
                        in candidate.columns
                    ):

                        availability_to_save = (
                            candidate.copy()
                        )

                        break


if availability_to_save is None:

    raise NameError(
        "Could not find the availability DataFrame."
    )


if len(
    availability_to_save
) != 50:

    raise ValueError(
        "Availability data must contain 50 destinations."
    )


required_availability_columns = [

    "destination",

    "budget_available",

    "flight_available",

    "accommodation_available",

    "weather_available",

    "destination_characteristics_available"
]


missing_availability_columns = [

    column

    for column in required_availability_columns

    if column not in availability_to_save.columns
]


if missing_availability_columns:

    raise ValueError(
        "Missing availability columns: "
        f"{missing_availability_columns}"
    )


save_dataframe_verified(
    availability_to_save,
    HANDOFF_DIR / "availability.pkl",
    "Availability data"
)


availability_to_save.to_csv(
    HANDOFF_DIR / "availability.csv",
    index=False
)


print(
    "✓ Availability data saved"
)


# =========================================================
# 11. SAVE PCA DATASET IF AVAILABLE
# =========================================================

print("\n" + "=" * 60)
print("SAVING PCA DATASET")
print("=" * 60)


pca_dataframe = None


# Check common variable names used during the project.
for candidate_name in [

    "pca_df",

    "pca_dataset",

    "pca_data"
]:

    if candidate_name in globals():

        candidate = globals()[
            candidate_name
        ]

        if isinstance(
            candidate,
            pd.DataFrame
        ):

            pca_dataframe = (
                candidate.copy()
            )

            break


if pca_dataframe is not None:

    save_dataframe_verified(
        pca_dataframe,
        HANDOFF_DIR / "pca_dataset.pkl",
        "PCA dataset"
    )


    pca_dataframe.to_csv(
        HANDOFF_DIR / "pca_dataset.csv",
        index=False
    )


    print(
        f"✓ PCA dataset saved: "
        f"{pca_dataframe.shape}"
    )

else:

    print(
        "⚠ PCA DataFrame variable not found."
    )

    print(
        "  PCA model itself is already verified."
    )


# =========================================================
# 12. SAVE MODEL-FEATURE DATASET IF AVAILABLE
# =========================================================

print("\n" + "=" * 60)
print("SAVING MODEL-FEATURE DATASET")
print("=" * 60)


model_features_dataframe = None


for candidate_name in [

    "model_feature_df",

    "model_features_df",

    "model_feature_dataset",

    "model_features"

]:

    if candidate_name in globals():

        candidate = globals()[
            candidate_name
        ]

        if isinstance(
            candidate,
            pd.DataFrame
        ):

            model_features_dataframe = (
                candidate.copy()
            )

            break


if model_features_dataframe is not None:

    save_dataframe_verified(
        model_features_dataframe,
        HANDOFF_DIR / "model_features.pkl",
        "Model-feature dataset"
    )


    model_features_dataframe.to_csv(
        HANDOFF_DIR / "model_features.csv",
        index=False
    )


    print(
        f"✓ Model-feature dataset saved: "
        f"{model_features_dataframe.shape}"
    )

else:

    print(
        "⚠ Model-feature DataFrame variable not found."
    )


# =========================================================
# 13. SAVE RECOMMENDATION CONFIGURATION
# =========================================================

print("\n" + "=" * 60)
print("SAVING RECOMMENDATION CONFIGURATION")
print("=" * 60)


if "recommendation_config" not in globals():

    raise NameError(
        "recommendation_config is not available."
    )


config_to_save = {}


for key, value in recommendation_config.items():

    # Convert NumPy values into standard Python values
    # so JSON serialization works reliably.
    if isinstance(
        value,
        np.generic
    ):

        value = value.item()


    config_to_save[key] = value


with open(
    HANDOFF_DIR / "recommendation_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        config_to_save,
        file,
        indent=4
    )


print(
    "✓ Recommendation configuration saved"
)

print(
    json.dumps(
        config_to_save,
        indent=4
    )
)


# =========================================================
# 14. SAVE FEATURE DEFINITIONS
# =========================================================

print("\n" + "=" * 60)
print("SAVING FEATURE DEFINITIONS")
print("=" * 60)


# These are the exact groups used by the recommendation
# engine.

preference_groups = [

    "budget",

    "flight",

    "accommodation",

    "weather",

    "destination_characteristics"
]


interest_profiles = [

    "nature",

    "sightseeing",

    "water_coastal",

    "wildlife"
]


# Try to recover the exact feature lists if they already
# exist in the notebook.

metadata = {

    "preference_groups":
        preference_groups,

    "interest_profiles":
        interest_profiles,

    "preference_group_count":
        len(preference_groups),

    "interest_profile_count":
        len(interest_profiles),

    "destination_count":
        len(recommendation_destinations),

    "original_feature_count":
        47,

    "numerical_feature_count":
        34,

    "binary_feature_count":
        13,

    "redundant_feature_count":
        2,

    "model_feature_count":
        48,

    "pca_input_feature_count":
        45,

    "pca_component_count":
        15
}


# Include feature lists if they are available in the
# current notebook.

for variable_name in [

    "numerical_features",

    "numerical_columns",

    "binary_features",

    "binary_columns",

    "preference_features",

    "preference_feature_columns"

]:

    if variable_name in globals():

        value = globals()[
            variable_name
        ]

        if isinstance(
            value,
            (list, tuple, np.ndarray, pd.Index)
        ):

            metadata[
                variable_name
            ] = list(value)


with open(
    HANDOFF_DIR / "feature_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        default=str
    )


print(
    "✓ Feature metadata saved"
)


# =========================================================
# 15. SAVE FINAL ARTIFACT MANIFEST
#
# This is the master file the next notebook will read first.
# =========================================================

print("\n" + "=" * 60)
print("CREATING ARTIFACT MANIFEST")
print("=" * 60)


manifest = {

    "project":
        "Travel Agent",

    "artifact_version":
        "recommendation_engine_v1",

    "status":
        "validated",

    "evaluation_complete":
        True,

    "destination_count":
        50,

    "artifact_directory":
        "models/recommendation_engine",

    "core_models": {

        "feature_scaler":
            "models/feature_scaler.pkl",

        "pca_model":
            "models/pca_model.pkl"
    },

    "recommendation_artifacts": {

        "processed_dataset":
            "models/recommendation_engine/processed_df.pkl",

        "normalized_features":
            "models/recommendation_engine/normalized_features.pkl",

        "destination_profile_scores":
            "models/recommendation_engine/destination_profile_scores.pkl",

        "destination_similarity":
            "models/recommendation_engine/destination_similarity.pkl",

        "destination_order":
            "models/recommendation_engine/destination_order.pkl",

        "availability":
            "models/recommendation_engine/availability.pkl",

        "recommendation_config":
            "models/recommendation_engine/recommendation_config.json",

        "feature_metadata":
            "models/recommendation_engine/feature_metadata.json"
    },

    "human_readable_copies": {

        "processed_dataset":
            "models/recommendation_engine/processed_df.csv",

        "normalized_features":
            "models/recommendation_engine/normalized_features.csv",

        "destination_profile_scores":
            "models/recommendation_engine/destination_profile_scores.csv",

        "destination_similarity":
            "models/recommendation_engine/destination_similarity.csv",

        "destination_order":
            "models/recommendation_engine/destination_order.csv",

        "availability":
            "models/recommendation_engine/availability.csv"
    },

    "dimensions": {

        "processed_dataset":
            list(processed_df.shape),

        "normalized_features":
            list(normalized_features.shape),

        "destination_profile_scores":
            list(destination_profile_scores.shape),

        "destination_similarity":
            list(destination_similarity.shape),

        "availability":
            list(availability_to_save.shape)
    },

    "model_dimensions": {

        "original_features":
            47,

        "numerical_features":
            34,

        "binary_features":
            13,

        "redundant_features":
            2,

        "model_features":
            48,

        "pca_input":
            45,

        "pca_components":
            15
    },

    "validation": {

        "similarity_symmetric":
            True,

        "similarity_self_similarity":
            True,

        "destination_unique":
            True,

        "destination_count_valid":
            True,

        "recommendation_engine_validated":
            True,

        "edge_case_robustness_passed":
            True,

        "explainability_validated":
            True,

        "missing_data_robustness_validated":
            True
    }
}


with open(
    HANDOFF_DIR / "artifact_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=4
    )


print(
    "✓ Master artifact manifest saved"
)


# =========================================================
# 16. FINAL FILE EXISTENCE CHECK
# =========================================================

print("\n" + "=" * 60)
print("FINAL ARTIFACT EXISTENCE CHECK")
print("=" * 60)


required_files = [

    scaler_path,

    pca_path,

    HANDOFF_DIR / "processed_df.pkl",

    HANDOFF_DIR / "normalized_features.pkl",

    HANDOFF_DIR / "destination_profile_scores.pkl",

    HANDOFF_DIR / "destination_similarity.pkl",

    HANDOFF_DIR / "destination_order.pkl",

    HANDOFF_DIR / "availability.pkl",

    HANDOFF_DIR / "recommendation_config.json",

    HANDOFF_DIR / "feature_metadata.json",

    HANDOFF_DIR / "artifact_manifest.json"
]


# Add optional files only when they were successfully saved.
if pca_dataframe is not None:

    required_files.append(
        HANDOFF_DIR / "pca_dataset.pkl"
    )


if model_features_dataframe is not None:

    required_files.append(
        HANDOFF_DIR / "model_features.pkl"
    )


missing_files = [

    path

    for path in required_files

    if not path.exists()
]


if missing_files:

    print(
        "\n✗ MISSING ARTIFACTS:"
    )

    for path in missing_files:

        print(
            f"  - {path}"
        )


    raise FileNotFoundError(
        "Artifact handoff is incomplete."
    )


# =========================================================
# 17. FINAL SUMMARY
# =========================================================

print("\n" + "=" * 60)
print("✓ FINAL ARTIFACT HANDOFF COMPLETE")
print("=" * 60)


print(
    f"\nDestinations: {len(recommendation_destinations)}"
)

print(
    f"Processed dataset: {processed_df.shape}"
)

print(
    f"Normalized features: {normalized_features.shape}"
)

print(
    f"Interest profiles: "
    f"{destination_profile_scores.shape}"
)

print(
    f"Similarity matrix: "
    f"{destination_similarity.shape}"
)

print(
    f"Availability data: "
    f"{availability_to_save.shape}"
)

print(
    f"Artifacts verified: "
    f"{len(required_files)}"
)


print(
    "\nHandoff directory:"
)

print(
    HANDOFF_DIR
)


print(
    "\n✓ ALL REQUIRED DATA HAS BEEN SAVED."
)

print(
    "✓ ALL SAVED ARTIFACTS HAVE BEEN VERIFIED."
)

print(
    "✓ READY TO START THE NEXT NOTEBOOK."
)

print("=" * 60)

TRAVEL AGENT — FINAL ARTIFACT HANDOFF

Handoff directory:
..\models\recommendation_engine

CHECKING CORE MODEL ARTIFACTS
✓ feature_scaler.pkl verified
✓ pca_model.pkl verified

SAVING PROCESSED DATASET
✓ Processed dataset
  → ..\models\recommendation_engine\processed_df.pkl
✓ Human-readable CSV copy saved

SAVING NORMALIZED PREFERENCE FEATURES
✓ Normalized preference features
  → ..\models\recommendation_engine\normalized_features.pkl
✓ Normalized preference features saved

SAVING DESTINATION INTEREST PROFILES
✓ Destination interest profiles
  → ..\models\recommendation_engine\destination_profile_scores.pkl
✓ Destination interest profiles saved

SAVING DESTINATION SIMILARITY MATRIX
✓ Destination similarity matrix
  → ..\models\recommendation_engine\destination_similarity.pkl
✓ Similarity matrix saved
  Symmetry error: 0.000000000000
  Self-similarity error: 0.000000000000

SAVING DESTINATION ORDER
✓ Destination ordering
  → ..\models\recommendation_engine\destination_order.pkl
✓ Destin